In [1]:
# =====================================================================
# MC AI BRAIN 5 - CELL 0: DATA - PINNED, CHECKED, GIVEN ROLES
# Runs first. Each file is kept only with the SHA-256 written here: used
# if already in mc_ai_brain5/data, copied if attached (any folder under
# /kaggle/input), else downloaded at a fixed commit (internet on).
# Writes data/sources.json (address, commit, SHA-256, bytes of each file)
# and data/roles.json. Copies the attached Brain 4 and Brain 3 ledgers,
# each only if its hash chain verifies, into mc_ai_brain5/earlier/, so
# Brain 5's ledger keeps their looks after the inputs are removed. Makes
# /kaggle/working/mcai for the library cells, on the import path.
# SOURCES  Brain 4's seventeen files, unchanged (see Brain 4 Cell 0), and
#   from Hugging Face at fixed commits (fetch check, 2026-11-16):
#   arc      ARC Easy and Challenge (Clark et al. 2018), allenai/ai2_arc
#            210d026f: train, validation, test. CC BY-SA 4.0
#   obqa     OpenBookQA (Mihaylov et al. 2018), allenai/openbookqa
#            388097ea: main, additional (with each question's fact).
#            Licence not stated: used here, never re-hosted
#   mmlu     MMLU (Hendrycks et al. 2021), cais/mmlu c30699e8, all 57
#            subjects: dev, validation, test. MIT
#   mmlupro  MMLU-Pro (Wang et al. 2024), TIGER-Lab/MMLU-Pro b189ec76:
#            validation, test. MIT
# ROLES
#   fit        counted into the model: SemCor and the train splits
#   practice   scored as often as needed: SemEval-2007, EWT dev, MMLU
#              dev, the validation splits
#   test       each model scored on it is a look, counted: TEST4 (SE2,
#              SE3, SE13, SE15), the test splits, the word-pair tests
#   knowledge  read, never scored: WordNet, gloss corpus, thesauri,
#              Simple Wikipedia
#   sealed     42D (Maru et al. 2022): pinned in roles.json, not fetched;
#              one final run, after the freeze
# PREDICTIONS (written before running)
#   P1 every file has the SHA-256 written here
#   P2 under 5 minutes
# =====================================================================
import hashlib, json, os, shutil, sys, tarfile, time, urllib.error, urllib.request, zipfile
from pathlib import Path

print("=" * 64); print("MC AI BRAIN 5 - CELL 0: DATA - PINNED, CHECKED, GIVEN ROLES"); print("=" * 64)
T0 = time.time()
HOME = Path(os.environ.get("MC_AI_HOME", "/kaggle/working"))
WORK = HOME / "mc_ai_brain5"
DATA, EARLIER = WORK / "data", WORK / "earlier"
BASE = os.environ.get("MC_AI_FETCH_BASE", "")
INPUT = Path(os.environ.get("MC_AI_INPUT", "/kaggle/input"))
AGENT = "MC-AI-Brain5/1.0 (Kaggle notebook; pinned SHA-256 downloads)"
GITHUB = "https://raw.githubusercontent.com/{repo}/{commit}/{file}"
HF = "https://huggingface.co/datasets/{repo}/resolve/{commit}/{file}"
(HOME / "mcai").mkdir(parents=True, exist_ok=True)
(HOME / "mcai" / "__init__.py").touch()
if str(HOME) not in sys.path:
    sys.path.insert(0, str(HOME))
P = "-00000-of-00001.parquet"
SOURCES = {  # source: (repo, commit, {file: SHA-256 or (SHA-256, bytes)}, address template(s))
    "ewt": ("UniversalDependencies/UD_English-EWT", "4a4d77f599ea53cc405f85d0cec4b2f14f81d42b", {
        "en_ewt-ud-train.conllu": "d68e06122a702464c613076523d56740f047e5bbe89dd90ec32737e04d952143",
        "en_ewt-ud-dev.conllu": "39239e0a60db3ae68f4b7036189f11b6692741d10ff8240dd91f74f2760d90f8",
        "en_ewt-ud-test.conllu": "fa024f43dc5da3c5ac02563bc9bd0e974f46cbb1560823976a8f342a37dc494a"}),
    "wsd": ("akranlu/wsd-data", "697cb66cf7ba874c276c863f2e9e2ea32b6c52d9", {
        "data/SemCor/semcor.data.xml": "45003c51336fcb15b9d8a6556be65eaadf5bf3285ddbae763675000e28b00753",
        "data/SemCor/semcor.gold.key.txt": "d5bb7966417fc1861c43a3851e03aa7b762b9004d9ffe7636637420d779259af",
        "data/ALL/ALL.data.xml": "27a9da01b6d4a1e4dc7c76d15148a38eaa56cc0df371cae4416d04e541b6fe36",
        "data/ALL/ALL.gold.key.txt": "0bf2ffc38f572587c72b59ea928da676423b99b5a54bd1a83eb4848b517d3170"}),
    "wordnet": ("nltk/nltk_data", "550b6625bcef1f2abff2ff770a5a0d272c9c6b2a", {
        "packages/corpora/wordnet.zip": "cbda5ea6eef7f36a97a43d4a75f85e07fccbb4f23657d27b4ccbc93e2646ab59"}),
    "gsm": ("openai/grade-school-math", "3101c7d5072418e28b9008a6636bde82a006892c", {
        "grade_school_math/data/train.jsonl": "17f347dc51477c50d4efb83959dbb7c56297aba886e5544ee2aaed3024813465",
        "grade_school_math/data/test.jsonl": "3730d312f6e3440559ace48831e51066acaca737f6eabec99bccb9e4b3c39d14"}),
    "encyclopedia": ("wikimedia/wikipedia", "b04c8d1ceb2f5cd4588862100d08de323dccfbaa", {
        "20231101.simple/train" + P: "31bded16768a47c286becd292079122f5d7d4397a17b87d4250a00ccd581e6f0"}, HF),
    "wordsim": ("mfaruqui/eval-word-vectors", "94b3dd9fdbac7ed08dbee7179cd5ee464acf0594", {
        "data/word-sim/EN-MEN-TR-3k.txt": "888b13580faa362c7b11da0a670258cf1b1eb4bf0ecbcd418918729a3bd3f43a",
        "data/word-sim/EN-SimVerb-3500.txt": "9d84274e7ad029fd3cf69dd0516e9ade624bbef9bd4e2050f6cc1fa01471e0f5",
        "data/word-sim/EN-RW-STANFORD.txt": "5108d54dfc746f44f9c3d7e31d2857c37cf71430bc1594175a61a3b89bb0ae1f"}),
    "simlex": ("fh295.github.io", "-", {
        "SimLex-999.zip": "30f40eb18f41b08b65dfbc328f473e8b69ba444b1c9d053c1adde2324fc76315"},
        "https://fh295.github.io/{file}"),
    "thesaurus": ("www.gutenberg.org", "-", {
        "3/2/0/3202/files.zip": "3a54e61dd3b4153313e00c3113bda1e939af505d1336a350a0fd93a80538a6d8"},
        ["https://gutenberg.pglaf.org/{file}", "https://mirror.cs.odu.edu/gutenberg/{file}",
         "https://www.gutenberg.org/dirs/{file}"]),
    "glosstag": ("wordnetcode.princeton.edu", "-", {
        "glosstag-files/WordNet-3.0-glosstag.tar.bz2":
            "c7b8318ac873ecbea14b225f33465210ec20ce19d9c6f7fc82bbbbe1e6b3080a"},
        "https://wordnetcode.princeton.edu/{file}"),
    "arc": ("allenai/ai2_arc", "210d026faf9955653af8916fad021475a3f00453", {
    "ARC-Challenge/train" + P: ("e488c1587ffdcfc8443f916c53488a95cd471c5790e0746c6bfe4cecf20962cb", 189909),
    "ARC-Challenge/validation" + P: ("395a5c88d1580d69855fbaee9450270578df1ad5af6259771cd0a42c20e99f05", 55743),
    "ARC-Challenge/test" + P: ("62f03257e737aed263f55c6abf87c7bb0028a44a6bdd2a26eb1279eb42c1d1e9", 203808),
    "ARC-Easy/train" + P: ("b315db8a4be597dc7daa50a4e70d48dd7c990c32085629e6ccd8c926beaa80b5", 330598),
    "ARC-Easy/validation" + P: ("ed890ff1e4cef7a7140d3a30dcea3ed2c9d467c6458f447ad9ef0176d8dcbb74", 86080),
    "ARC-Easy/test" + P: ("4160597d618ae851c7eb04e281574f3f654776216ac6b6641588d64527b47177", 346257)}, HF),
    "obqa": ("allenai/openbookqa", "388097ea7776314e93a529163e0fea805b8a6454", {
    "main/train" + P: ("98148f8a54e62eb862346a75192d5fb824d6cbb68f2f59aecd793d39ecb5cd8b", 495845),
    "main/validation" + P: ("35370b9cfee8c1ff325ccc74adc434d12c47ca0ac3244aa87f3fa77069285206", 58233),
    "main/test" + P: ("cd5483e366daa230c1c87bbdc512d8b7229f14f6dd04d19fc8b1a3855aaaa8a3", 55535),
    "additional/train" + P: ("d16d719e87efb86ed0a2ac4c8cdf380f7bfb94b602088393674c0a64ce9ed3d3", 635446),
    "additional/validation" + P: ("92e5e68e4da7bec7d130d925385abf377c2d82b89a16de502b4e1b9cf3f50a26", 75882),
    "additional/test" + P: ("33b318ea8e2354484868bc601c1b30a58149e9deb93162ff422bb8de980c7105", 72461)}, HF),
    "mmlu": ("cais/mmlu", "c30699e8356da336a370243923dbaf21066bb9fe", {
    "all/dev" + P: ("2b19bde1ed8ca6b482fb283abc90e8e0d9d228947029c0b91795d64b28b3bc3f", 76504),
    "all/validation" + P: ("66cdf0b090ccb657d18d13cd81e31fdc55c3467da9642ffb178653268a97c8ef", 408449),
    "all/test" + P: ("74a41822ce7d3def56e1682f958469c04642a5336a5ce912fa375fdb90fb25d7", 3504718)}, HF),
    "mmlupro": ("TIGER-Lab/MMLU-Pro", "b189ec765aa7ed75c8acfea42df31fdae71f97be", {
    "data/validation" + P: ("139423c23722e480c807ac4a191409a710cfce4eba744c1d641cf88e730e2078", 42857),
    "data/test" + P: ("0e24a191921c2f453518a537a8b2117bd137e7714d4ef1565e9ba06c1ecb9ad8", 4144185)}, HF),
}
KEEP = {"arc", "obqa", "mmlu", "mmlupro"}  # their files keep their folder: one name is in several folders
SEALED = {"source": "https://raw.githubusercontent.com/SapienzaNLP/wsd-hard-benchmark/"
                    "74ff055519b3c426330df9974a6603a0e638a3db/wsd_hard_benchmark/42D/",
          "42D.data.xml": ["45ae38a0f3758c03ca4a568e5d031ac0baa09784b70cb257bbd64e3a1be5cf1a", 334360],
          "42D.gold.key.txt": ["d6b1a81dec6f6418238a880727f033cca8a3317142a25012183483f731d53c78", 14741]}
ARCHIVES = {"wordnet.zip": ("wordnet/", "wordnet/"), "SimLex-999.zip": ("SimLex-999/", "simlex/"),
            "files.zip": ("files/", "thesaurus/"),
            "WordNet-3.0-glosstag.tar.bz2": ("WordNet-3.0/glosstag/", "glosstag/", ("standoff/", "dtd/"))}
# archive: (folder inside it, folder in DATA, subfolders left out)
SETS = {"semeval2007": "practice", "senseval2": "test", "senseval3": "test", "semeval2013": "test",
        "semeval2015": "test"}  # the five sets in ALL, by the start of their sentence ids


def role_of(source: str, rel: str):
    """A file's role, from its source and name; ALL holds five sets, so it gets a role for each."""
    name = rel.rsplit("/", 1)[-1]
    if source in ("wordnet", "encyclopedia", "thesaurus", "glosstag"):
        return "knowledge"
    if source in ("wordsim", "simlex"):
        return "test"
    if source == "wsd":
        return "fit" if "/SemCor/" in rel else dict(SETS)
    for split, role in (("train", "fit"), ("dev", "practice"), ("validation", "practice"), ("test", "test")):
        if split in name:
            return role
    raise ValueError(f"no role written for {source} {rel}")


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def size(n: float) -> str:
    for unit in ("B", "KB", "MB"):
        if n < 1024:
            return f"{n:.0f} {unit}"
        n /= 1024
    return f"{n:.1f} GB"


def origin(p: Path) -> str:
    """The input a path is in (Kaggle mounts inputs at <kind>/<owner>/<name>)."""
    parts = p.relative_to(INPUT).parts
    return parts[2] if parts[0] in ("notebooks", "datasets", "models") and len(parts) > 3 else parts[0]


def address(repo: str, commit: str, rel: str, template: str) -> str:
    """The file's address at that commit (MC_AI_FETCH_BASE, if set, replaces the host: for tests)."""
    return f"{BASE}{repo}/{commit}/{rel}" if BASE else template.format(repo=repo, commit=commit, file=rel)


def fetch(source: str, repo: str, commit: str, rel: str, want: str, nbytes, templates: list):
    """(path, how, address) of the file at repo/commit/rel, kept only with the SHA-256 wanted: already here,
    or copied from an input, or downloaded from the first address that gives those bytes."""
    name = rel.rsplit("/", 1)[-1]
    sub = rel.rsplit("/", 2)[-2] if source in KEEP else ""
    folder = DATA / ("_archives" if name in ARCHIVES else source) / sub
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / name
    first = address(repo, commit, rel, templates[0])
    if target.exists() and sha256(target) == want:
        return target, "already here", first
    for p in sorted(INPUT.rglob(name)) if INPUT.exists() else []:
        if p.is_file() and (nbytes is None or p.stat().st_size == nbytes) and sha256(p) == want:
            shutil.copyfile(p, target)
            return target, f"copied from {origin(p)}", str(p)
    tried, wrong = [], []
    for template in templates:
        url = address(repo, commit, rel, template)
        part = target.with_name(name + ".part")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": AGENT})
            with urllib.request.urlopen(req, timeout=120) as r, open(part, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
        except (urllib.error.URLError, OSError) as err:
            part.unlink(missing_ok=True)
            tried.append(f"{url}: {err}")
            continue
        got = sha256(part)
        if got != want:
            part.unlink()
            wrong.append(f"{url} gave SHA-256 {got[:16]}...")
            continue
        part.replace(target)
        return target, "downloaded" + (f" from {url.split('/')[2]}" if len(templates) > 1 else ""), url
    if wrong:
        return target, f"REFUSED: {'; '.join(wrong)}, not {want[:16]}...", first
    raise RuntimeError(f"could not download {name}: {'; '.join(tried)}. Is the internet on (Settings -> "
                       f"Internet)? Or attach the file (Add Input): it is found by its SHA-256")


def wanted(n: str, inside: str, skip: tuple) -> bool:
    if not n.startswith(inside) or n.endswith("/") or any(n[len(inside):].startswith(s) for s in skip):
        return False
    if n.startswith("/") or ".." in Path(n).parts:
        raise ValueError(f"archive entry outside its folder: {n}")
    return True


def unpack(archive: Path, inside: str, dest: str, skip: tuple = ()) -> int:
    """Unpack the entries under `inside` (not under `skip`) into DATA/dest; returns how many files. An entry
    with an absolute path or '..' is refused."""
    count = 0
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            for info in z.infolist():
                if wanted(info.filename, inside, skip):
                    out, count = DATA / dest / info.filename[len(inside):], count + 1
                    if not (out.exists() and out.stat().st_size == info.file_size):
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with z.open(info) as src, open(out, "wb") as dst:
                            shutil.copyfileobj(src, dst, 1 << 20)
        return count
    with tarfile.open(archive, "r:*") as t:
        for m in t:
            if m.isfile() and wanted(m.name, inside, skip):
                out, count = DATA / dest / m.name[len(inside):], count + 1
                if not (out.exists() and out.stat().st_size == m.size):
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with t.extractfile(m) as src, open(out, "wb") as dst:
                        shutil.copyfileobj(src, dst, 1 << 20)
    return count


def canonical(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)


def chain(path: Path):
    """The ledger's entries if every entry's number, link and SHA-256 are right (Brain 4's test), else None."""
    try:
        entries = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    except (OSError, ValueError):
        return None
    prev = ""
    for i, e in enumerate(entries):
        body = {k: v for k, v in e.items() if k != "sha"}
        if e.get("n") != i or e.get("prev") != prev or e.get("sha") != hashlib.sha256(
                canonical(body).encode()).hexdigest():
            return None
        prev = e["sha"]
    return entries


# ---------------- files: kept, copied or downloaded, and checked ----------------
RECORD, REFUSED, ROLES, FROM = [], [], {}, {}
print(f"\n  into {DATA}")
for source, (repo, commit, files, *rest) in SOURCES.items():
    templates = [rest[0]] if rest and isinstance(rest[0], str) else list(rest[0]) if rest else [GITHUB]
    print(f"\n  {source}: {repo} " + (f"at {commit[:8]}" if commit != "-" else "(no commit: pinned by SHA-256)"))
    for rel, pin in files.items():
        want, nbytes = (pin, None) if isinstance(pin, str) else pin
        t1 = time.time()
        path, how, url = fetch(source, repo, commit, rel, want, nbytes, templates)
        ok = not how.startswith("REFUSED")
        if not ok:
            REFUSED.append(rel)
        shown = ("/".join(rel.split("/")[-2:]) if source in KEEP else rel.rsplit("/", 1)[-1]).replace(P, ".parquet")
        print(f"    {shown:<36} {size(path.stat().st_size) if ok else '':>7}  {how} ({time.time() - t1:.1f}s)")
        if ok:
            kept = str(path.relative_to(DATA))
            RECORD.append(dict(source=source, url=url, repo=repo, commit=commit, file=kept, sha256=want,
                               bytes=path.stat().st_size))
            ROLES[kept], FROM[kept] = role_of(source, rel), source
            if path.name in ARCHIVES:
                n = unpack(path, *ARCHIVES[path.name])
                print(f"    {'':<36} {'':>7}  unpacked: {n} files in {DATA / ARCHIVES[path.name][1].rstrip('/')}")
(DATA / "sources.json").write_text(json.dumps(RECORD, indent=1))
(DATA / "roles.json").write_text(json.dumps(dict(roles=ROLES, sealed=SEALED, TEST4=[k for k, v in SETS.items()
                                                                                    if v == "test"]), indent=1))

# ---------------- roles ----------------
print("\n  ROLES (data/roles.json)")
for role in ("fit", "practice", "test", "knowledge"):
    names = [k for k, v in ROLES.items() if v == role or (isinstance(v, dict) and role in v.values())]
    print(f"    {role:<9} {len(names):>2} files: " + ", ".join(sorted({FROM[n] for n in names})))
print("    sealed     2 files: 42D.data.xml, 42D.gold.key.txt (pinned, not fetched)")
print("    ALL (wsd) by set: " + ", ".join(f"{k} {v}" for k, v in SETS.items()))

# ---------------- earlier ledgers ----------------
EARLIER.mkdir(parents=True, exist_ok=True)
print(f"\n  EARLIER LEDGERS (into {EARLIER})")
MISSING = []
for brain in ("mc_ai_brain4", "mc_ai_brain3"):
    target = EARLIER / f"{brain}_ledger.jsonl"
    here = chain(target) if target.exists() else None
    attached = [(p, chain(p)) for p in sorted(INPUT.rglob(f"{brain}/ledger.jsonl"))] if INPUT.exists() else []
    good = [(p, e) for p, e in attached if e]
    how = "kept from an earlier run" if here else ""
    if good:
        p, e = max(good, key=lambda pe: len(pe[1]))
        if here is None or len(e) > len(here):
            shutil.copyfile(p, target)
            here, how = e, f"copied from {origin(p)}"
    bad = len(attached) - len(good)
    if here is None:
        MISSING.append(brain)
        print(f"    {brain}: none here and none attached that verifies" + (f" ({bad} did not)" if bad else ""))
        continue
    looks = {}
    for e in here:
        if e.get("kind") == "look":
            looks.setdefault(e["split"], set()).add(e["model"])
    print(f"    {brain}: {len(here)} entries, chain verifies; {how}" + (f"; {bad} attached did not verify" if bad
                                                                       else ""))
    print(f"      head {here[-1]['sha']}")
    print("      looks: " + ", ".join(f"{k} {len(v)}" for k, v in sorted(looks.items())))

total = time.time() - T0
print(f"\n  {len(RECORD)} files kept, {size(sum(r['bytes'] for r in RECORD))}; refused: "
      f"{', '.join(REFUSED) if REFUSED else 'none'}")
print("\n  PREDICTIONS vs RESULTS")
print(f"    P1 every file has its SHA-256: {len(RECORD)} of {len(RECORD) + len(REFUSED)} -> "
      f"{'HELD' if not REFUSED else 'DID NOT HOLD'}")
print(f"    P2 under 5 minutes: {total:.0f}s -> {'HELD' if total <= 300 else 'DID NOT HOLD'}")
if REFUSED:
    raise RuntimeError(f"files with other bytes than pinned were not kept: {', '.join(REFUSED)}")
if "mc_ai_brain4" in MISSING:
    raise RuntimeError("Brain 4's ledger is not here: attach the MC AI Brain 4 notebook (Add Input) and run "
                       "this cell again; without it TEST4's looks would be counted from zero")
print(f"\n  ({total:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 0")


MC AI BRAIN 5 - CELL 0: DATA - PINNED, CHECKED, GIVEN ROLES

  into /kaggle/working/mc_ai_brain5/data

  ewt: UniversalDependencies/UD_English-EWT at 4a4d77f5
    en_ewt-ud-train.conllu                 14 MB  copied from mc-ai-brain-4 (1.3s)
    en_ewt-ud-dev.conllu                    2 MB  copied from mc-ai-brain-4 (0.6s)
    en_ewt-ud-test.conllu                   2 MB  copied from mc-ai-brain-4 (0.5s)

  wsd: akranlu/wsd-data at 697cb66c
    semcor.data.xml                        37 MB  copied from mc-ai-brain-4 (0.7s)
    semcor.gold.key.txt                     7 MB  copied from mc-ai-brain-4 (0.5s)
    ALL.data.xml                            1 MB  copied from mc-ai-brain-4 (0.4s)
    ALL.gold.key.txt                      317 KB  copied from mc-ai-brain-4 (0.4s)

  wordnet: nltk/nltk_data at 550b6625
    wordnet.zip                            10 MB  copied from mc-ai-brain-4 (0.6s)
                                                  unpacked: 18 files in /kaggle/working/mc_ai_brain5/

In [2]:
# =====================================================================
# MC AI BRAIN 5 - CELL 1: LIBRARY AND LEDGER - BRAIN 4'S CODE, CHECKED;
# THE LEDGER, CONTINUED
# Needs Cell 0. No internet, nothing trained.
# LIBRARY  Copies Brain 4's seven modules (mcai core, stats, read, book,
#   words, count, sense) from the attached MC AI Brain 4 notebook into
#   /kaggle/working/mcai, each only if its text is the code written for
#   Brain 4: the SHA-256 of the text with line ends trimmed, written here.
#   A file already there with that text is used as it is. Then imports
#   all seven and prints the versions of Python, NumPy and SciPy.
# LEDGER   Opens Brain 5's ledger, /kaggle/working/mc_ai_brain5/
#   ledger.jsonl. A new one starts as Brain 4's 348 entries (copied by
#   Cell 0 into earlier/). Each look in Brain 3's ledger (also in
#   earlier/) is added as a look by the model brain3:<its model>, on the
#   split Brain 4 names for the same data, so TEST4's count covers both
#   brains. Brain 5's keys start with B5 (Brain 4 used the plain ones).
# PREDICTIONS (written before running)
#   B5-K1 all seven modules match the code written for Brain 4
#   B5-L1 the ledger's first 348 entries are Brain 4's, unchanged
#   B5-L2 TEST4 (WSD) has 51 looks: Brain 4's 50 and Brain 3's 1
#   B5-T1 under 30 seconds
# =====================================================================
import hashlib, importlib, platform, shutil, sys, time
from pathlib import Path

import numpy, scipy

print("=" * 64); print("MC AI BRAIN 5 - CELL 1: LIBRARY AND LEDGER"); print("=" * 64)
T0 = time.time()
if "EARLIER" not in globals() or "chain" not in globals():
    raise RuntimeError("run Cell 0 first")
LIB = HOME / "mcai"
MODULES = {  # module: SHA-256 of its text with line ends trimmed, as written for Brain 4 (Cells 2-10)
    "core.py": "29b3fae477711f8f85eba914dd802f4353a1274b75f1c54430d6f7df60fc7b57",
    "stats.py": "fcefe440002348e2f24978eefedb9e2e5f7d693d8e91211291488afa241f378b",
    "read.py": "cf2c00f21cd4f7e1d5e4ca990bf04eb5a6ff79af98c9cf365271517a1d43c990",
    "book.py": "d5aa08db951fbe70495e4df52fa2191ebc3e810a07e4cbdb399cc6e80a2cacd4",
    "words.py": "5690e7300e8de75de778d7d84083e38963f94a741ea46175a24b6ce1f0404cfc",
    "count.py": "02ab5624f1a30dc760b24e0c5f3de88d1e6d3701ebeae3210fa17d386e20219c",
    "sense.py": "d0cbf80548d650eff5ef4b3313d2611efc3a9f6c0dac965ad7d196b58f406257",
}
SPLITS = {"WSD TEST4 (SE2, SE3, SE13, SE15)": "TEST4 (WSD)", "EWT test (XPOS tags)": "EWT test (XPOS)"}
# Brain 3's name for a split -> Brain 4's name for the same data; any other name is kept
B4, B3 = EARLIER / "mc_ai_brain4_ledger.jsonl", EARLIER / "mc_ai_brain3_ledger.jsonl"


def text_sha(path) -> str:
    """SHA-256 of a file's text with each line's trailing spaces and the file's final newlines trimmed: the same
    for the code as written and as pasted into a cell."""
    t = Path(path).read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n")
    return hashlib.sha256(("\n".join(line.rstrip() for line in t.split("\n")).rstrip() + "\n").encode()).hexdigest()


# ---------------- library ----------------
print(f"\n  LIBRARY into {LIB}")
LIB.mkdir(parents=True, exist_ok=True)
(LIB / "__init__.py").touch()
MATCH, PROBLEMS = 0, []
for name, want in MODULES.items():
    target, how = LIB / name, ""
    if target.exists() and text_sha(target) == want:
        how = "already here"
    else:
        seen = sorted(INPUT.rglob(f"mcai/{name}")) if INPUT.exists() else []
        for p in seen:
            if text_sha(p) == want:
                shutil.copyfile(p, target)
                how = f"copied from {origin(p)}"
                break
        else:
            how = (f"{len(seen)} attached, none with the text written for Brain 4" if seen else "not attached")
            PROBLEMS.append(name)
    MATCH += name not in PROBLEMS
    print(f"    {name:<10} {target.stat().st_size if target.exists() else 0:>7,} bytes  {how}")
if PROBLEMS:
    raise RuntimeError(f"not the code written for Brain 4: {', '.join(PROBLEMS)}. Send this output: those modules "
                       f"will be given as cells to paste")
if str(HOME) not in sys.path:
    sys.path.insert(0, str(HOME))
importlib.invalidate_caches()
for name in MODULES:
    importlib.reload(importlib.import_module(f"mcai.{name[:-3]}"))
from mcai.core import Ledger, at_most, between

print(f"    imported all {len(MODULES)}; Python {platform.python_version()}, NumPy {numpy.__version__}, "
      f"SciPy {scipy.__version__}")

# ---------------- ledger ----------------
B4_ENTRIES = chain(B4) if B4.exists() else None
if not B4_ENTRIES:
    raise RuntimeError("Brain 4's ledger is not in earlier/: attach MC AI Brain 4 (Add Input) and run Cell 0")
LEDGER = Ledger(WORK / "ledger.jsonl", earlier=[B4])
LEDGER.predict("B5-K1", "Brain 4 modules whose text is the code written for Brain 4", between(7, 7))
LEDGER.predict("B5-L1", "Brain 4's entries at the start of this ledger, unchanged", between(348, 348))
LEDGER.predict("B5-L2", "models scored on TEST4 (WSD), Brain 3's included", between(51, 51))
LEDGER.predict("B5-T1", "Cell 1 seconds", at_most(30))
B3_ENTRIES = chain(B3) if B3.exists() else None
added = 0
for e in B3_ENTRIES or []:
    if e.get("kind") == "look":
        split = SPLITS.get(e["split"], e["split"])
        before = LEDGER.looks(split)
        added += LEDGER.look(split, f"brain3:{e['model']}", note=f"carried from Brain 3's ledger, entry {e['n']} "
                             f"(head {B3_ENTRIES[-1]['sha'][:16]}); Brain 3 named the split {e['split']}") > before
same = 0
for a, b in zip(LEDGER.entries, B4_ENTRIES):
    if a != b:
        break
    same += 1
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-K1", MATCH)
LEDGER.result("B5-L1", same)
LEDGER.result("B5-L2", LEDGER.looks("TEST4 (WSD)"))
LEDGER.result("B5-T1", time.time() - T0)
print(f"\n  LEDGER {LEDGER.path}")
print(f"    {len(LEDGER.entries)} entries; starts with {same} of Brain 4's {len(B4_ENTRIES)} unchanged; "
      f"Brain 3's ledger: {'not in earlier/' if B3_ENTRIES is None else f'{len(B3_ENTRIES)} entries'}; "
      f"looks carried from it this run: {added}")
splits = sorted({e["split"] for e in LEDGER.entries if e["kind"] == "look"})
print("    looks: " + ", ".join(f"{s} {LEDGER.looks(s)}" for s in splits))
print(f"    chain verifies: {LEDGER.verify()}; head {LEDGER.head}")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 1")


MC AI BRAIN 5 - CELL 1: LIBRARY AND LEDGER

  LIBRARY into /kaggle/working/mcai
    core.py     16,234 bytes  copied from mc-ai-brain-4
    stats.py     6,788 bytes  copied from mc-ai-brain-4
    read.py     18,116 bytes  copied from mc-ai-brain-4
    book.py      4,337 bytes  copied from mc-ai-brain-4
    words.py     7,630 bytes  copied from mc-ai-brain-4
    count.py     7,763 bytes  copied from mc-ai-brain-4
    sense.py     7,302 bytes  copied from mc-ai-brain-4
    imported all 7; Python 3.13.18, NumPy 2.1.3, SciPy 1.16.3

  PREDICTIONS vs RESULTS
    B5-K1 Brain 4 modules whose text is the code written for Brain 4: 7 (predicted between 7 and 7) -> HELD
    B5-L1 Brain 4's entries at the start of this ledger, unchanged: 348 (predicted between 348 and 348) -> HELD
    B5-L2 models scored on TEST4 (WSD), Brain 3's included: 50 (predicted between 51 and 51) -> DID NOT HOLD
    B5-T1 Cell 1 seconds: 2.455 (predicted <= 30) -> HELD

  LEDGER /kaggle/working/mc_ai_brain5/ledger.jsonl
 

In [3]:
# =====================================================================
# MC AI BRAIN 5 - CELL 1B: FEVER DATA - PINNED, CHECKED, GIVEN ROLES
# Needs Cells 0 and 1 (put it right after Cell 1).
# FEVER 1.0 (Thorne et al. 2018), from fever.ai, CC BY-SA 3.0. A file is
# used where an attached input holds it (found by name, size and
# SHA-256; nothing is copied), else downloaded into data/fever (fever.ai,
# then the old Amazon S3 address; a broken download resumes).
# ROLES  train: fit; paper_dev: practice; paper_test: test;
#   shared_task_dev: test (it is paper_dev and paper_test together:
#   18,999 + 18,567 = 37,566 evidence rows); shared_task_dev_public and
#   shared_task_test: unlabelled (no gold); wiki-pages.zip: knowledge
#   (the June 2017 Wikipedia pages the labels were made against)
# The zip also holds a macOS "._" copy of each page file; those are left
# out. Pages are counted once per zip (kept in fever_pages.json).
# PREDICTIONS (written before running)
#   B5-D1 FEVER files kept with their SHA-256: 7 of 7
#   B5-D2 page files in wiki-pages.zip (macOS copies left out): 109
#   B5-D3 pages in them: 5,416,537
# =====================================================================
import json, time, urllib.request, zipfile
from pathlib import Path

print("=" * 64); print("MC AI BRAIN 5 - CELL 1B: FEVER DATA - PINNED, CHECKED, GIVEN ROLES"); print("=" * 64)
T0 = time.time()
for need in ("DATA", "WORK", "INPUT", "RECORD", "ROLES", "FROM", "SEALED", "SETS", "LEDGER", "sha256", "size"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0 and 1 first ({need} is missing)")
from mcai.core import between

FEVER_BASES = ["https://fever.ai/download/fever/", "https://s3-eu-west-1.amazonaws.com/fever.public/"]
FEVER_PINS = {
    "train.jsonl": ("eba7e8f87076753f8494718b9a857827af7bf73e76c9e4b75420207d26e588b6", 33_024_303, "fit"),
    "paper_dev.jsonl": ("41158707810008747946bf23471e82df53e77a513524b9e3ec1c2e674ef5ef8c", 2_168_767, "practice"),
    "paper_test.jsonl": ("fb7b0280a0adc2302bbb29bfb7af37274fa585de3171bcf908f180642d11d88e", 2_181_168, "test"),
    "shared_task_dev.jsonl": ("e89865bfe1b4dd054e03dd57d7241a6fde24862905f31117cf0cd719f7c78df7", 4_349_935,
                              "test"),
    "shared_task_dev_public.jsonl": ("acda01ae5ee7e75c73909a665f465cec20704ea26e9d676cd7423ff2c8ab0e8b",
                                     1_530_640, "unlabelled"),
    "shared_task_test.jsonl": ("76dd0872d8fa1f49efe1194fe8a88b7dd4c715c77d87a142b615d4be583e1e51", 1_599_159,
                               "unlabelled"),
    "wiki-pages.zip": ("4b06d95da6adf7fe02d2796176c670dacccb21348da89cba4c50676ab99665f2", 1_713_485_474,
                       "knowledge")}
for key, claim, rule in (("B5-D1", "FEVER files kept with their SHA-256 (of 7)", between(7, 7)),
                         ("B5-D2", "page files in wiki-pages.zip, macOS copies left out", between(109, 109)),
                         ("B5-D3", "pages in them", between(5_416_537, 5_416_537))):
    LEDGER.predict(key, claim, rule)
CHECKED = WORK / "fever_checked.json"            # SHA-256 already computed, by path, size and time
MEMO = json.loads(CHECKED.read_text()) if CHECKED.exists() else {}


def sha_of(p: Path) -> str:
    st = p.stat()
    key = f"{p}|{st.st_size}|{st.st_mtime_ns}"
    if key not in MEMO:
        MEMO[key] = sha256(p)
    return MEMO[key]


def fever_get(name: str, want: str, nbytes: int):
    """(path, how, address): kept here, used in an attached input, or downloaded and checked"""
    here = DATA / "fever" / name
    if here.exists() and here.stat().st_size == nbytes and sha_of(here) == want:
        return here, "already here", FEVER_BASES[0] + name
    for p in sorted(INPUT.rglob(name)) if INPUT.exists() else []:
        if p.is_file() and p.stat().st_size == nbytes and sha_of(p) == want:
            return p, f"used where it is ({p.parent})", FEVER_BASES[0] + name
    here.parent.mkdir(parents=True, exist_ok=True)
    tried = []
    for base in FEVER_BASES:
        url = base + name
        for attempt in range(3):
            have = here.stat().st_size if here.exists() else 0
            if have > nbytes:
                here.unlink()
                have = 0
            head = {"User-Agent": "MC_AI_Brain5/1.0", **({"Range": f"bytes={have}-"} if have else {})}
            try:
                with urllib.request.urlopen(urllib.request.Request(url, headers=head), timeout=60) as r:
                    mode = "ab" if have and r.status == 206 else "wb"
                    with open(here, mode) as f:
                        for b in iter(lambda: r.read(1 << 20), b""):
                            f.write(b)
                if here.stat().st_size == nbytes:
                    if sha_of(here) == want:
                        return here, "downloaded", url
                    here.unlink()
                    tried.append(f"{url}: other bytes than pinned")
                    break
            except Exception as err:
                tried.append(f"{url} try {attempt + 1}: {type(err).__name__}: {err}")
    return None, "REFUSED: " + " | ".join(tried[-3:]), FEVER_BASES[0] + name


def fever_parts(z: zipfile.ZipFile) -> list:
    """The page files in wiki-pages.zip, without the macOS '._' copies"""
    return sorted(n for n in z.namelist() if n.endswith(".jsonl") and not n.startswith("__MACOSX/")
                  and not n.rsplit("/", 1)[-1].startswith("._"))


FEVER, FEVER_REFUSED = {}, []
print("\n  fever: fever.ai (no commit: pinned by SHA-256)")
for name, (want, nbytes, role) in FEVER_PINS.items():
    t1 = time.time()
    path, how, url = fever_get(name, want, nbytes)
    print(f"    {name:<30} {size(nbytes) if path else '':>7}  {role:<10} {how} ({time.time() - t1:.1f}s)")
    if path is None:
        FEVER_REFUSED.append(name)
        continue
    FEVER[name] = path
    kept = str(path.relative_to(DATA)) if path.is_relative_to(DATA) else str(path)
    RECORD[:] = [r for r in RECORD if not (r["source"] == "fever" and r["file"].endswith(name))]
    RECORD.append(dict(source="fever", url=url, repo="fever.ai", commit="-", file=kept, sha256=want, bytes=nbytes))
    ROLES[kept], FROM[kept] = role, "fever"
CHECKED.write_text(json.dumps(MEMO, indent=1))
(DATA / "sources.json").write_text(json.dumps(RECORD, indent=1))
(DATA / "roles.json").write_text(json.dumps(dict(roles=ROLES, sealed=SEALED, TEST4=[k for k, v in SETS.items()
                                                                                    if v == "test"]), indent=1))
FEVER_COUNT = {}
if "wiki-pages.zip" in FEVER:
    counted = WORK / "fever_pages.json"
    FEVER_COUNT = json.loads(counted.read_text()) if counted.exists() else {}
    if FEVER_COUNT.get("sha256") != FEVER_PINS["wiki-pages.zip"][0]:
        t1 = time.time()
        with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
            parts = fever_parts(z)
            pages = 0
            for n in parts:
                with z.open(n) as f:
                    pages += sum(1 for line in f if line.strip())
            other = len([n for n in z.namelist() if n.endswith(".jsonl")]) - len(parts)
        FEVER_COUNT = dict(sha256=FEVER_PINS["wiki-pages.zip"][0], files=len(parts), pages=pages, macos=other)
        counted.write_text(json.dumps(FEVER_COUNT, indent=1))
        print(f"\n  wiki-pages.zip counted ({time.time() - t1:.0f}s)")
    print(f"  wiki-pages.zip: {FEVER_COUNT['files']} page files ({FEVER_COUNT['macos']} macOS copies left out), "
          f"{FEVER_COUNT['pages']:,} pages")
print("\n  ROLES (data/roles.json): " + ", ".join(f"{n} {r}" for n, (_, _, r) in FEVER_PINS.items()))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-D1", len(FEVER))
LEDGER.result("B5-D2", FEVER_COUNT.get("files", 0))
LEDGER.result("B5-D3", FEVER_COUNT.get("pages", 0))
if FEVER_REFUSED:
    raise RuntimeError(f"FEVER files not kept: {', '.join(FEVER_REFUSED)} (attach the MC AI FEVER notebook, "
                       f"Add Input, or turn the internet on)")
if "sync" in globals():
    sync("Cell 1B")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 1B")


MC AI BRAIN 5 - CELL 1B: FEVER DATA - PINNED, CHECKED, GIVEN ROLES

  fever: fever.ai (no commit: pinned by SHA-256)
    train.jsonl                      31 MB  fit        used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    paper_dev.jsonl                   2 MB  practice   used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.5s)
    paper_test.jsonl                  2 MB  test       used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.5s)
    shared_task_dev.jsonl             4 MB  test       used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.5s)
    shared_task_dev_public.jsonl      1 MB  unlabelled used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    shared_task_test.jsonl            2 MB  unlabelled used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.5s)
    wiki-pages.zip                  1.6 GB  knowledge  

In [4]:
# =====================================================================
# MC AI BRAIN 5 - SYNC: THE LEDGER TO GITHUB
# Run after Cell 1 in each session. Needs the notebook's internet and
# the Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets): a fine-grained
# token for Coffey333/mc-ai-brain5 with Contents: Read and write. The
# token is read from the secret, handed to git through environment
# variables for one command at a time, and never printed or written to
# a file.
# ACCESS CHECK, one line per step, stopping at the first that fails: the
#   secret can be read (its length and kind are printed, nothing of it
#   else); GitHub can be reached; the token can see the repository.
# SYNC  Clones the repository's main branch (one commit deep) into a
#   temporary folder outside /kaggle/working and puts into its kaggle/:
#   ledger.jsonl          Brain 5's ledger, only if its chain verifies
#   earlier/              the Brain 4 and Brain 3 ledgers it continues
#   data/                 roles.json and sources.json (no data file)
#   reports/              any report a cell wrote to mc_ai_brain5/reports
#   runs/<time>.json      this sync: time, machine, versions, the ledger's
#                         length and head, the SHA-256 of each file synced
#   Commits only if something changed, pushes, and reads back from
#   GitHub that main is the commit just pushed. If GitHub moved on
#   meanwhile, it clones again and tries once more.
# Defines sync(note) for later cells: the same steps; a failure is
# printed and returns False instead of stopping the cell. Any failure
# here stops this cell with the reason in the error itself.
# =====================================================================
import base64, json, os, platform, shutil, subprocess, tempfile, time, urllib.error, urllib.request
from pathlib import Path

import numpy

print("=" * 64); print("MC AI BRAIN 5 - SYNC: THE LEDGER TO GITHUB"); print("=" * 64)
if "LEDGER" not in globals() or "sha256" not in globals():
    raise RuntimeError("run Cells 0 and 1 first")
OWNER_REPO = "Coffey333/mc-ai-brain5"
REMOTE = os.environ.get("MC_AI_SYNC_REMOTE", f"https://github.com/{OWNER_REPO}.git")
BRANCH = "main"
SYNC_ERROR = ""


def _token() -> str:
    t = os.environ.get("MC_AI_GITHUB_TOKEN")  # for tests outside Kaggle
    if t:
        return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as err:
        raise RuntimeError(f"no Kaggle secret named GITHUB_TOKEN could be read ({type(err).__name__}): Add-ons -> "
                           f"Secrets; its label must be exactly GITHUB_TOKEN and its box checked for this "
                           f"notebook") from None


def _api(path: str, token: str = "") -> int | None:
    """HTTP status GitHub's API gives for path (None if GitHub cannot be reached); the token, if given, goes
    in the Authorization header."""
    head = {"Accept": "application/vnd.github+json", "User-Agent": "MC-AI-Brain5"}
    if token:
        head["Authorization"] = f"Bearer {token}"
    try:
        with urllib.request.urlopen(urllib.request.Request(f"https://api.github.com{path}", headers=head),
                                    timeout=30) as r:
            return r.status
    except urllib.error.HTTPError as err:
        return err.code
    except OSError:
        return None


def _git(args: list, cwd, token: str) -> str:
    """Run one git command; the token goes in as an HTTP header set through environment variables, and is cut
    out of anything git prints."""
    header = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
               GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {header}")
    r = subprocess.run(["git", *args], cwd=str(cwd), env=env, capture_output=True, text=True, timeout=300)
    out = (r.stdout + r.stderr).replace(token, "***").replace(header, "***")
    if r.returncode != 0:
        hint = {"clone": " (can the token read mc-ai-brain5? is the internet on?)",
                "push": " (the token can read but not write: GitHub -> the token -> Contents: Read and write)"}
        raise RuntimeError(f"git {args[0]} failed{hint.get(args[0], '')}: {out.strip()[-400:]}")
    return out


def check_access() -> str:
    """One line per step; raises at the first step that fails. Returns the token."""
    token = _token()
    kind = ("fine-grained" if token.startswith("github_pat_") else "classic" if token.startswith("ghp_")
            else "not a GitHub token shape")
    print(f"  1 secret GITHUB_TOKEN read: {len(token)} characters, {kind}")
    if "MC_AI_SYNC_REMOTE" in os.environ:  # tests: no GitHub
        return token
    status = _api(f"/repos/{OWNER_REPO}")
    if status is None:
        raise RuntimeError("GitHub cannot be reached: is the notebook's internet on? (Settings -> Internet)")
    print(f"  2 GitHub reached (without the token it answers {status}, as for any private repository)")
    status = _api(f"/repos/{OWNER_REPO}", token)
    if status == 401:
        raise RuntimeError("GitHub refuses the token (401): it is mistyped, expired or deleted; make a new one")
    if status != 200:
        raise RuntimeError(f"the token cannot see {OWNER_REPO} ({status}): on GitHub, edit the token -> "
                           f"Repository access -> Only select repositories -> add mc-ai-brain5")
    print(f"  3 the token can see {OWNER_REPO}")
    return token


def _attempt(note: str, token: str, tmp: Path):
    repo = tmp / "repo"
    shutil.rmtree(repo, ignore_errors=True)
    _git(["clone", "--quiet", "--depth", "1", "--branch", BRANCH, REMOTE, str(repo)], tmp, token)
    dest = repo / "kaggle"
    files = {"ledger.jsonl": WORK / "ledger.jsonl"}
    files.update({f"earlier/{p.name}": p for p in sorted(EARLIER.glob("*.jsonl"))})
    files.update({f"data/{n}": DATA / n for n in ("roles.json", "sources.json") if (DATA / n).exists()})
    if (WORK / "reports").exists():
        files.update({f"reports/{p.name}": p for p in sorted((WORK / "reports").glob("*.txt"))})
    for rel, src in files.items():
        (dest / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest / rel)
    if not _git(["status", "--porcelain", "--", "kaggle"], repo, token).strip():
        return None, files
    stamp = time.strftime("%Y-%m-%dT%H-%M-%SZ", time.gmtime())
    run = dict(time=stamp, note=note, machine=platform.platform(), cpus=os.cpu_count(),
               python=platform.python_version(), numpy=numpy.__version__, ledger_entries=len(LEDGER.entries),
               ledger_head=LEDGER.head, files={rel: sha256(dest / rel) for rel in files})
    (dest / "runs").mkdir(parents=True, exist_ok=True)
    (dest / "runs" / f"{stamp}.json").write_text(json.dumps(run, indent=1))
    _git(["add", "kaggle"], repo, token)
    message = f"Kaggle {stamp}: ledger {len(LEDGER.entries)} entries, head {LEDGER.head[:16]}"
    _git(["-c", "user.name=MC AI Brain 5 (Kaggle run)", "-c", "user.email=kaggle-run@localhost", "commit", "-q",
          "-m", message + (f"; {note}" if note else "")], repo, token)
    _git(["push", "--quiet", "origin", f"HEAD:{BRANCH}"], repo, token)
    mine = _git(["rev-parse", "HEAD"], repo, token).strip()
    there = _git(["ls-remote", "origin", f"refs/heads/{BRANCH}"], repo, token).split()[0]
    if there != mine:
        raise RuntimeError(f"GitHub's {BRANCH} is {there[:12]}, not the commit pushed ({mine[:12]})")
    return mine, files


def sync(note: str = "", token: str = "") -> bool:
    """Push the ledger and reports to kaggle/ in mc-ai-brain5. True if GitHub now holds them; on failure the
    reason is printed and kept in SYNC_ERROR."""
    global SYNC_ERROR
    t0 = time.time()
    try:
        if not LEDGER.verify():
            raise RuntimeError("the ledger's chain does not verify: nothing synced")
        token = token or _token()
        with tempfile.TemporaryDirectory() as tmp:
            for attempt in (1, 2):
                try:
                    commit, files = _attempt(note, token, Path(tmp))
                    break
                except RuntimeError as err:
                    if attempt == 2 or "git push failed" not in str(err):
                        raise
        if commit is None:
            print(f"  sync: GitHub already has these {len(files)} files (ledger head {LEDGER.head[:16]}) "
                  f"({time.time() - t0:.0f}s)")
        else:
            print(f"  sync: pushed {len(files)} files to kaggle/ on {BRANCH} as commit {commit[:12]}; GitHub "
                  f"confirms it ({len(LEDGER.entries)} entries, head {LEDGER.head[:16]}) ({time.time() - t0:.0f}s)")
        SYNC_ERROR = ""
        return True
    except Exception as err:
        SYNC_ERROR = str(err)
        print(f"  sync FAILED: {err}")
        return False


print(f"  {_git(['--version'], WORK, 'none').strip()}; repository {REMOTE}, branch {BRANCH}")
_TOKEN = None
try:
    _TOKEN = check_access()
    SYNCED = sync("sync cell", _TOKEN)
except RuntimeError as err:
    raise RuntimeError(f"access check: {err}") from None
finally:
    _TOKEN = None
if not SYNCED:
    raise RuntimeError(f"the sync failed: {SYNC_ERROR}")
print("  LAST LINE OF MC AI BRAIN 5 SYNC")


MC AI BRAIN 5 - SYNC: THE LEDGER TO GITHUB
  git version 2.43.0; repository https://github.com/Coffey333/mc-ai-brain5.git, branch main
  1 secret GITHUB_TOKEN read: 93 characters, fine-grained
  2 GitHub reached (without the token it answers 404, as for any private repository)
  3 the token can see Coffey333/mc-ai-brain5
  sync: pushed 4 files to kaggle/ on main as commit 88d06a29393e; GitHub confirms it (362 entries, head 39ae7bba05ca0a52) (3s)
  LAST LINE OF MC AI BRAIN 5 SYNC


In [5]:
# =====================================================================
# MC AI BRAIN 5 - CELL 2: WORD VECTORS AND DEFINITION VECTORS - BUILT
# ONCE, SAVED
# Needs Cells 0 and 1. No internet, nothing trained.
# Builds the vectors of Brain 4's Cells 14 and 16 with the same code,
# settings and seeds, and saves them with mcai.core's Artifacts (.npz and
# .json, never pickle) in mc_ai_brain5/artifacts, in a folder named by
# what they depend on: the data's SHA-256s and the settings below. A later
# session with the same dependencies loads them instead of building them.
# WORD VECTORS (Cell 14)  Simple Wikipedia, lower case, words of letters
#   only; the 50,000 commonest seen >= 5 times; the words 1 and 2 places
#   either side counted with weight 1 and 1/2, never across two articles;
#   PPMI with context counts to the power 0.75 (Levy, Goldberg & Dagan
#   2015); the 300 strongest directions by randomized SVD (Halko et al.
#   2011; seed 14, 5 power iterations), each scaled by the square root of
#   its singular value; rows at length 1
# DEFINITION VECTORS (Cell 16)  each WordNet synset: the SVD-PPMI vectors
#   of the words in its definition and examples (not its own lemmas), each
#   weighted a / (a + p), a = 0.001, p = the word's share of the counted
#   text; the synsets' first principal component removed; length 1: SIF
#   (Arora, Liang & Ma 2017)
# CHECK  Brain 4's synonym test, rebuilt with Cell 14's seed: 2,000 held-
#   out WordNet synonym questions, four choices each; answer = the most
#   similar word vector. Brain 4 scored 0.651 (its V1). Run every time, on
#   built or loaded vectors
# PREDICTIONS (written before running)
#   B5-V1 synonym test minus Brain 4's 0.651: exactly 0 (same data, code,
#         seeds, Python and NumPy as Brain 4's run)
#   B5-V2 the size of that difference: at most 0.002 (4 of 2,000)
#   B5-T2 the cell, when it builds: under 300 seconds
#   B5-T3 the cell, when it loads: under 60 seconds
# =====================================================================
import importlib, re, time

import numpy as np
import scipy
import scipy.sparse as sp
import sklearn
from sklearn.utils.extmath import randomized_svd

print("=" * 64); print("MC AI BRAIN 5 - CELL 2: WORD VECTORS AND DEFINITION VECTORS"); print("=" * 64)
T0 = time.time()
if "LEDGER" not in globals() or "RECORD" not in globals():
    raise RuntimeError("run Cells 0 and 1 first")
import mcai.book, mcai.read
importlib.reload(mcai.book); importlib.reload(mcai.read)
from mcai.book import read_encyclopedia
from mcai.core import Artifacts, at_most, between
from mcai.read import read_wordnet

LEDGER.predict("B5-V1", "synonym test minus Brain 4's V1 (0.651)", between(0, 0))
LEDGER.predict("B5-V2", "size of the synonym test's difference from Brain 4's", at_most(0.002))
LEDGER.predict("B5-T2", "Cell 2 seconds, building", at_most(300))
LEDGER.predict("B5-T3", "Cell 2 seconds, loading", at_most(60))
ARTS = Artifacts(WORK / "artifacts")
SETTINGS = dict(vocabulary=50000, min_count=5, window={"1": 1.0, "2": 0.5}, context_power=0.75, dims=300,
                svd_seed=14, svd_iterations=5, sif_a=0.001)
DEPENDS = dict(what="Brain 4 Cells 14 and 16 vectors, rebuilt in Brain 5 Cell 2", settings=SETTINGS,
               data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("encyclopedia", "wordnet")})
WN = read_wordnet(DATA / "wordnet")
SYNLIST = list(WN.synsets.values())


def unit(X: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return (X / np.where(n == 0, 1, n)).astype(np.float32)


def norm(w: str) -> str:
    return " ".join(w.lower().replace("_", " ").split())


def build() -> dict:
    """Brain 4 Cell 14's counting, PPMI and SVD, then Cell 16's definition vectors."""
    t1 = time.time()
    enc = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])
    type_id, chunks = {}, []
    letters = str.maketrans({c: " " for c in map(chr, range(128)) if not c.isalpha()})
    for a in enc.articles():
        toks = [t for t in a.text.lower().translate(letters).split() if t.isascii()]
        chunks.append(np.fromiter((type_id.setdefault(t, len(type_id)) for t in toks), np.int32, len(toks)))
        chunks.append(np.array([-1], np.int32))        # article boundary
    ids = np.concatenate(chunks)
    del chunks
    types = sorted(type_id, key=type_id.get)
    freq = np.bincount(ids[ids >= 0], minlength=len(types))
    top = [int(i) for i in np.argsort(-freq, kind="stable") if freq[i] >= SETTINGS["min_count"]]
    top = top[:SETTINGS["vocabulary"]]
    v = len(top)
    rank = np.full(len(types) + 1, -2, np.int32)       # -2: outside the vocabulary
    rank[-1] = -1                                       # ids -1 (boundary) read rank[-1]
    rank[np.array(top)] = np.arange(v)
    vocab, vfreq = [types[i] for i in top], freq[top]
    r_ids, tokens = rank[ids], int((ids >= 0).sum())
    del ids
    print(f"\n  COUNT: {tokens:,} words in {enc.groups[-1]:,} articles; {len(types):,} different words; vocabulary "
          f"{v:,} (seen >= {vfreq[-1]:,} times) covers {vfreq.sum() / tokens:.1%} of the text "
          f"({time.time() - t1:.0f}s)")
    t1 = time.time()
    m = sp.csr_matrix((v, v), dtype=np.float32)
    step = 20_000_000
    for s0 in range(0, len(r_ids), step):
        part = r_ids[s0:s0 + step + 2]
        rows, cols, wts = [], [], []
        for d, w in ((1, 1.0), (2, 0.5)):
            n = min(step, len(part) - d)
            a, b = part[:n], part[d:d + n]
            ok = (a >= 0) & (b >= 0)
            if d == 2:
                ok &= part[1:1 + n] != -1              # no pair across an article boundary
            a, b = a[ok], b[ok]
            rows += [a, b]; cols += [b, a]; wts.append(np.full(2 * len(a), w, np.float32))
        m = m + sp.coo_matrix((np.concatenate(wts), (np.concatenate(rows), np.concatenate(cols))),
                              shape=(v, v)).tocsr()
    del r_ids, rows, cols, wts
    row = np.asarray(m.sum(1)).ravel()
    col = np.asarray(m.sum(0)).ravel() ** SETTINGS["context_power"]
    coo = m.tocoo()
    del m
    pmi = np.log(coo.data) - np.log(row[coo.row]) - np.log(col[coo.col] / col.sum())
    keep = pmi > 0
    ppmi = sp.csr_matrix((pmi[keep].astype(np.float32), (coo.row[keep], coo.col[keep])), shape=(v, v))
    del coo, pmi, keep
    u, s, _ = randomized_svd(ppmi, n_components=min(SETTINGS["dims"], v - 1), n_iter=SETTINGS["svd_iterations"],
                             random_state=SETTINGS["svd_seed"])
    svd = unit(u * np.sqrt(s))
    print(f"    PPMI: {ppmi.nnz:,} positive pairs; SVD-PPMI {svd.shape[1]} numbers per word ({time.time() - t1:.0f}s)")
    del u, ppmi
    t1 = time.time()
    wrow = {w: i for i, w in enumerate(vocab)}
    gl = []
    for si, syn in enumerate(SYNLIST):
        own = {x for w in syn.words for x in norm(w).split()}
        for x in re.findall(r"[a-z]+", (syn.definition + " " + " ".join(syn.examples)).lower()):
            r = wrow.get(x, -1)
            if r >= 0 and x not in own:
                gl.append((si, r))
    gl = np.array(gl).T
    sif = (SETTINGS["sif_a"] / (SETTINGS["sif_a"] + vfreq / vfreq.sum())).astype(np.float32)
    g = sp.csr_matrix((sif[gl[1]], (gl[0], gl[1])), shape=(len(SYNLIST), v))
    synv = g @ svd
    has_def = np.abs(synv).sum(1) > 0
    pc = np.linalg.svd(synv[has_def], full_matrices=False)[2][0]
    synv = unit(np.where(has_def[:, None], synv - np.outer(synv @ pc, pc), 0))
    print(f"  DEFINITIONS: {int(has_def.sum()):,} of {len(SYNLIST):,} synsets have a definition vector "
          f"({time.time() - t1:.0f}s)")
    return dict(vocab=np.array(vocab), vfreq=vfreq.astype(np.int64), svd=svd,
                synsets=np.array([x.id for x in SYNLIST]), synv=synv, pc=pc.astype(np.float32), has_def=has_def,
                tokens=np.array(tokens), articles=np.array(enc.groups[-1]))


VEC, BUILT = ARTS.get("vectors", DEPENDS, build)
if list(VEC["synsets"]) != [x.id for x in SYNLIST]:
    raise RuntimeError("the saved definition vectors are for another order of synsets: delete mc_ai_brain5/"
                       "artifacts and run this cell again")
VOCAB, VFREQ, SVD, SYNV = [str(w) for w in VEC["vocab"]], VEC["vfreq"], VEC["svd"], VEC["synv"]
V, WROW = len(VOCAB), {w: i for i, w in enumerate(VEC["vocab"].tolist())}
SID = {s: i for i, s in enumerate(VEC["synsets"].tolist())}
print(f"\n  VECTORS {'built and saved' if BUILT else 'loaded'}: {V:,} words x {SVD.shape[1]}, {SYNV.shape[0]:,} "
      f"synsets x {SYNV.shape[1]}; {int(VEC['tokens']):,} words counted in {int(VEC['articles']):,} articles "
      f"({time.time() - T0:.0f}s)")

# ---------------- check: Brain 4's synonym test (Cell 14), same seed ----------------
t1 = time.time()
RNG = np.random.default_rng(14)
HASPOS = {p: np.array([bool(WN.senses.get(f"{w}.{p}")) for w in VOCAB]) for p in "nvar"}
PAIRPOS = {}
for s in WN.synsets.values():
    ws = sorted({x.lower() for x in s.words if x.isalpha() and x.lower() in WROW})
    for i, a in enumerate(ws):
        for b in ws[i + 1:]:
            PAIRPOS.setdefault((WROW[a], WROW[b]), s.id[0])
PAIRS = np.array(sorted(PAIRPOS))
HELD = RNG.random(len(PAIRS)) < 0.2
TEST = PAIRS[HELD]
LINKED = {int(i): set() for i in range(V)}
for s in WN.synsets.values():
    near = {s.id} | {p.target for p in s.pointers if p.symbol in ("@", "@i", "~", "~i")}
    parents = [p.target for p in s.pointers if p.symbol in ("@", "@i")]
    near |= {p.target for q in parents for p in WN.synsets[q].pointers if p.symbol in ("~", "~i")}
    words = {WROW[x.lower()] for n in near for x in WN.synsets[n].words if x.lower() in WROW}
    for x in s.words:
        if x.lower() in WROW:
            LINKED[WROW[x.lower()]] |= words
QUESTIONS = []
for k in RNG.permutation(len(TEST)):
    w, s = (int(x) for x in TEST[k][RNG.permutation(2)])
    pos = PAIRPOS[tuple(TEST[k])]
    if VFREQ[w] < 50 or VFREQ[s] < 50:
        continue
    lo, hi = max(0, s - 200), min(V, s + 200)
    pool = [x for x in RNG.permutation(np.arange(lo, hi)).tolist()
            if x not in (w, s) and x not in LINKED[w] and VFREQ[x] >= 50 and HASPOS[pos][x]][:3]
    if len(pool) == 3:
        QUESTIONS.append((w, s, pool))
    if len(QUESTIONS) == 2000:
        break
RIGHT = np.array([([s] + pool)[int(np.argmax(SVD[[s] + pool] @ SVD[w]))] == s for w, s, pool in QUESTIONS])
SCORE = float(RIGHT.mean())
print(f"\n  SYNONYM TEST: {len(PAIRS):,} synonym pairs in the vocabulary, {len(TEST):,} held out; {len(QUESTIONS):,} "
      f"questions; {int(RIGHT.sum()):,} right = {SCORE:.4f} (Brain 4: 0.651) ({time.time() - t1:.0f}s)")
for w, s, pool in QUESTIONS[:3]:
    pick = ([s] + pool)[int(np.argmax(SVD[[s] + pool] @ SVD[w]))]
    print(f"    {VOCAB[w]}: {', '.join(VOCAB[x] for x in [s] + pool)} -> {VOCAB[pick]} "
          f"({'right' if pick == s else 'wrong'})")
print(f"  NumPy {np.__version__}, SciPy {scipy.__version__}, scikit-learn {sklearn.__version__}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-V1", round(SCORE - 0.651, 6))
LEDGER.result("B5-V2", round(abs(SCORE - 0.651), 6))
LEDGER.result("B5-T2" if BUILT else "B5-T3", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 2")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 2")


MC AI BRAIN 5 - CELL 2: WORD VECTORS AND DEFINITION VECTORS

  COUNT: 39,324,314 words in 241,787 articles; 522,412 different words; vocabulary 50,000 (seen >= 22 times) covers 96.4% of the text (45s)
    PPMI: 14,032,154 positive pairs; SVD-PPMI 300 numbers per word (50s)
  DEFINITIONS: 117,228 of 117,659 synsets have a definition vector (7s)

  VECTORS built and saved: 50,000 words x 300, 117,659 synsets x 300; 39,324,314 words counted in 241,787 articles (106s)

  SYNONYM TEST: 18,132 synonym pairs in the vocabulary, 3,669 held out; 2,000 questions; 1,302 right = 0.6510 (Brain 4: 0.651) (5s)
    folk: sept, vasari, bending, lapland -> bending (wrong)
    restore: repair, slang, imagine, vacuum -> repair (right)
    civilized: cultivated, seasonal, thirteenth, digestive -> cultivated (right)
  NumPy 2.1.3, SciPy 1.16.3, scikit-learn 1.6.1

  PREDICTIONS vs RESULTS
    B5-V1 synonym test minus Brain 4's V1 (0.651): 0 (predicted between 0 and 0) -> HELD
    B5-V2 size of the synonym te

In [6]:
# =====================================================================
# MC AI BRAIN 5 - CELL 3: WHICH MEANING - BRAIN 4'S POOLED (hand) MODEL,
# BUILT ONCE, SAVED, CHECKED ON SEMEVAL-2007
# Needs Cells 0-2. No internet, nothing trained. TEST4 is not scored here.
# Rebuilds Brain 4's word-sense model of Cells 18, 19 and 21 with their
# code, settings and seeds, and saves it with Artifacts (mc_ai_brain5/
# artifacts; .npz and .json, never pickle). A later session loads it.
# TASK     each tagged word of SemCor's text and of ALL (Raganato et al.
#          2017) gets one WordNet synset; SemEval-2007 (SE07, 455 words)
#          is the practice set that chooses the weights
# CONTEXT  the sentence's content lemmas: their SVD-PPMI vectors (Cell 2),
#          weighted a / (a + p), the definition vectors' common direction
#          removed, length 1
# SCORE of each candidate synset, all from counting and the dictionary:
#   (g + e) / 2 + beta x prior + g1 x ow + g2 x om
#   g      cosine of the context with the definition vector (Cell 2)
#   e      cosine with the synset's stored examples: the sum of the context
#          vectors of every word tagged with it in SemCor and, by hand, in
#          the Princeton WordNet Gloss Corpus (POOLED (hand)); a synset with
#          no example uses its definition vector
#   prior  log((times SemCor tagged it for the lemma + 1) / (times any
#          candidate was tagged + candidates))
#   ow     WORDS AT PLACES: FHRR, a random phasor per lemma and per place
#          -3 ... +3 (D = 1,024, seeds 19 and 21); context = sum of place x
#          lemma; cosine with the synset's stored sum
#   om     MEANINGS AT PLACES: the SVD-PPMI vectors of the words 2 either
#          side, one block per place; cosine with the synset's stored sum
# WEIGHTS  chosen on SE07 only, as Brain 4 chose them: beta from 0, 0.01,
#          0.02, 0.05, 0.1, 0.2, 0.5 (SemCor examples, no order); then g1
#          and g2 from 0, 0.25, 0.5, 1, 2 (ties to the smaller sum)
# BRAIN 4 ON SE07 (its printed output): BOTH + PRIOR 62.2, ALL 64.6,
#          POOLED (hand) 63.5; beta 0.1, g1 0.5, g2 0.25
# PREDICTIONS (written before running)
#   B5-S1 beta chosen on SE07: 0.1
#   B5-S2 g1 chosen on SE07: 0.5
#   B5-S3 g2 chosen on SE07: 0.25
#   B5-S4 SE07 words right, BOTH + PRIOR: 283 of 455 (Brain 4's 62.2%)
#   B5-S5 SE07 words right, ALL: 294 of 455 (64.6%)
#   B5-S6 SE07 words right, POOLED (hand): 289 of 455 (63.5%)
#   B5-S7 synsets with stored examples, POOLED (hand): 36,615
#   B5-T4 the cell, when it builds: under 600 seconds
#   B5-T5 the cell, when it loads: under 90 seconds
# =====================================================================
import importlib, time

import numpy as np
import scipy.sparse as sp

print("=" * 64); print("MC AI BRAIN 5 - CELL 3: WHICH MEANING - POOLED (hand)"); print("=" * 64)
T0 = time.time()
for need in ("SVD", "SYNV", "SID", "WROW", "ARTS", "WN"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 first ({need} is missing)")
import mcai.sense, mcai.words
importlib.reload(mcai.sense); importlib.reload(mcai.words)
from mcai.core import at_most, between
from mcai.read import read_wsd
from mcai.sense import SenseCounts, candidates, items_of
from mcai.words import read_glosstag

for key, claim, rule in (("B5-S1", "beta chosen on SE07", between(0.1, 0.1)),
                         ("B5-S2", "g1 chosen on SE07", between(0.5, 0.5)),
                         ("B5-S3", "g2 chosen on SE07", between(0.25, 0.25)),
                         ("B5-S4", "SE07 words right of 455, BOTH + PRIOR", between(283, 283)),
                         ("B5-S5", "SE07 words right of 455, ALL", between(294, 294)),
                         ("B5-S6", "SE07 words right of 455, POOLED (hand)", between(289, 289)),
                         ("B5-S7", "synsets with stored examples, POOLED (hand)", between(36615, 36615)),
                         ("B5-T4", "Cell 3 seconds, building", at_most(600)),
                         ("B5-T5", "Cell 3 seconds, loading", at_most(90))):
    LEDGER.predict(key, claim, rule)
SIF = (0.001 / (0.001 + VFREQ / VFREQ.sum())).astype(np.float32)
PC = VEC["pc"]
ODIM, PLACES, MPLACES = 1024, (-3, -2, -1, 1, 2, 3), (-2, -1, 1, 2)
BETAS, GRID = (0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5), (0, 0.25, 0.5, 1, 2)

t1 = time.time()
TEXTS = {n: read_wsd(DATA / "wsd" / f"{n}.data.xml", DATA / "wsd" / f"{n}.gold.key.txt") for n in ("semcor", "ALL")}
TRAIN, ITEMS = items_of(TEXTS["semcor"], WN), items_of(TEXTS["ALL"], WN)
SET_OF = np.array([it.id.split(".")[0] for it in ITEMS])
DEV = SET_OF == "semeval2007"
WHERE = {t.instance: (toks, k) for name in ("semcor", "ALL") for toks in TEXTS[name].sentences.values()
         for k, t in enumerate(toks) if t.instance}
print(f"\n  TEXT ({time.time() - t1:.0f}s): {len(TRAIN):,} SemCor words; {len(ITEMS):,} words in ALL, {int(DEV.sum())} "
      f"of them SE07 (practice); the other {int((~DEV).sum()):,} are TEST4 and are not scored here")


def cunit(Z: np.ndarray) -> np.ndarray:
    n = np.sqrt((np.abs(Z) ** 2).sum(1, keepdims=True))
    return Z / np.where(n == 0, 1, n)


def contexts(items) -> np.ndarray:
    """One context vector per item (rows), from its content lemmas, as Cell 2 made definition vectors."""
    rows, cols = [], []
    for i, it in enumerate(items):
        for lem in it.context:
            r = WROW.get(lem, -1)
            if r >= 0:
                rows.append(i); cols.append(r)
    A = sp.csr_matrix((SIF[np.array(cols, int)], (np.array(rows, int), np.array(cols, int))), shape=(len(items), V))
    C = A @ SVD
    C -= np.outer(C @ PC, PC)
    return unit(C)


def around(toks, k) -> list:
    """The lemma at each place -3 ... +3 (None outside the sentence or not a word)."""
    out = []
    for d in PLACES:
        j = k + d
        lem = toks[j].lemma.lower() if 0 <= j < len(toks) else ""
        out.append(lem if lem.replace("'", "").isalpha() else None)
    return out


def ids_of(arounds) -> np.ndarray:
    return np.array([[LID.get(x, -1) if x else -1 for x in a] for a in arounds])


def words_at_places(ids: np.ndarray) -> np.ndarray:
    """Sum over places of place x lemma (FHRR binding), one row per word."""
    return sum(PLACEV[p][None, :] * LEMV[ids[:, p]] for p in range(len(PLACES)))


def meanings_at_places(arounds) -> np.ndarray:
    X = np.zeros((len(arounds), len(MPLACES) * SVD.shape[1]), np.float32)
    for i, a in enumerate(arounds):
        for b, d in enumerate(MPLACES):
            lem = a[PLACES.index(d)]
            r = WROW.get(lem, -1) if lem else -1
            if r >= 0:
                X[i, b * SVD.shape[1]:(b + 1) * SVD.shape[1]] = SVD[r]
    return X


def stored(items, ctx) -> dict:
    """The synsets these items are examples of, and per synset the sums of their context, order-word and
    order-meaning vectors (each example at length 1)."""
    pairs = np.array([(SID[s], i) for i, it in enumerate(items) for s in it.gold if s in SID])
    rows = np.unique(pairs[:, 0])
    pos = np.full(len(SID), -1)
    pos[rows] = np.arange(len(rows))
    T = sp.csr_matrix((np.ones(len(pairs), np.float32), (pos[pairs[:, 0]], pairs[:, 1])), shape=(len(rows), len(items)))
    E = T @ ctx
    W = np.zeros((len(rows), ODIM), np.complex64)
    M = np.zeros((len(rows), len(MPLACES) * SVD.shape[1]), np.float32)
    arounds = [around(*WHERE[it.id]) for it in items]
    for c0 in range(0, len(items), 20000):
        c1 = min(c0 + 20000, len(items))
        Z = cunit(words_at_places(ids_of(arounds[c0:c1])))
        W += (T[:, c0:c1] @ Z.real) + 1j * (T[:, c0:c1] @ Z.imag)
        M += T[:, c0:c1] @ unit(meanings_at_places(arounds[c0:c1]))
    return dict(rows=rows, E=E, W=W, M=M)


def combine(srcs: list) -> dict:
    """Stored examples of several sources summed per synset; a synset with none keeps its definition vector."""
    rows = np.unique(np.concatenate([s["rows"] for s in srcs]))
    pos = np.full(len(SID), -1)
    pos[rows] = np.arange(len(rows))
    out = {k: np.zeros((len(rows),) + srcs[0][k].shape[1:], srcs[0][k].dtype) for k in ("E", "W", "M")}
    for s in srcs:
        for k in ("E", "W", "M"):
            out[k][pos[s["rows"]]] += s[k]
    exv = SYNV.copy()
    exv[rows] = unit(out["E"])
    return dict(pos=pos, n=len(rows), EXV=exv, W=cunit(out["W"]).astype(np.complex64), M=unit(out["M"]))


def parts(counts_of) -> list:
    """Per word of ALL: candidate synsets, their rows, definition cosines and log priors."""
    out = []
    for k, it in enumerate(ITEMS):
        cands = candidates(WN, it)
        ids = np.array([SID[s] for s in cands])
        n = np.array([counts_of(it.lemma, s) for s in cands], float)
        out.append((cands, ids, SYNV[ids] @ C_ALL[k], np.log((n + 1) / (n.sum() + len(cands)))))
    return out


def right(v: dict, beta: float, g1: float, g2: float, mask: np.ndarray) -> np.ndarray:
    """1 where the best-scoring candidate is a gold synset, for the words in mask."""
    out = np.zeros(len(ITEMS), bool)
    for k in np.flatnonzero(mask):
        cands, ids, g, prior = PARTS[k]
        e = v["EXV"][ids] @ C_ALL[k]
        j = v["pos"][ids]
        ow = np.where(j >= 0, (v["W"][np.maximum(j, 0)] @ np.conj(O_W[k])).real, 0.0) if g1 else 0.0
        om = np.where(j >= 0, v["M"][np.maximum(j, 0)] @ O_M[k], 0.0) if g2 else 0.0
        out[k] = cands[int(np.argmax((g + e) / 2 + beta * prior + g1 * ow + g2 * om))] in ITEMS[k].gold
    return out


def build() -> dict:
    """Brain 4 Cells 18, 19 and 21 in one pass; the weights chosen on SE07; the POOLED (hand) arrays."""
    global LID, LEMV, PLACEV, C_ALL, O_W, O_M, PARTS
    t1 = time.time()
    counts = SenseCounts.count(TRAIN)
    orng = np.random.default_rng(19)
    at_train, at_all = [around(*WHERE[it.id]) for it in TRAIN], [around(*WHERE[it.id]) for it in ITEMS]
    lemmas = sorted({x for a in at_train + at_all for x in a if x})
    LID = {x: i for i, x in enumerate(lemmas)}
    LEMV = np.exp(1j * orng.uniform(0, 2 * np.pi, (len(lemmas) + 1, ODIM))).astype(np.complex64)
    LEMV[-1] = 0                                      # no word at that place
    PLACEV = np.exp(1j * orng.uniform(0, 2 * np.pi, (len(PLACES), ODIM))).astype(np.complex64)
    gfiles = sorted(p for p in (DATA / "glosstag").rglob("*.xml") if p.name in ("noun.xml", "verb.xml", "adj.xml",
                                                                                 "adv.xml"))
    gc = read_glosstag(gfiles)
    g_items = [it for it in items_of(gc.text, WN) if any(s in SID for s in it.gold)]
    g_hand = [it for it in g_items if it.id in gc.manual]
    WHERE.update({t.instance: (toks, k) for toks in gc.text.sentences.values() for k, t in enumerate(toks)
                  if t.instance})
    newlem = sorted({x for it in g_items for x in around(*WHERE[it.id]) if x} - set(LID))
    LID.update({x: len(lemmas) + i for i, x in enumerate(newlem)})
    lemmas = lemmas + newlem
    LEMV = np.vstack([LEMV[:-1], np.exp(1j * np.random.default_rng(21).uniform(0, 2 * np.pi, (len(newlem), ODIM)))
                      .astype(np.complex64), np.zeros((1, ODIM), np.complex64)])
    print(f"  GLOSS CORPUS ({time.time() - t1:.0f}s): {len(gc.text.sentences):,} definitions and examples; "
          f"{len(g_hand):,} words tagged by hand; {len(lemmas):,} lemmas at places")
    t1 = time.time()
    semcor = stored(TRAIN, contexts(TRAIN))
    hand = stored(g_hand, contexts(g_hand))
    only, pooled = combine([semcor]), combine([semcor, hand])
    del semcor, hand
    C_ALL = contexts(ITEMS)
    at = [around(*WHERE[it.id]) for it in ITEMS]
    O_W, O_M = cunit(words_at_places(ids_of(at))), unit(meanings_at_places(at))
    PARTS = parts(lambda lem, s: counts.tagged[(lem, s)])
    print(f"  STORED EXAMPLES ({time.time() - t1:.0f}s): synsets with examples, SemCor {only['n']:,}; SemCor and "
          f"gloss by hand {pooled['n']:,}")
    dev_b = {b: int(right(only, b, 0, 0, DEV).sum()) for b in BETAS}
    beta = max(BETAS, key=lambda b: (round(100 * dev_b[b] / DEV.sum(), 6), -b))
    dev_g = {(a, b): int(right(only, beta, a, b, DEV).sum()) for a in GRID for b in GRID}
    g1, g2 = max(dev_g, key=lambda ab: (round(100 * dev_g[ab] / DEV.sum(), 6), -(ab[0] + ab[1])))
    print("  BETA on SE07 (words right of 455): " + ", ".join(f"{b:g} -> {n}" for b, n in dev_b.items()) +
          f"; chosen {beta:g}\n  g1, g2 on SE07: best {g1:g}, {g2:g} -> {dev_g[g1, g2]} (no order: {dev_g[0, 0]})")
    keys = sorted(counts.tagged)
    return dict(exv=pooled["EXV"], w=pooled["W"], m=pooled["M"], pos=pooled["pos"].astype(np.int32),
                n=np.array(pooled["n"]), lemmas=np.array(lemmas), lemv=LEMV, placev=PLACEV,
                weights=np.array([beta, g1, g2]), dev_both=np.array(dev_b[beta]), dev_all=np.array(dev_g[g1, g2]),
                tagged_lemma=np.array([k[0] for k in keys]), tagged_synset=np.array([k[1] for k in keys]),
                tagged_n=np.array([counts.tagged[k] for k in keys], np.int64))


DEPENDS3 = dict(what="Brain 4 Cells 18, 19 and 21: POOLED (hand), rebuilt in Brain 5 Cell 3",
                vectors=ARTS.key("vectors", DEPENDS), places=[ODIM, list(PLACES), list(MPLACES)], seeds=[19, 21],
                betas=list(BETAS), grid=list(GRID),
                data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("wsd", "wordnet", "glosstag")})
MODEL, BUILT3 = ARTS.get("pooled_hand", DEPENDS3, build)
BETA, G1, G2 = (float(x) for x in MODEL["weights"])
POOLED = dict(EXV=MODEL["exv"], W=MODEL["w"], M=MODEL["m"], pos=MODEL["pos"], n=int(MODEL["n"]))
if not BUILT3:
    LID = {str(x): i for i, x in enumerate(MODEL["lemmas"].tolist())}
    LEMV, PLACEV = MODEL["lemv"], MODEL["placev"]
    C_ALL = contexts(ITEMS)
    at = [around(*WHERE[it.id]) for it in ITEMS]
    O_W, O_M = cunit(words_at_places(ids_of(at))), unit(meanings_at_places(at))
    TAGGED = {(str(a), str(b)): int(n) for a, b, n in zip(MODEL["tagged_lemma"].tolist(),
                                                          MODEL["tagged_synset"].tolist(), MODEL["tagged_n"])}
    PARTS = parts(lambda lem, s: TAGGED.get((lem, s), 0))
DEV_POOLED = int(right(POOLED, BETA, G1, G2, DEV).sum())
print(f"\n  MODEL {'built and saved' if BUILT3 else 'loaded'}: POOLED (hand), {POOLED['n']:,} synsets with stored "
      f"examples; beta {BETA:g}, g1 {G1:g}, g2 {G2:g} ({time.time() - T0:.0f}s)")
print(f"  SE07, words right of {int(DEV.sum())}: BOTH + PRIOR {int(MODEL['dev_both'])} "
      f"({100 * int(MODEL['dev_both']) / DEV.sum():.1f}%), ALL {int(MODEL['dev_all'])} "
      f"({100 * int(MODEL['dev_all']) / DEV.sum():.1f}%), POOLED (hand) {DEV_POOLED} "
      f"({100 * DEV_POOLED / DEV.sum():.1f}%); Brain 4: 62.2, 64.6, 63.5")

print("\n  PREDICTIONS vs RESULTS")
for key, value in (("B5-S1", BETA), ("B5-S2", G1), ("B5-S3", G2), ("B5-S4", int(MODEL["dev_both"])),
                   ("B5-S5", int(MODEL["dev_all"])), ("B5-S6", DEV_POOLED), ("B5-S7", POOLED["n"])):
    LEDGER.result(key, value)
LEDGER.result("B5-T4" if BUILT3 else "B5-T5", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 3")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 3")


MC AI BRAIN 5 - CELL 3: WHICH MEANING - POOLED (hand)

  TEXT (9s): 226,036 SemCor words; 7,253 words in ALL, 455 of them SE07 (practice); the other 6,798 are TEST4 and are not scored here
  GLOSS CORPUS (39s): 169,336 definitions and examples; 290,582 words tagged by hand; 37,927 lemmas at places
  STORED EXAMPLES (65s): synsets with examples, SemCor 25,942; SemCor and gloss by hand 36,615
  BETA on SE07 (words right of 455): 0 -> 210, 0.01 -> 236, 0.02 -> 251, 0.05 -> 279, 0.1 -> 283, 0.2 -> 277, 0.5 -> 268; chosen 0.1
  g1, g2 on SE07: best 0.5, 0.25 -> 294 (no order: 283)

  MODEL built and saved: POOLED (hand), 36,615 synsets with stored examples; beta 0.1, g1 0.5, g2 0.25 (118s)
  SE07, words right of 455: BOTH + PRIOR 283 (62.2%), ALL 294 (64.6%), POOLED (hand) 289 (63.5%); Brain 4: 62.2, 64.6, 63.5

  PREDICTIONS vs RESULTS
    B5-S1 beta chosen on SE07: 0.1 (predicted between 0.1 and 0.1) -> HELD
    B5-S2 g1 chosen on SE07: 0.5 (predicted between 0.5 and 0.5) -> HELD
    B5-S

In [7]:
# =====================================================================
# MC AI BRAIN 5 - CELL 4: THE THESAURUS CHECK - DOES A WORD THAT COULD
# STAND IN FOR EACH MEANING FIT THE SENTENCE?
# Needs Cells 0-3 in the same session. No internet, nothing trained.
# Mark's process: look the word up; if a meaning's definition does not
# fit, take a thesaurus word for it and check that word against the
# sentence. The published form is the substitution test with WordNet
# relatives (Leacock, Chodorow & Miller 1998).
# RELATIVES of a candidate synset: the one-word lemmas of the synset and
#   of the synsets one hypernym, hyponym or similar-to link away, the
#   target word itself left out, each with a word vector (Cell 2).
#   ALL: every relative; MONO: only relatives with one WordNet meaning
# FIT of a relative, from counted text only:
#   CONTEXT  cosine of its word vector with the sentence's context vector
#   PLACES   mean cosine with the words 1 and 2 places either side
#   a meaning's SUB = its best-fitting relative's fit; a meaning with no
#   relative gets the mean of the other candidates' SUB
# SCORE    POOLED (hand)'s score (Cell 3) + g3 x SUB
# PRACTICE SET  Brain 4 Cell 23's: every 5th SemCor document (sorted ids)
#   held out (45,727 words); the model rebuilt from the other documents
#   and the hand-tagged gloss examples, with Cell 3's weights. Brain 4:
#   73.2% right. The variant (CONTEXT or PLACES, ALL or MONO) and g3 (0.25,
#   0.5, 1, 2, 4) are chosen here, most words right, ties to smaller g3
# TEST4    on Kaggle only (never in the lab): POOLED (hand) scored again
#   under Brain 4's own name for it (the same model: not a new look); the
#   chosen variant once (one new look) only if B5-U1 and B5-U2 both held
# NOT BLIND  the lab ran this cell on GitHub (practice data only) before
#   Kaggle; the predictions below were committed to the repository first
#   (commit 7027a01). The gain did not hold there (+0.04 points), so the
#   rule above, added after that run, keeps the variant off TEST4
# PREDICTIONS (written before running)
#   B5-H1 held-out SemCor, POOLED (hand) from the other documents: between
#         73.15% and 73.25% right (Brain 4: 73.2)
#   B5-U1 chosen variant minus POOLED (hand), held-out SemCor: >= 0.2 points
#   B5-U2 low end of that gain's 95% interval (documents resampled): >= 0
#   B5-U3 chosen variant minus POOLED (hand), SE07: >= 0 points
#   B5-U4 TEST4 words right, POOLED (hand) scored again: 4,853 (Brain 4's
#         71.4%)
#   B5-U5 chosen variant minus POOLED (hand), TEST4: >= 0 points
#   B5-T6 the cell, when it builds: under 600 seconds
#   B5-T7 the cell, when it loads: under 240 seconds
# =====================================================================
import os, time
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 4: THE THESAURUS CHECK"); print("=" * 64)
T0 = time.time()
for need in ("POOLED", "PARTS", "stored", "combine", "contexts", "LID", "BETA"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-3 first ({need} is missing)")
from mcai.core import at_least, at_most, between
from mcai.sense import SenseCounts, candidates, items_of
from mcai.stats import paired_bootstrap
from mcai.words import read_glosstag

LAB = os.environ.get("MC_AI_LAB") == "1"
for key, claim, rule in (("B5-H1", "held-out SemCor % right, POOLED (hand) from the other documents",
                          between(73.15, 73.25)),
                         ("B5-U1", "chosen substitution variant minus POOLED (hand), held-out SemCor, points",
                          at_least(0.2)),
                         ("B5-U2", "low end of that gain's 95% interval, documents resampled, points", at_least(0.0)),
                         ("B5-U3", "chosen substitution variant minus POOLED (hand), SE07, points", at_least(0.0)),
                         ("B5-U4", "TEST4 words right of 6,798, POOLED (hand) scored again", between(4853, 4853)),
                         ("B5-U5", "chosen substitution variant minus POOLED (hand), TEST4, points", at_least(0.0)),
                         ("B5-T6", "Cell 4 seconds, building", at_most(600)),
                         ("B5-T7", "Cell 4 seconds, loading", at_most(240))):
    LEDGER.predict(key, claim, rule)
VARIANTS = ("CONTEXT, ALL", "CONTEXT, MONO", "PLACES, ALL", "PLACES, MONO")
G3S = (0.25, 0.5, 1, 2, 4)

# ---------------- the practice set: held-out SemCor documents ----------------
t1 = time.time()
DOCS = sorted({it.doc for it in TRAIN})
HELD = set(DOCS[::5])
HOLD_I = np.array([i for i, it in enumerate(TRAIN) if it.doc in HELD])
STORE_I = np.array([i for i, it in enumerate(TRAIN) if it.doc not in HELD])
C_TRAIN = contexts(TRAIN)


def build_store() -> dict:
    """POOLED (hand) from the store documents and the hand-tagged gloss examples (Brain 4 Cell 23)."""
    gfiles = sorted(p for p in (DATA / "glosstag").rglob("*.xml") if p.name in ("noun.xml", "verb.xml", "adj.xml",
                                                                                 "adv.xml"))
    gc = read_glosstag(gfiles)
    g_hand = [it for it in items_of(gc.text, WN) if any(s in SID for s in it.gold) and it.id in gc.manual]
    WHERE.update({t.instance: (toks, k) for toks in gc.text.sentences.values() for k, t in enumerate(toks)
                  if t.instance})
    store_items = [TRAIN[i] for i in STORE_I]
    model = combine([stored(store_items, C_TRAIN[STORE_I]), stored(g_hand, contexts(g_hand))])
    counts = SenseCounts.count(store_items).tagged
    keys = sorted(counts)
    return dict(exv=model["EXV"], w=model["W"], m=model["M"], pos=model["pos"].astype(np.int32),
                n=np.array(model["n"]), tagged_lemma=np.array([k[0] for k in keys]),
                tagged_synset=np.array([k[1] for k in keys]), tagged_n=np.array([counts[k] for k in keys], np.int64))


DEPENDS4 = dict(what="POOLED (hand) without every 5th SemCor document (Brain 4 Cell 23), Brain 5 Cell 4",
                model=ARTS.key("pooled_hand", DEPENDS3), held="sorted documents [::5]")
SM, BUILT4 = ARTS.get("pooled_hand_store", DEPENDS4, build_store)
STORE = dict(EXV=SM["exv"], W=SM["w"], M=SM["m"], pos=SM["pos"])
SCOUNT = Counter({(str(a), str(b)): int(n) for a, b, n in zip(SM["tagged_lemma"].tolist(),
                                                               SM["tagged_synset"].tolist(), SM["tagged_n"])})
print(f"\n  PRACTICE SET ({time.time() - t1:.0f}s; store model {'built and saved' if BUILT4 else 'loaded'}): "
      f"{len(DOCS)} SemCor documents, {len(HELD)} held out ({len(HOLD_I):,} words); the model from the other "
      f"{len(DOCS) - len(HELD)} and the gloss examples: {int(SM['n']):,} synsets with stored examples")

# ---------------- one record per word: what the model and the thesaurus check need ----------------
NEAR = (1, 2, 3, 4)                                       # places -2, -1, +1, +2 in around()'s order


def records(items, ctxs, arounds, model, counts_of, cands_ids=None) -> list:
    out = []
    ow_all, om_all = cunit(words_at_places(ids_of(arounds))), unit(meanings_at_places(arounds))
    for k, it in enumerate(items):
        if cands_ids is None:
            cands = candidates(WN, it)
            ids = np.array([SID[s] for s in cands])
            n = np.array([counts_of(it.lemma, s) for s in cands], float)
            g, prior = SYNV[ids] @ ctxs[k], np.log((n + 1) / (n.sum() + len(cands)))
        else:
            cands, ids, g, prior = cands_ids[k]
        e = model["EXV"][ids] @ ctxs[k]
        j = model["pos"][ids]
        ow = np.where(j >= 0, (model["W"][np.maximum(j, 0)] @ np.conj(ow_all[k])).real, 0.0)
        om = np.where(j >= 0, model["M"][np.maximum(j, 0)] @ om_all[k], 0.0)
        near = [WROW[arounds[k][p]] for p in NEAR if arounds[k][p] in WROW]
        out.append((cands, (g + e) / 2 + BETA * prior + G1 * ow + G2 * om, ctxs[k], near, it))
    return out


t1 = time.time()
HOLD_ITEMS = [TRAIN[i] for i in HOLD_I]
HOLD = records(HOLD_ITEMS, C_TRAIN[HOLD_I], [around(*WHERE[it.id]) for it in HOLD_ITEMS], STORE,
               lambda lem, s: SCOUNT[(lem, s)])
ALLR = records(ITEMS, C_ALL, [around(*WHERE[it.id]) for it in ITEMS], POOLED, None, PARTS)
print(f"  WORDS ({time.time() - t1:.0f}s): {len(HOLD):,} held-out SemCor, {len(ALLR):,} in ALL")

# ---------------- relatives and their fit ----------------
NSENSES = Counter()
for key, ss in WN.senses.items():
    NSENSES[key.rsplit(".", 1)[0]] += len(ss)
LINKS, REL = ("@", "@i", "~", "~i", "&"), {}


def relatives(sid: str) -> dict:
    """{word: vocabulary row} for the one-word lemmas of the synset and of the synsets one link away."""
    if sid not in REL:
        s = WN.synsets[sid]
        near = [s] + [WN.synsets[p.target] for p in s.pointers if p.symbol in LINKS]
        REL[sid] = {w: WROW[w] for w in {x.lower() for n in near for x in n.words if x.isalpha()} if w in WROW}
    return REL[sid]


def sub(rec) -> np.ndarray:
    """4 x candidates: SUB for CONTEXT ALL, CONTEXT MONO, PLACES ALL, PLACES MONO."""
    cands, _, ctx, near, it = rec
    out = np.full((4, len(cands)), np.nan)
    nv = SVD[near] if near else None
    for i, s in enumerate(cands):
        rel = relatives(s)
        for m, mono in enumerate((False, True)):
            rows = [r for w, r in rel.items() if w != it.lemma.lower() and (not mono or NSENSES[w] == 1)]
            if rows:
                R = SVD[rows]
                out[m, i] = float((R @ ctx).max())
                if nv is not None:
                    out[2 + m, i] = float((R @ nv.T).mean(1).max())
    for f in range(4):
        ok = ~np.isnan(out[f])
        out[f, ~ok] = out[f, ok].mean() if ok.any() else 0.0
    return out


t1 = time.time()
SUB_HOLD = [sub(r) for r in HOLD]
SUB_ALL = [sub(r) for r in ALLR]
print(f"  RELATIVES ({time.time() - t1:.0f}s): {len(REL):,} synsets looked at; candidates with a relative in the "
      f"vocabulary, held-out SemCor: {np.mean([np.isfinite(s).all() for s in SUB_HOLD]):.1%} of words fully covered")


def right(recs, subs, f: int, g3: float, mask=None) -> np.ndarray:
    idx = range(len(recs)) if mask is None else np.flatnonzero(mask)
    out = np.zeros(len(recs), bool)
    for k in idx:
        cands, base, _, _, it = recs[k]
        out[k] = cands[int(np.argmax(base + g3 * subs[k][f] if g3 else base))] in it.gold
    return out


BASE_H = right(HOLD, SUB_HOLD, 0, 0)
GRID4 = {(f, g3): int(right(HOLD, SUB_HOLD, f, g3).sum()) for f in range(4) for g3 in G3S}
F, G3 = max(GRID4, key=lambda fg: (GRID4[fg], -fg[1], -fg[0]))
BEST_H = right(HOLD, SUB_HOLD, F, G3)
DOC_H = np.array([it.doc for it in HOLD_ITEMS])
D_H = paired_bootstrap(BEST_H, BASE_H, groups=DOC_H)
BASE_D, BEST_D = right(ALLR, SUB_ALL, 0, 0, DEV), right(ALLR, SUB_ALL, F, G3, DEV)
print(f"\n  HELD-OUT SEMCOR, % right of {len(HOLD):,} (POOLED (hand): {100 * BASE_H.mean():.2f}; Brain 4: 73.2)")
print("    variant          " + "".join(f"{'g3 ' + format(g, 'g'):>9}" for g in G3S))
for f, name in enumerate(VARIANTS):
    print(f"    {name:<17}" + "".join(f"{100 * GRID4[f, g] / len(HOLD):>9.2f}" for g in G3S))
print(f"  CHOSEN: {VARIANTS[F]}, g3 {G3:g}: {100 * BEST_H.mean():.2f} on held-out SemCor, "
      f"{100 * (BEST_H.mean() - BASE_H.mean()):+.2f} points (95% interval {100 * D_H[1]:+.2f} to {100 * D_H[2]:+.2f}); "
      f"SE07 {int(BEST_D.sum())} of {int(DEV.sum())} (POOLED (hand) {int(BASE_D.sum())})")
CHANGED = np.flatnonzero(BEST_H != BASE_H)
print(f"    words whose answer changed: {len(CHANGED):,}; now right {int(BEST_H[CHANGED].sum()):,}, now wrong "
      f"{int(BASE_H[CHANGED].sum()):,}")
for k in CHANGED[:4]:
    cands, base, _, _, it = HOLD[k]
    a, b = cands[int(np.argmax(base))], cands[int(np.argmax(base + G3 * SUB_HOLD[k][F]))]
    print(f"    {it.lemma}: {a} ({WN.synsets[a].definition[:40]}) -> {b} ({WN.synsets[b].definition[:40]}); "
          f"{'right' if b in it.gold else 'wrong'} now")

# ---------------- TEST4: Kaggle only ----------------
TEST4 = ~DEV
results = [("B5-H1", round(100 * BASE_H.mean(), 4)), ("B5-U1", round(100 * (BEST_H.mean() - BASE_H.mean()), 4)),
           ("B5-U2", round(100 * D_H[1], 4)), ("B5-U3", round(100 * (BEST_D.sum() - BASE_D.sum()) / DEV.sum(), 4))]
if LAB:
    print("\n  TEST4: not scored (lab run)")
else:
    LEDGER.look("TEST4 (WSD)", "ALL with stored examples POOLED (hand): SemCor + Princeton gloss corpus (c7b8318a)")
    BASE_T = right(ALLR, SUB_ALL, 0, 0, TEST4)
    results.append(("B5-U4", int(BASE_T.sum())))
    print(f"\n  TEST4 ({int(TEST4.sum()):,} words): POOLED (hand) scored again: {int(BASE_T.sum()):,} right "
          f"({100 * BASE_T[TEST4].mean():.2f}; Brain 4: 71.39); looks {LEDGER.looks('TEST4 (WSD)')}")
    if 100 * (BEST_H.mean() - BASE_H.mean()) >= 0.2 and 100 * D_H[1] >= 0:
        n_looks = LEDGER.look("TEST4 (WSD)", f"POOLED (hand) + SUBSTITUTION {VARIANTS[F]}, g3 {G3:g}, chosen on "
                                             f"held-out SemCor (Brain 5 Cell 4)")
        BEST_T = right(ALLR, SUB_ALL, F, G3, TEST4)
        D_T = paired_bootstrap(BEST_T[TEST4], BASE_T[TEST4], groups=np.array([it.doc for it in ITEMS])[TEST4])
        print(f"    {VARIANTS[F]} g3 {G3:g}: {int(BEST_T.sum()):,} right ({100 * BEST_T[TEST4].mean():.2f}); "
              f"difference {100 * D_T[0]:+.2f} points (95% interval {100 * D_T[1]:+.2f} to {100 * D_T[2]:+.2f}); "
              f"looks now {n_looks}")
        results.append(("B5-U5", round(100 * D_T[0], 4)))
    else:
        print("    the substitution variant is not scored on TEST4: its gain did not hold on the practice set")
print("\n  PREDICTIONS vs RESULTS")
for key, value in results:
    LEDGER.result(key, value)
LEDGER.result("B5-T6" if BUILT4 else "B5-T7", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 4")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 4")


MC AI BRAIN 5 - CELL 4: THE THESAURUS CHECK

  PRACTICE SET (103s; store model built and saved): 352 SemCor documents, 71 held out (45,727 words); the model from the other 281 and the gloss examples: 34,780 synsets with stored examples
  WORDS (12s): 45,727 held-out SemCor, 7,253 in ALL
  RELATIVES (22s): 24,232 synsets looked at; candidates with a relative in the vocabulary, held-out SemCor: 100.0% of words fully covered

  HELD-OUT SEMCOR, % right of 45,727 (POOLED (hand): 73.23; Brain 4: 73.2)
    variant            g3 0.25   g3 0.5     g3 1     g3 2     g3 4
    CONTEXT, ALL         73.21    72.86    71.80    68.94    64.60
    CONTEXT, MONO        73.27    73.23    73.11    72.63    71.66
    PLACES, ALL          73.27    73.21    72.46    70.60    66.73
    PLACES, MONO         73.27    73.26    73.21    72.95    72.29
  CHOSEN: CONTEXT, MONO, g3 0.25: 73.27 on held-out SemCor, +0.04 points (95% interval -0.01 to +0.10); SE07 289 of 455 (POOLED (hand) 289)
    words whose answer 

In [8]:
# =====================================================================
# MC AI BRAIN 5 - CELL 5: QUESTIONS - FIRST BASELINES ON ARC, OPENBOOKQA
# AND MMLU, FROM COUNTING ONLY
# Needs Cells 0-2 (Cell 3 not needed). No internet, nothing trained.
# Practice splits only: ARC-Easy and ARC-Challenge validation, OpenBookQA
# validation, MMLU validation (and ARC train, to choose two weights). No
# test split is read.
# KNOWLEDGE  Simple Wikipedia (Cell 0's copy) cut into sentences, and
#   WordNet's definitions with their examples: an index of each one's
#   words (lower case, letters only, common words left out), weighted by
#   BM25 (Robertson et al. 1994; k1 1.2, b 0.75); saved with Artifacts
# SOLVERS  each scores every answer choice; the highest is the answer
#   (a tie of k choices holding the right one earns 1/k, as ARC scores):
#   SEARCH  Clark et al. (2016)'s IR solver: the best BM25 score, for the
#           question's and the choice's words, of a sentence holding a word
#           of the question and a word of the choice not in the question
#   ASSOC   their PMI solver's idea with Cell 2's word vectors: the mean
#           cosine of the choice's words with the question's words (SVD-
#           PPMI cosines standing in for pointwise mutual information)
#   VECTOR  cosine of the question's and the choice's context vectors
#           (SIF over Cell 2's word vectors, as Cell 3's contexts)
#   ALL     the three z-scored over each question's choices and added;
#           SEARCH weight 1, ASSOC and VECTOR 0, 0.5 or 1 each, chosen on
#           the ARC train splits (most right, ties to the smaller sum)
# SURE     margin = ALL's best minus second best; accuracy on the 30% of
#   questions with the largest margins
# PUBLISHED  ARC test sets (Clark et al. 2018): IR with the 14-million-
#   sentence ARC Corpus 62.6 Easy, 20.3 Challenge. Chance: 25 on ARC
#   (some questions have 3 or 5 choices), OpenBookQA and MMLU
# PREDICTIONS (written before running)
#   B5-Q1 SEARCH, ARC-Easy validation: between 40% and 62%
#   B5-Q2 SEARCH, ARC-Challenge validation: at most 30%
#   B5-Q3 best solver, MMLU validation: between 24% and 33%
#   B5-Q4 best solver, OpenBookQA validation: between 25% and 40%
#   B5-Q5 ALL, ARC-Easy validation, 30% most sure minus all: >= 10 points
#   B5-T8 the cell, when it builds the index: under 900 seconds
#   B5-T9 the cell, when it loads the index: under 300 seconds
# =====================================================================
import re, time
from array import array

import numpy as np
import pyarrow.parquet as pq
import scipy.sparse as sp

print("=" * 64); print("MC AI BRAIN 5 - CELL 5: QUESTIONS - FIRST BASELINES"); print("=" * 64)
T0 = time.time()
for need in ("SVD", "VEC", "VFREQ", "WROW", "ARTS", "WN", "SYNLIST", "LEDGER", "RECORD"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 first ({need} is missing)")
from mcai.book import read_encyclopedia
from mcai.core import at_least, at_most, between

for key, claim, rule in (("B5-Q1", "SEARCH, ARC-Easy validation, % right", between(40, 62)),
                         ("B5-Q2", "SEARCH, ARC-Challenge validation, % right", at_most(30)),
                         ("B5-Q3", "best solver, MMLU validation, % right", between(24, 33)),
                         ("B5-Q4", "best solver, OpenBookQA validation, % right", between(25, 40)),
                         ("B5-Q5", "ALL, ARC-Easy validation: 30% most sure minus all, points", at_least(10)),
                         ("B5-T8", "Cell 5 seconds, building the index", at_most(900)),
                         ("B5-T9", "Cell 5 seconds, loading the index", at_most(300))):
    LEDGER.predict(key, claim, rule)
STOPW = set("""a an the of to in on at for by with from as is are was were be been being and or but if then than
that this these those it its they them their there here what which who whom whose when where why how not no do does
did done have has had having can could would should will shall may might must also into about over under after before
between during each other some any all most more many much such only own same so too very just both either neither
nor per via up down out off again once i me my we our you your he him his she her us""".split())
LETTERS = str.maketrans({c: " " for c in map(chr, range(128)) if not c.isalpha()})
SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")
K1, B, MAX_LEN, MIN_WORDS, MIN_DF = 1.2, 0.75, 25, 3, 2


def content_words(text: str) -> list:
    return [t for t in text.lower().translate(LETTERS).split()
            if t.isascii() and len(t) <= MAX_LEN and t not in STOPW]


# ---------------- the index ----------------
def build_index() -> dict:
    """BM25-weighted words of every Simple Wikipedia sentence and every WordNet definition, column by column."""
    t1 = time.time()
    enc = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])
    term, cols, lens, art, sen = {}, [], array("i"), array("i"), array("i")

    def add(ws: list, a: int, s: int, out: list) -> None:
        out.extend(term.setdefault(x, len(term)) for x in ws)
        lens.append(len(ws)); art.append(a); sen.append(s)

    for a_i, a in enumerate(enc.articles()):
        ids = []
        for s_i, s in enumerate(SPLIT.split(a.text)):
            w = content_words(s)
            if len(w) >= MIN_WORDS:
                add(w, a_i, s_i, ids)
        cols.append(np.array(ids, np.int32))
    n_wiki = len(lens)
    ids = []
    for si, syn in enumerate(SYNLIST):
        w = content_words(" ".join(x.replace("_", " ") for x in syn.words) + " " + syn.definition + " "
                          + " ".join(syn.examples))
        if w:
            add(w, -1, si, ids)
    cols.append(np.array(ids, np.int32))
    n, c = len(lens), np.concatenate(cols)
    del cols, ids
    r = np.repeat(np.arange(n, dtype=np.int32), np.frombuffer(lens, np.int32))
    tf = sp.csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(n, len(term)))
    del r, c
    tf.sum_duplicates()
    length = np.frombuffer(lens, np.int32).astype(np.float32)
    df = np.bincount(tf.indices, minlength=len(term))
    idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
    norm_ = K1 * (1 - B + B * length / length.mean())
    tf.data = (idf[tf.indices] * tf.data * (K1 + 1) / (tf.data + np.repeat(norm_, np.diff(tf.indptr))))
    tf.data = tf.data.astype(np.float32)
    keep = np.flatnonzero(df >= MIN_DF)
    X = tf.tocsc()[:, keep]
    del tf
    terms = sorted(term, key=term.get)
    print(f"  INDEX ({time.time() - t1:.0f}s): {n_wiki:,} Simple Wikipedia sentences and {n - n_wiki:,} WordNet "
          f"definitions; {len(term):,} different words, {len(keep):,} of them in >= {MIN_DF} entries and kept; "
          f"{X.nnz:,} entries")
    return dict(data=X.data, indices=X.indices, indptr=X.indptr.astype(np.int64), n=np.array(n),
                n_wiki=np.array(n_wiki), terms=np.array([terms[k] for k in keep]),
                src=np.stack([np.frombuffer(art, np.int32), np.frombuffer(sen, np.int32)], 1))


DEPENDS5 = dict(what="sentence index (BM25) of Simple Wikipedia and WordNet definitions, Brain 5 Cell 5",
                version=1, k1=K1, b=B, stopwords=sorted(STOPW), max_len=MAX_LEN, min_words=MIN_WORDS,
                min_df=MIN_DF, split=SPLIT.pattern,
                data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("encyclopedia", "wordnet")})
IDX, BUILT5 = ARTS.get("sentence_index", DEPENDS5, build_index)
N_DOCS = int(IDX["n"])
XI, XP, XD = IDX["indices"], IDX["indptr"], IDX["data"]
TERM = {str(w): i for i, w in enumerate(IDX["terms"].tolist())}
print(f"\n  INDEX {'built and saved' if BUILT5 else 'loaded'}: {N_DOCS:,} entries to search, {len(TERM):,} words "
      f"({time.time() - T0:.0f}s)")

# ---------------- the questions ----------------
def table_rows(path) -> list:
    return pq.read_table(path).to_pylist()


def arc(split: str, part: str) -> list:
    out = []
    for r in table_rows(DATA / "arc" / part / f"{split}-00000-of-00001.parquet"):
        lab = list(r["choices"]["label"])
        out.append((r["question"], list(r["choices"]["text"]), lab.index(r["answerKey"])))
    return out


QSETS = {"ARC-Easy validation": arc("validation", "ARC-Easy"),
         "ARC-Challenge validation": arc("validation", "ARC-Challenge"),
         "OpenBookQA validation": [(r["question_stem"], list(r["choices"]["text"]),
                                    list(r["choices"]["label"]).index(r["answerKey"]))
                                   for r in table_rows(DATA / "obqa" / "main" / "validation-00000-of-00001.parquet")],
         "MMLU validation": [(r["question"], list(r["choices"]), int(r["answer"]))
                             for r in table_rows(DATA / "mmlu" / "all" / "validation-00000-of-00001.parquet")]}
TRAIN5 = arc("train", "ARC-Easy") + arc("train", "ARC-Challenge")
print("  QUESTIONS: " + ", ".join(f"{k} {len(v):,}" for k, v in QSETS.items()) + f"; ARC train {len(TRAIN5):,}")

# ---------------- the solvers ----------------
SIF5 = (0.001 / (0.001 + VFREQ / VFREQ.sum())).astype(np.float32)
PC5 = VEC["pc"]


def postings(ids: list):
    if not ids:
        return np.zeros(0, np.int64), np.zeros(0, np.float32)
    return (np.concatenate([XI[XP[t]:XP[t + 1]] for t in ids]), np.concatenate([XD[XP[t]:XP[t + 1]] for t in ids]))


def vec(ws: list):
    rows = [WROW[w] for w in ws if w in WROW]
    if not rows:
        return None
    v = (SIF5[rows][:, None] * SVD[rows]).sum(0)
    v -= (v @ PC5) * PC5
    n = np.linalg.norm(v)
    return v / n if n > 0 else None


def solve(question: str, choices: list) -> tuple:
    """(3 x choices: SEARCH, ASSOC, VECTOR scores; the index row of SEARCH's best sentence per choice)"""
    qw = list(dict.fromkeys(content_words(question)))
    q_ids = [TERM[w] for w in qw if w in TERM]
    qr, qv = postings(q_ids)
    sq = np.bincount(qr, weights=qv, minlength=N_DOCS) if len(qr) else None
    q_rows = [WROW[w] for w in qw if w in WROW]
    q_vec = vec(qw)
    out, best = np.zeros((3, len(choices))), []
    for j, ch in enumerate(choices):
        cw = [w for w in dict.fromkeys(content_words(ch)) if w not in qw]
        r, v = postings([TERM[w] for w in cw if w in TERM])
        b = -1
        if len(r) and sq is not None:
            u, inv = np.unique(r, return_inverse=True)
            so = np.bincount(inv, weights=v)
            ok = sq[u] > 0
            if ok.any():
                k = int(np.argmax(np.where(ok, so + sq[u], -1)))
                out[0, j], b = so[k] + sq[u[k]], int(u[k])
        best.append(b)
        c_rows = [WROW[w] for w in cw if w in WROW]
        if c_rows and q_rows:
            out[1, j] = float((SVD[c_rows] @ SVD[q_rows].T).mean())
        c_vec = vec(cw or content_words(ch))
        if c_vec is not None and q_vec is not None:
            out[2, j] = float(c_vec @ q_vec)
    return out, best


def z(s: np.ndarray) -> np.ndarray:
    sd = s.std()
    return (s - s.mean()) / sd if sd > 0 else np.zeros_like(s)


def credit(scores: np.ndarray, gold: int) -> float:
    top = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-9))
    return 1 / len(top) if gold in top else 0.0


t1 = time.time()
SOLVED = {name: [solve(q, ch) for q, ch, _ in qs] for name, qs in QSETS.items()}
SOLVED_TRAIN = [solve(q, ch) for q, ch, _ in TRAIN5]
print(f"  SOLVED ({time.time() - t1:.0f}s): {sum(len(v) for v in SOLVED.values()) + len(SOLVED_TRAIN):,} questions")


def combined(s: np.ndarray, wa: float, wv: float) -> np.ndarray:
    return z(s[0]) + wa * z(s[1]) + wv * z(s[2])


GRID5 = {(wa, wv): sum(credit(combined(s, wa, wv), g) for (s, _), (_, _, g) in zip(SOLVED_TRAIN, TRAIN5))
         for wa in (0, 0.5, 1) for wv in (0, 0.5, 1)}
WA, WV = max(GRID5, key=lambda k: (round(GRID5[k], 6), -(k[0] + k[1])))
print(f"  ALL's weights on ARC train ({len(TRAIN5):,} questions): ASSOC {WA:g}, VECTOR {WV:g} -> "
      f"{100 * GRID5[WA, WV] / len(TRAIN5):.1f}% (SEARCH alone {100 * GRID5[0, 0] / len(TRAIN5):.1f}%)")

# ---------------- results ----------------
ACC, SURE = {}, {}
print("\n  % right (ties share credit)        SEARCH   ASSOC  VECTOR     ALL  chance   ALL, 30% most sure")
for name, qs in QSETS.items():
    sv = SOLVED[name]
    row = [100 * np.mean([credit(s[f], g) for (s, _), (_, _, g) in zip(sv, qs)]) for f in range(3)]
    allc = np.array([credit(combined(s, WA, WV), g) for (s, _), (_, _, g) in zip(sv, qs)])
    margin = np.array([np.diff(np.sort(combined(s, WA, WV))[-2:])[0] if len(s[0]) > 1 else 0 for s, _ in sv])
    top = np.argsort(-margin, kind="stable")[:max(1, int(round(0.3 * len(qs))))]
    ACC[name], SURE[name] = row + [100 * allc.mean()], 100 * allc[top].mean()
    chance = 100 * np.mean([1 / len(ch) for _, ch, _ in qs])
    print(f"    {name:<30}" + "".join(f"{x:>8.1f}" for x in ACC[name]) + f"{chance:>8.1f}   {SURE[name]:>8.1f}")


ENC5 = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])


def sentence(row: int) -> str:
    a_i, s_i = (int(x) for x in IDX["src"][row])
    if a_i < 0:
        syn = SYNLIST[s_i]
        return f"[WordNet] {syn.words[0]}: {syn.definition}"
    return f"[Wikipedia: {ENC5.titles[a_i]}] " + SPLIT.split(ENC5.texts([a_i])[a_i])[s_i][:150]


print("\n  TWO ARC-EASY QUESTIONS, with SEARCH's best sentence for its answer")
for (q, ch, g), (s, best) in list(zip(QSETS["ARC-Easy validation"], SOLVED["ARC-Easy validation"]))[:2]:
    pick = int(np.argmax(combined(s, WA, WV)))
    said = "right" if pick == g else "wrong; right: " + ch[g][:40]
    print(f"    Q: {q[:110]}\n       answer: {ch[pick][:60]} ({said})")
    if best[pick] >= 0:
        print(f"       {sentence(best[pick])}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-Q1", round(ACC["ARC-Easy validation"][0], 2))
LEDGER.result("B5-Q2", round(ACC["ARC-Challenge validation"][0], 2))
LEDGER.result("B5-Q3", round(max(ACC["MMLU validation"]), 2))
LEDGER.result("B5-Q4", round(max(ACC["OpenBookQA validation"]), 2))
LEDGER.result("B5-Q5", round(SURE["ARC-Easy validation"] - ACC["ARC-Easy validation"][3], 2))
LEDGER.result("B5-T8" if BUILT5 else "B5-T9", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 5")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 5")


MC AI BRAIN 5 - CELL 5: QUESTIONS - FIRST BASELINES
  INDEX (62s): 3,353,949 Simple Wikipedia sentences and 117,659 WordNet definitions; 503,403 different words, 260,674 of them in >= 2 entries and kept; 23,113,969 entries

  INDEX built and saved: 3,471,608 entries to search, 260,674 words (63s)
  QUESTIONS: ARC-Easy validation 570, ARC-Challenge validation 299, OpenBookQA validation 500, MMLU validation 1,531; ARC train 3,370
  SOLVED (73s): 6,270 questions
  ALL's weights on ARC train (3,370 questions): ASSOC 0, VECTOR 0.5 -> 42.2% (SEARCH alone 41.8%)

  % right (ties share credit)        SEARCH   ASSOC  VECTOR     ALL  chance   ALL, 30% most sure
    ARC-Easy validation               46.7    43.9    42.0    47.6    25.0       57.9
    ARC-Challenge validation          26.5    18.4    16.4    20.9    25.1       24.4
    OpenBookQA validation             27.6    20.8    23.5    27.5    25.0       24.0
    MMLU validation                   29.2    26.1    26.3    29.7    25.0       3

In [9]:
# =====================================================================
# MC AI BRAIN 5 - CELL 6: QUESTIONS - WORD FORMS, CHAPTERS, OPPOSITES,
# PINS AND NOT-QUESTIONS
# Needs Cells 0-2 and 5. No internet, nothing trained. Settings are
# chosen on ARC train (Easy + Challenge, 3,370 questions); the practice
# validation splits are scored once, with the chosen settings.
# WORD FORMS  each word becomes its WordNet base form (morphy: exception
#   lists, then suffix rules; noun, verb, adjective, adverb in that order)
# INDEXES  BM25 (k1 1.2, b 0.75) over base forms:
#   SENTENCES  Simple Wikipedia sentences and WordNet entries (as Cell 5)
#   CHAPTERS   whole Simple Wikipedia articles
# PER CHOICE (the question's words taken out of the choice):
#   S  sentence support: the best BM25 score of a sentence holding a word
#      of the question and a word of the choice (Cell 5's SEARCH)
#   C  chapter support: the same over whole articles
#   X  contradiction: S for the choice words' WordNet antonyms ("!"
#      links); 0 for a choice without any
#   PINNED  S = C = 0: no evidence either way
# SCORE  z(S) + wc z(C) - wx z(X), z over the question's choices
#   OPEN WORLD  a pinned choice gets the mean score of the other choices
#     (closed world, as Cell 5: no evidence scores as the lowest)
#   FLIP  if the question's last sentence asks NOT, EXCEPT or LEAST ("at
#     least" left out), the lowest score is the answer
#   wc 0/0.25/0.5/1, wx 0/0.5/1, open world and flip off/on: 48 settings
#   scored on ARC train; the most right is chosen (ties: fewest changes)
# PIN LOG  right answers that were pinned (ARC train, MMLU validation)
#   and their words in no index entry: reports/cell06_pins.txt
# PREDICTIONS (written before running; points = percentage points)
#   B5-R1 base forms alone, ARC train, points vs Cell 5's SEARCH: -1..+2
#   B5-R2 flip alone, the ARC train questions it flips, points: >= 5
#   B5-R3 chapters alone (best wc), ARC-Challenge train, points: 0..+4
#   B5-R4 open world alone, ARC train, points: -1..+1
#   B5-R5 contradiction alone (best wx), ARC train, points: -1..+1
#   B5-R6 chosen settings, ARC-Easy validation, % right: 47..55
#   B5-R7 chosen settings, ARC-Challenge validation, % right: 22..32
#   B5-R8 chosen settings, MMLU validation, % right: 26..33
#   B5-R9 chosen settings, OpenBookQA validation, % right: 25..35
#   B5-T10 the cell, building the indexes: <= 900 s
#   B5-T11 the cell, loading the indexes: <= 300 s
# =====================================================================
import re, time
from array import array
from collections import Counter

import numpy as np
import scipy.sparse as sp

print("=" * 64); print("MC AI BRAIN 5 - CELL 6: QUESTIONS - FORMS, CHAPTERS, OPPOSITES, PINS"); print("=" * 64)
T0 = time.time()
for need in ("SOLVED_TRAIN", "SOLVED", "TRAIN5", "QSETS", "STOPW", "LETTERS", "SPLIT", "MAX_LEN", "SYNLIST",
             "credit", "z", "arc", "table_rows"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 and 5 first ({need} is missing)")
from mcai.book import read_encyclopedia
from mcai.core import at_least, at_most, between
from mcai.sense import Gloss

for key, claim, rule in (
        ("B5-R1", "base forms alone, ARC train, points vs Cell 5 SEARCH", between(-1, 2)),
        ("B5-R2", "flip alone, ARC train questions it flips, points", at_least(5)),
        ("B5-R3", "chapters alone (best wc), ARC-Challenge train, points", between(0, 4)),
        ("B5-R4", "open world alone, ARC train, points", between(-1, 1)),
        ("B5-R5", "contradiction alone (best wx), ARC train, points", between(-1, 1)),
        ("B5-R6", "chosen settings, ARC-Easy validation, % right", between(47, 55)),
        ("B5-R7", "chosen settings, ARC-Challenge validation, % right", between(22, 32)),
        ("B5-R8", "chosen settings, MMLU validation, % right", between(26, 33)),
        ("B5-R9", "chosen settings, OpenBookQA validation, % right", between(25, 35)),
        ("B5-T10", "Cell 6 seconds, building the indexes", at_most(900)),
        ("B5-T11", "Cell 6 seconds, loading the indexes", at_most(300))):
    LEDGER.predict(key, claim, rule)
GLOSS, FORM = Gloss(WN), {}


def base(t: str) -> str:
    b = FORM.get(t)
    if b is None:
        b = FORM[t] = GLOSS.lemma(t)
    return b


def forms(text: str) -> list:
    out = []
    for t in text.lower().translate(LETTERS).split():
        if t.isascii() and len(t) <= MAX_LEN and t not in STOPW:
            b = base(t)
            if b not in STOPW:
                out.append(b)
    return out


def bm25(rows, cols, n_rows: int, n_terms: int):
    """(BM25 weight of each word in each entry as a CSC matrix, the number of entries holding each word)"""
    tf = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n_rows, n_terms))
    tf.sum_duplicates()
    length = np.asarray(tf.sum(1)).ravel()
    df = np.bincount(tf.indices, minlength=n_terms)
    idf = np.log(1 + (n_rows - df + 0.5) / (df + 0.5)).astype(np.float32)
    norm = (K1 * (1 - B + B * length / length[length > 0].mean())).astype(np.float32)
    tf.data = (idf[tf.indices] * tf.data * (K1 + 1) / (tf.data + np.repeat(norm, np.diff(tf.indptr))))
    tf.data = tf.data.astype(np.float32)
    return tf.tocsc(), df


def build6() -> dict:
    t1 = time.time()
    enc = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])
    term, cols, lens, art, sen = {}, [], array("i"), array("i"), array("i")

    def add(ws: list, a: int, s: int, out: list) -> None:
        out.extend(term.setdefault(x, len(term)) for x in ws)
        lens.append(len(ws)); art.append(a); sen.append(s)

    for a_i, a in enumerate(enc.articles()):
        ids = []
        for s_i, s in enumerate(SPLIT.split(a.text)):
            w = forms(s)
            if len(w) >= MIN_WORDS:
                add(w, a_i, s_i, ids)
        cols.append(np.array(ids, np.int32))
    n_wiki, n_art = len(lens), enc.groups[-1]
    ids = []
    for si, syn in enumerate(SYNLIST):
        w = forms(" ".join(x.replace("_", " ") for x in syn.words) + " " + syn.definition + " "
                  + " ".join(syn.examples))
        if w:
            add(w, -1, si, ids)
    cols.append(np.array(ids, np.int32))
    c = np.concatenate(cols)
    del cols, ids
    n, ln, a_of = len(lens), np.frombuffer(lens, np.int32), np.frombuffer(art, np.int32)
    r = np.repeat(np.arange(n, dtype=np.int32), ln)
    k = int(ln[:n_wiki].sum())
    S, df = bm25(r, c, n, len(term))
    A, _ = bm25(a_of[r[:k]], c[:k], n_art, len(term))
    del r, c
    keep = np.flatnonzero(df >= MIN_DF)
    S, A = S[:, keep], A[:, keep]
    terms = sorted(term, key=term.get)
    print(f"  INDEXES ({time.time() - t1:.0f}s): {n_wiki:,} sentences and {n - n_wiki:,} WordNet entries "
          f"({S.nnz:,} entries); {n_art:,} chapters ({A.nnz:,}); {len(term):,} base forms, {len(keep):,} kept "
          f"(in >= {MIN_DF} sentences)")
    out = dict(terms=np.array([terms[j] for j in keep]), n=np.array(n), n_wiki=np.array(n_wiki),
               n_art=np.array(n_art), src=np.stack([a_of, np.frombuffer(sen, np.int32)], 1))
    for name, M in (("s", S), ("a", A)):
        out.update({f"{name}_data": M.data, f"{name}_indices": M.indices,
                    f"{name}_indptr": M.indptr.astype(np.int64)})
    return out


DEPENDS6 = dict(what="base-form BM25 indexes of sentences and chapters, Brain 5 Cell 6", version=1,
                forms="mcai.sense.Gloss.lemma (morphy; n, v, a, r)", k1=K1, b=B, stopwords=sorted(STOPW),
                max_len=MAX_LEN, min_words=MIN_WORDS, min_df=MIN_DF, split=SPLIT.pattern,
                data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("encyclopedia", "wordnet")})
IX6, BUILT6 = ARTS.get("form_indexes", DEPENDS6, build6)
TERM6 = {str(w): i for i, w in enumerate(IX6["terms"].tolist())}
MATS = {"s": (IX6["s_indices"], IX6["s_indptr"], IX6["s_data"], int(IX6["n"])),
        "a": (IX6["a_indices"], IX6["a_indptr"], IX6["a_data"], int(IX6["n_art"]))}
ANT = {}
for syn in SYNLIST:
    for p in syn.pointers:
        if p.symbol == "!" and p.source_word > 0 and p.target_word > 0:
            a = syn.words[p.source_word - 1].split("(")[0].lower()
            b = WN.synsets[p.target].words[p.target_word - 1].split("(")[0].lower()
            if "_" not in a + b:
                ANT.setdefault(a, set()).add(b)
print(f"\n  INDEXES {'built and saved' if BUILT6 else 'loaded'}: {MATS['s'][3]:,} sentences and entries, "
      f"{MATS['a'][3]:,} chapters, {len(TERM6):,} base forms; {len(ANT):,} words with WordNet antonyms "
      f"({time.time() - T0:.0f}s)")
EMPTY = (np.zeros(0, np.int64), np.zeros(0, np.float32))


def post(m: str, ids: list):
    xi, xp, xd, _ = MATS[m]
    if not ids:
        return EMPTY
    return (np.concatenate([xi[xp[t]:xp[t + 1]] for t in ids]), np.concatenate([xd[xp[t]:xp[t + 1]] for t in ids]))


def best(sq, r, v) -> tuple:
    """(best score of an entry holding a question word and one of these words, its row), or (0, -1)"""
    if sq is None or not len(r):
        return 0.0, -1
    u, inv = np.unique(r, return_inverse=True)
    so = np.bincount(inv, weights=v)
    ok = sq[u] > 0
    if not ok.any():
        return 0.0, -1
    k = int(np.argmax(np.where(ok, so + sq[u], -1)))
    return float(so[k] + sq[u[k]]), int(u[k])


def solve6(question: str, choices: list) -> np.ndarray:
    """3 x choices: S, C, X"""
    qw = list(dict.fromkeys(forms(question)))
    q_ids = [TERM6[w] for w in qw if w in TERM6]
    sq = {}
    for m in "sa":
        r, v = post(m, q_ids)
        sq[m] = np.bincount(r, weights=v, minlength=MATS[m][3]) if len(r) else None
    out = np.zeros((3, len(choices)))
    for j, ch in enumerate(choices):
        cw = [w for w in dict.fromkeys(forms(ch)) if w not in qw]
        c_ids = [TERM6[w] for w in cw if w in TERM6]
        out[0, j] = best(sq["s"], *post("s", c_ids))[0]
        out[1, j] = best(sq["a"], *post("a", c_ids))[0]
        anti = sorted({TERM6[a] for w in cw for a in ANT.get(w, ()) if a in TERM6 and a not in qw})
        if anti:
            out[2, j] = best(sq["s"], *post("s", anti))[0]
    return out


NEG = re.compile(r"\b(not|except|least)\b", re.I)


def asks_not(q: str) -> bool:
    last = ([p for p in re.split(r"(?<=[.!?])\s+", q.strip()) if p] or [q])[-1]
    return bool(NEG.search(re.sub(r"\bat least\b", " ", last, flags=re.I)))


def score6(s: np.ndarray, wc: float, wx: float, open_world: bool, flip: bool, neg: bool) -> np.ndarray:
    f = z(s[0]) + wc * z(s[1]) - wx * z(s[2])
    pinned = (s[0] == 0) & (s[1] == 0)
    if open_world and pinned.any() and (~pinned).any():
        f = f.copy()
        f[pinned] = f[~pinned].mean()
    return -f if flip and neg else f


t1 = time.time()
SOL6_TRAIN = [solve6(q, ch) for q, ch, _ in TRAIN5]
NEG_TRAIN = [asks_not(q) for q, _, _ in TRAIN5]
SOL6 = {name: [solve6(q, ch) for q, ch, _ in qs] for name, qs in QSETS.items()}
print(f"  SOLVED ({time.time() - t1:.0f}s): {len(TRAIN5) + sum(len(v) for v in QSETS.values()):,} questions")
N_EASY = len(arc("train", "ARC-Easy"))
EASY, CHAL, EVERY = range(N_EASY), range(N_EASY, len(TRAIN5)), range(len(TRAIN5))


def acc(st: tuple, idx=EVERY) -> float:
    return 100 * np.mean([credit(score6(SOL6_TRAIN[i], *st, NEG_TRAIN[i]), TRAIN5[i][2]) for i in idx])


def acc5(idx=EVERY) -> float:
    return 100 * np.mean([credit(SOLVED_TRAIN[i][0][0], TRAIN5[i][2]) for i in idx])


BASE = (0, 0, False, False)
GRID6 = {(wc, wx, o, f): acc((wc, wx, o, f)) for wc in (0, 0.25, 0.5, 1) for wx in (0, 0.5, 1)
         for o in (False, True) for f in (False, True)}
CHOSEN = max(GRID6, key=lambda k: (round(GRID6[k], 6), -((k[0] > 0) + (k[1] > 0) + k[2] + k[3])))
WC1 = max((0.25, 0.5, 1), key=lambda w: (round(GRID6[w, 0, False, False], 6), -w))
WX1 = max((0.5, 1), key=lambda w: (round(GRID6[0, w, False, False], 6), -w))
FLIPPED = [i for i in EVERY if NEG_TRAIN[i]]
print("\n  ARC TRAIN, % right (ties share credit)              all    Easy  Chall.")
for name, st in (("Cell 5 SEARCH (words as written)", None), ("base forms", BASE),
                 ("base forms + flip", (0, 0, False, True)),
                 (f"base forms + chapters (wc {WC1:g})", (WC1, 0, False, False)),
                 ("base forms + open world", (0, 0, True, False)),
                 (f"base forms + contradiction (wx {WX1:g})", (0, WX1, False, False)),
                 (f"chosen: wc {CHOSEN[0]:g}, wx {CHOSEN[1]:g}, open world {'on' if CHOSEN[2] else 'off'}, "
                  f"flip {'on' if CHOSEN[3] else 'off'}", CHOSEN)):
    vals = [acc5(ix) if st is None else acc(st, ix) for ix in (EVERY, EASY, CHAL)]
    print(f"    {name:<46}" + "".join(f"{v:>8.1f}" for v in vals))
print(f"    flip changes {len(FLIPPED)} questions (NOT, EXCEPT, LEAST): {acc(BASE, FLIPPED):.1f}% -> "
      f"{acc((0, 0, False, True), FLIPPED):.1f}%")

ACC6, SURE6 = {}, {}
print("\n  VALIDATION, chosen settings           % right  chance  Cell 5 SEARCH  30% most sure")
for name, qs in QSETS.items():
    sc = [score6(s, *CHOSEN, asks_not(q)) for s, (q, _, _) in zip(SOL6[name], qs)]
    right = np.array([credit(f, g) for f, (_, _, g) in zip(sc, qs)])
    margin = np.array([np.diff(np.sort(f)[-2:])[0] if len(f) > 1 else 0 for f in sc])
    top = np.argsort(-margin, kind="stable")[:max(1, int(round(0.3 * len(qs))))]
    ACC6[name], SURE6[name] = 100 * right.mean(), 100 * right[top].mean()
    chance = 100 * np.mean([1 / len(ch) for _, ch, _ in qs])
    old = 100 * np.mean([credit(s[0], g) for (s, _), (_, _, g) in zip(SOLVED[name], qs)])
    print(f"    {name:<34}{ACC6[name]:>9.1f}{chance:>8.1f}{old:>15.1f}{SURE6[name]:>15.1f}")

# ---------------- pin log ----------------
MMLU_SUBJ = [r["subject"] for r in table_rows(DATA / "mmlu" / "all" / "validation-00000-of-00001.parquet")]
lines = ["MC AI Brain 5 Cell 6 pin log: right answers with no evidence (S = C = 0), and their words in no index",
         "entry (unknown words). Words only; no question text.", ""]
for name, qs, sols in (("ARC train", TRAIN5, SOL6_TRAIN), ("MMLU validation", QSETS["MMLU validation"],
                                                         SOL6["MMLU validation"])):
    pins = [i for i, ((_, ch, g), s) in enumerate(zip(qs, sols)) if s[0, g] == 0 and s[1, g] == 0 and forms(ch[g])]
    unknown = Counter(w for i in pins for w in set(forms(qs[i][1][qs[i][2]])) if w not in TERM6)
    has_unknown = sum(any(w not in TERM6 for w in forms(qs[i][1][qs[i][2]])) for i in pins)
    head = (f"{name}: {len(pins)} of {len(qs)} right answers pinned; {has_unknown} hold an unknown word, "
            f"{len(pins) - has_unknown} have only known words with no link found")
    lines += [head, "  unknown words, most often first: " + ", ".join(f"{w} {k}" for w, k in unknown.most_common(40))]
    print(f"\n  PINS {head}")
    print("    unknown words: " + ", ".join(w for w, _ in unknown.most_common(15)))
    if name.startswith("MMLU"):
        by = Counter(MMLU_SUBJ[i] for i in pins)
        tot = Counter(MMLU_SUBJ)
        worst = sorted(by, key=lambda s: -by[s] / tot[s])[:8]
        lines.append("  subjects with the most pinned right answers: "
                     + ", ".join(f"{s} {by[s]}/{tot[s]}" for s in worst))
        print("    most pinned subjects: " + ", ".join(f"{s} {by[s]}/{tot[s]}" for s in worst))
    lines.append("")
(WORK / "reports").mkdir(exist_ok=True)
(WORK / "reports" / "cell06_pins.txt").write_text("\n".join(lines) + "\n")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-R1", round(acc(BASE) - acc5(), 2))
LEDGER.result("B5-R2", round(acc((0, 0, False, True), FLIPPED) - acc(BASE, FLIPPED), 2))
LEDGER.result("B5-R3", round(acc((WC1, 0, False, False), CHAL) - acc(BASE, CHAL), 2))
LEDGER.result("B5-R4", round(acc((0, 0, True, False)) - acc(BASE), 2))
LEDGER.result("B5-R5", round(acc((0, WX1, False, False)) - acc(BASE), 2))
LEDGER.result("B5-R6", round(ACC6["ARC-Easy validation"], 2))
LEDGER.result("B5-R7", round(ACC6["ARC-Challenge validation"], 2))
LEDGER.result("B5-R8", round(ACC6["MMLU validation"], 2))
LEDGER.result("B5-R9", round(ACC6["OpenBookQA validation"], 2))
LEDGER.result("B5-T10" if BUILT6 else "B5-T11", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 6")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 6")


MC AI BRAIN 5 - CELL 6: QUESTIONS - FORMS, CHAPTERS, OPPOSITES, PINS
  INDEXES (76s): 3,353,322 sentences and 117,659 WordNet entries (23,002,333 entries); 241,787 chapters (12,980,610); 472,951 base forms, 238,109 kept (in >= 2 sentences)

  INDEXES built and saved: 3,470,981 sentences and entries, 241,787 chapters, 238,109 base forms; 6,159 words with WordNet antonyms (78s)
  SOLVED (117s): 6,270 questions

  ARC TRAIN, % right (ties share credit)              all    Easy  Chall.
    Cell 5 SEARCH (words as written)                  41.8    48.8    27.6
    base forms                                        42.3    49.8    27.3
    base forms + flip                                 42.5    49.8    27.9
    base forms + chapters (wc 0.5)                    45.0    53.2    28.4
    base forms + open world                           42.3    49.8    27.2
    base forms + contradiction (wx 0.5)               41.2    48.8    26.0
    chosen: wc 0.5, wx 0, open world off, flip on     45.2    5

In [10]:
# =====================================================================
# MC AI BRAIN 5 - CELL 7A: FEVER - PAGES FOR EACH CLAIM, BY TITLE
# Needs Cells 0-2, 1B, 5 and 6. No internet, nothing trained. Test files
# (paper_test, shared_task_dev) are not read.
# CLAIMS  a train sample (3,000 per label, seed 7; fit) and paper_dev
#   (9,999; practice)
# PASS 1  every page title of wiki-pages.zip (June 2017, 5,416,537 pages)
#   as a key: lower case, letters and digits only, a closing
#   "(disambiguation)" dropped; one key may name several pages
# PAGES PER CLAIM  every run of 1-8 words of the claim that is a key; a
#   run of only common words is left out, and so is a run inside a longer
#   matched run; longer runs first, then earlier; up to 6 pages per run
#   (the page without a parenthesis first), up to 12 per claim. The
#   choice reads the claim's words only, never its label or evidence
# PASS 2  the sentences of every chosen page (the "lines" of the dump),
#   saved with Artifacts
# PREDICTIONS (written before running)
#   B5-F1 paper_dev SUPPORTS/REFUTES claims with every page of one gold
#         evidence set among their pages, %: 70-95
#   B5-T12 Cells 7A + 7B, building: <= 1500 s; B5-T13 loading: <= 300 s
# =====================================================================
import json, re, time, zipfile
from array import array

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 7A: FEVER - PAGES FOR EACH CLAIM, BY TITLE"); print("=" * 64)
T7 = time.time()
for need in ("FEVER", "FEVER_PINS", "fever_parts", "forms", "STOPW", "ARTS", "LEDGER"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2, 1B, 5 and 6 first ({need} is missing)")
from mcai.core import at_most, between

for key, claim, rule in (("B5-F1", "paper_dev S/R claims with a whole gold evidence set's pages found, %",
                          between(70, 95)),
                         ("B5-T12", "Cells 7A + 7B seconds, building", at_most(1500)),
                         ("B5-T13", "Cells 7A + 7B seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
FV_SEED, FV_PER, MAX_N, PER_RUN, PER_CLAIM = 7, 3000, 8, 6, 12
WORD = re.compile(r"[^\W_]+")
LABELS = ("SUPPORTS", "REFUTES", "NOT ENOUGH INFO")


def tkey(s: str) -> str:
    return " ".join(WORD.findall(s.casefold()))


def unbracket(s: str) -> str:
    return (s.replace("_", " ").replace("-LRB-", "(").replace("-RRB-", ")").replace("-LSB-", "[")
            .replace("-RSB-", "]").replace("-COLON-", ":"))


def claims_of(name: str) -> list:
    with open(FEVER[name], encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def raw_pages(z: zipfile.ZipFile):
    for n in fever_parts(z):
        with z.open(n) as f:
            for line in f:
                if line.strip():
                    yield line


def page_id(line: bytes) -> str:
    if line.startswith(b'{"id": "'):
        e = line.find(b'", "', 8)
        if e > 0:
            raw = line[8:e]
            return json.loads(b'"' + raw + b'"') if b"\\" in raw else raw.decode("utf-8")
    return json.loads(line)["id"]


def gold_sets(c: dict) -> list:
    """Each evidence set as a list of (page, sentence) pairs; [] for NOT ENOUGH INFO"""
    return [[(e[2], e[3]) for e in s] for s in c["evidence"] if all(e[2] is not None for e in s)]


def build7() -> dict:
    t1 = time.time()
    train = claims_of("train.jsonl")
    rng = np.random.default_rng(FV_SEED)
    pick = sorted(int(i) for lab in LABELS
                  for i in rng.choice([j for j, c in enumerate(train) if c["label"] == lab], FV_PER, replace=False))
    dev = claims_of("paper_dev.jsonl")
    claims = [train[i]["claim"] for i in pick] + [c["claim"] for c in dev]
    keys, multi, paren = {}, [], array("b")
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for i, line in enumerate(raw_pages(z)):
            t = page_id(line)
            b = unbracket(t)
            p = bool(re.search(r"\([^()]*\)\s*$", b))
            paren.append(p)
            k = tkey(re.sub(r"\s*\([^()]*\)\s*$", "", b)) if p else tkey(b)
            if not k:
                continue
            v = keys.get(k)
            if v is None:
                keys[k] = i
            elif v >= 0:
                multi.append([v, i])
                keys[k] = -len(multi)
            else:
                multi[-v - 1].append(i)
    n_pages = len(paren)
    paren = np.frombuffer(paren, np.int8).astype(bool)
    print(f"  PASS 1 ({time.time() - t1:.0f}s): {n_pages:,} pages, {len(keys):,} title keys "
          f"({len(multi):,} name several pages)")
    t1 = time.time()
    cand_lists = []
    for text in claims:
        tok = WORD.findall(text.casefold())
        runs = [(i, j) for i in range(len(tok)) for j in range(i + 1, min(len(tok), i + MAX_N) + 1)
                if " ".join(tok[i:j]) in keys and not all(t in STOPW for t in tok[i:j])]
        runs = [(i, j) for i, j in runs if not any(a <= i and j <= b and (a, b) != (i, j) for a, b in runs)]
        got = []
        for i, j in sorted(runs, key=lambda r: (r[0] - r[1], r[0])):
            v = keys[" ".join(tok[i:j])]
            pages = [v] if v >= 0 else sorted(multi[-v - 1], key=lambda p: (paren[p], p))
            for p in pages[:PER_RUN]:
                if p not in got and len(got) < PER_CLAIM:
                    got.append(p)
        cand_lists.append(got)
    del keys, multi
    kept = sorted({p for g in cand_lists for p in g})
    where = {p: k for k, p in enumerate(kept)}
    keep = np.zeros(n_pages, bool)
    keep[kept] = True
    titles, s_page, s_id, texts = [], array("i"), array("i"), []
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for i, line in enumerate(raw_pages(z)):
            if not keep[i]:
                continue
            d = json.loads(line)
            titles.append(d["id"])
            for row in d.get("lines", "").split("\n"):
                parts = row.split("\t")
                if len(parts) > 1 and parts[0].isdigit() and parts[1].strip():
                    s_page.append(len(titles) - 1); s_id.append(int(parts[0])); texts.append(parts[1])
    blob = "\n".join(texts).encode("utf-8")
    lens = np.array([len(t.encode("utf-8")) for t in texts], np.int64)
    off = np.concatenate([[0], np.cumsum(lens + 1)])
    print(f"  PAGES PER CLAIM + PASS 2 ({time.time() - t1:.0f}s): {len(claims):,} claims, {len(kept):,} pages "
          f"kept, {len(texts):,} sentences ({len(blob) / 1e6:.0f} MB)")
    flat = np.array([where[p] for g in cand_lists for p in g], np.int32)
    c_off = np.concatenate([[0], np.cumsum([len(g) for g in cand_lists])]).astype(np.int64)
    return dict(titles=np.array(titles), paren=paren[kept], s_page=np.frombuffer(s_page, np.int32),
                s_id=np.frombuffer(s_id, np.int32), blob=np.frombuffer(blob, np.uint8), off=off,
                train_pick=np.array(pick, np.int64), cand=flat, cand_off=c_off, n_pages=np.array(n_pages))


DEPENDS7 = dict(what="FEVER pages chosen by title for a train sample and paper_dev, Brain 5 Cell 7A", version=1,
                seed=FV_SEED, per_label=FV_PER, max_n=MAX_N, per_run=PER_RUN, per_claim=PER_CLAIM,
                stopwords=sorted(STOPW), word=WORD.pattern,
                data={n: FEVER_PINS[n][0] for n in ("wiki-pages.zip", "train.jsonl", "paper_dev.jsonl")})
FV, BUILT7 = ARTS.get("fever_pages", DEPENDS7, build7)
TRAIN_FV_ALL = claims_of("train.jsonl")
FV_CLAIMS = [TRAIN_FV_ALL[int(i)] for i in FV["train_pick"]] + claims_of("paper_dev.jsonl")
del TRAIN_FV_ALL
N_TR = len(FV["train_pick"])
FV_TITLES = FV["titles"].tolist()
SENT_BY_PAGE = np.concatenate([[0], np.cumsum(np.bincount(FV["s_page"], minlength=len(FV_TITLES)))])
BLOB, OFF = FV["blob"].tobytes(), FV["off"]


def sentence_text(k: int) -> str:
    return BLOB[OFF[k]:OFF[k + 1] - 1].decode("utf-8")


def cands(ci: int) -> list:
    return FV["cand"][FV["cand_off"][ci]:FV["cand_off"][ci + 1]].tolist()


found = whole = 0
for ci in range(N_TR, len(FV_CLAIMS)):
    sets = gold_sets(FV_CLAIMS[ci])
    if sets:
        have = {FV_TITLES[p] for p in cands(ci)}
        found += 1
        whole += any(all(pg in have for pg, _ in s) for s in sets)
PAGE_RECALL = 100 * whole / found
print(f"\n  FEVER PAGES {'built and saved' if BUILT7 else 'loaded'}: {len(FV_CLAIMS):,} claims ({N_TR:,} train sample, "
      f"{len(FV_CLAIMS) - N_TR:,} paper_dev), {len(FV_TITLES):,} pages, {len(FV['s_id']):,} sentences; "
      f"pages per claim {np.mean(np.diff(FV['cand_off'])):.1f} ({time.time() - T7:.0f}s)")
print(f"  paper_dev SUPPORTS/REFUTES claims with a whole gold evidence set's pages found: {whole:,} of {found:,} "
      f"({PAGE_RECALL:.1f}%)")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-F1", round(PAGE_RECALL, 2))
print("  LAST LINE OF MC AI BRAIN 5 CELL 7A")


MC AI BRAIN 5 - CELL 7A: FEVER - PAGES FOR EACH CLAIM, BY TITLE
  PASS 1 (123s): 5,416,537 pages, 4,865,621 title keys (247,813 name several pages)
  PAGES PER CLAIM + PASS 2 (83s): 18,999 claims, 31,218 pages kept, 193,248 sentences (23 MB)

  FEVER PAGES built and saved: 18,999 claims (9,000 train sample, 9,999 paper_dev), 31,218 pages, 193,248 sentences; pages per claim 8.9 (207s)
  paper_dev SUPPORTS/REFUTES claims with a whole gold evidence set's pages found: 5,629 of 6,666 (84.4%)

  PREDICTIONS vs RESULTS
    B5-F1 paper_dev S/R claims with a whole gold evidence set's pages found, %: 84.44 (predicted between 70 and 95) -> HELD
  LAST LINE OF MC AI BRAIN 5 CELL 7A


In [11]:
# =====================================================================
# MC AI BRAIN 5 - CELL 7B: FEVER - EVIDENCE, CONTRADICTION, VERDICT
# Needs Cell 7A. Settings are chosen on the train sample (fit); paper_dev
# (practice) is scored once, with the chosen settings.
# EVIDENCE  each sentence of the claim's pages scores the summed IDF
#   (over the kept sentences) of the claim's words (Cell 6's base forms)
#   it holds, its page title's words counting as its own; numbers count
#   too. The 5 best sentences are the evidence; COVER = share of the
#   claim's summed IDF held by the best 2 together
# CONTRADICTION (in the best 2 sentences; from Mark's FEVER notebook):
#   NUM  the claim's number is not there, another number is
#   NEG  one of claim and best sentence is negated (not, never, no,
#        cannot, n't), the other is not
#   ANT  a claim word is missing and its WordNet antonym is there
#   SIB  a claim word is missing and a word that shares a WordNet
#        hypernym with it (within 3 steps, the hypernym 5 or more steps
#        below the top) is there
#   ONLY the claim says only, exclusively or solely
# VERDICT  no page: NOT ENOUGH INFO (unknown entity); COVER < t1: NOT
#   ENOUGH INFO (missing fact); a chosen contradiction: REFUTES; COVER >=
#   t2: SUPPORTS; otherwise NOT ENOUGH INFO (below threshold)
#   t1 0.1-0.6, t2 0.4-1.0, any set of the 5 contradictions: the most
#   right on the train sample is chosen (ties: fewer contradictions)
# SCORES  label accuracy; FEVER score (Thorne et al. 2018): label right
#   and, for SUPPORTS/REFUTES, one whole gold evidence set in the 5.
#   Published (test, trained classifier): 50.91 label, 31.87 FEVER score
# PREDICTIONS (written before running)
#   B5-F2 paper_dev S/R claims with a whole gold set in the 5, %: 35-75
#   B5-F3 paper_dev label accuracy, %: 38-50
#   B5-F4 paper_dev FEVER score, %: 20-40
#   B5-F5 train sample, contradictions chosen vs none, points: 1-10
# =====================================================================
import re, time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 7B: FEVER - EVIDENCE, CONTRADICTION, VERDICT"); print("=" * 64)
for need in ("FV", "FV_CLAIMS", "N_TR", "cands", "sentence_text", "gold_sets", "unbracket", "forms", "ANT", "WN"):
    if need not in globals():
        raise RuntimeError(f"run Cell 7A first ({need} is missing)")
for key, claim, rule in (("B5-F2", "paper_dev S/R claims with a whole gold set in the 5 sentences, %",
                          between(35, 75)),
                         ("B5-F3", "paper_dev label accuracy, %", between(38, 50)),
                         ("B5-F4", "paper_dev FEVER score, %", between(20, 40)),
                         ("B5-F5", "train sample, contradictions chosen vs none, points", between(1, 10))):
    LEDGER.predict(key, claim, rule)
t1 = time.time()
NUMS = re.compile(r"\d+(?:[.,]\d+)*")
NEGW = re.compile(r"\b(not|never|no|cannot|none|nor)\b|n't\b", re.I)
ONLYW = re.compile(r"\b(only|exclusively|solely)\b", re.I)
KINDS = ("NUM", "NEG", "ANT", "SIB", "ONLY")
S_WORDS = [frozenset(forms(unbracket(sentence_text(k)))) for k in range(len(FV["s_id"]))]
S_NUMS = [frozenset(NUMS.findall(unbracket(sentence_text(k)))) for k in range(len(FV["s_id"]))]
T_WORDS = [frozenset(forms(unbracket(t))) for t in FV["titles"].tolist()]
DF7 = {}
for ws in S_WORDS:
    for w in ws:
        DF7[w] = DF7.get(w, 0) + 1
N7 = len(S_WORDS)


def idf7(w: str) -> float:
    d = DF7.get(w, 0)
    return float(np.log(1 + (N7 - d + 0.5) / (d + 0.5)))


NUM_IDF = idf7("")
UP = {}


def ups(w: str) -> set:
    """Hypernyms within 3 steps of the word's noun senses, keeping those 5+ steps below the top"""
    if w not in UP:
        out = set()
        for syn in WN.synsets_of(w, "n")[:3]:
            path = WN.path_up(syn.id)
            out |= {s for s in path[1:4] if len(WN.path_up(s)) >= 6}
        UP[w] = out
    return UP[w]


def read_claim(ci: int):
    """(predicted evidence [(page, sentence)], COVER, has page, contradictions as 0/1 by KINDS)"""
    c = FV_CLAIMS[ci]
    text = unbracket(c["claim"])
    cw, cn = set(forms(text)), set(NUMS.findall(text))
    pages = cands(ci)
    if not pages:
        return [], 0.0, False, np.zeros(len(KINDS), bool)
    total = sum(idf7(w) for w in cw) + NUM_IDF * len(cn)
    scored = []
    for p in pages:
        for k in range(SENT_BY_PAGE[p], SENT_BY_PAGE[p + 1]):
            ws = S_WORDS[k] | T_WORDS[p]
            scored.append((sum(idf7(w) for w in cw & ws) + NUM_IDF * len(cn & S_NUMS[k]), -len(scored), k))
    scored.sort(reverse=True)
    top = [k for _, _, k in scored[:5]]
    best2 = top[:2]
    ws2 = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in best2))
    ns2 = set().union(*(S_NUMS[k] for k in best2))
    cover = (sum(idf7(w) for w in cw & ws2) + NUM_IDF * len(cn & ns2)) / total if total else 0.0
    missing = cw - ws2
    s_neg = bool(NEGW.search(unbracket(sentence_text(top[0])))) if top else False
    clash = np.array([bool(cn) and not cn <= ns2 and bool(ns2 - cn),
                      bool(NEGW.search(text)) != s_neg,
                      any(a in ws2 for w in missing for a in ANT.get(w, ())),
                      any(ups(w) & ups(v) for w in missing for v in ws2 - cw if v.isalpha()),
                      bool(ONLYW.search(text))])
    ev = [(FV_TITLES[FV["s_page"][k]], int(FV["s_id"][k])) for k in top]
    return ev, cover, True, clash


READ = [read_claim(ci) for ci in range(len(FV_CLAIMS))]
EV = [r[0] for r in READ]
COVER = np.array([r[1] for r in READ])
HASP = np.array([r[2] for r in READ])
CLASH = np.stack([r[3] for r in READ])
GOLD = np.array([LABELS.index(c["label"]) for c in FV_CLAIMS])
print(f"  READ ({time.time() - t1:.0f}s): {len(READ):,} claims; {N7:,} sentences give IDF; "
      f"contradictions found on the train sample: " + ", ".join(f"{k} {CLASH[:N_TR, i].mean():.0%}"
                                                              for i, k in enumerate(KINDS)))


def verdict(t1_: float, t2_: float, use: tuple, idx: slice) -> np.ndarray:
    """0 SUPPORTS, 1 REFUTES, 2 NOT ENOUGH INFO"""
    cov, cl = COVER[idx], CLASH[idx][:, list(use)].any(1) if use else np.zeros(len(COVER[idx]), bool)
    out = np.full(len(cov), 2)
    ok = HASP[idx] & (cov >= t1_)
    out[ok & cl] = 1
    out[ok & ~cl & (cov >= t2_)] = 0
    return out


TR, DV = slice(0, N_TR), slice(N_TR, len(FV_CLAIMS))
USES = [tuple(i for i in range(len(KINDS)) if m >> i & 1) for m in range(2 ** len(KINDS))]
GRID7 = {(a, b, u): 100 * np.mean(verdict(a, b, u, TR) == GOLD[TR])
         for a in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6) for b in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0) if b >= a
         for u in USES}
CH7 = max(GRID7, key=lambda k: (round(GRID7[k], 6), -len(k[2])))
NONE7 = max((k for k in GRID7 if not k[2]), key=lambda k: round(GRID7[k], 6))
print(f"\n  TRAIN SAMPLE ({N_TR:,} claims, a third of each label), label accuracy %:")
rows7 = [(f"no contradiction (t1 {NONE7[0]}, t2 {NONE7[1]})", GRID7[NONE7])]
for i, k in enumerate(KINDS):
    best = max((g for g in GRID7 if g[2] == (i,)), key=lambda g: round(GRID7[g], 6))
    rows7.append((f"{k} alone (t1 {best[0]}, t2 {best[1]})", GRID7[best]))
rows7.append((f"chosen: t1 {CH7[0]}, t2 {CH7[1]}, {'+'.join(KINDS[i] for i in CH7[2]) or 'none'}", GRID7[CH7]))
for label, value in rows7:
    print(f"    {label:<48}{value:>7.1f}")


def fever_scores(pred: np.ndarray, idx: range) -> tuple:
    """(label accuracy %, FEVER score %, S/R claims with a whole gold set in the 5 %)"""
    right = strict = sr = inside = 0
    for p, ci in zip(pred, idx):
        g = GOLD[ci]
        sets = gold_sets(FV_CLAIMS[ci])
        got = set(EV[ci])
        whole = any(all(e in got for e in s) for s in sets)
        if g < 2:
            sr += 1
            inside += whole
        right += p == g
        strict += p == g and (g == 2 or whole)
    return 100 * right / len(idx), 100 * strict / len(idx), 100 * inside / max(sr, 1)


PRED = verdict(*CH7, DV)
DV_IDX = range(N_TR, len(FV_CLAIMS))
ACC7, FS7, EV7 = fever_scores(PRED, DV_IDX)
print(f"\n  PAPER_DEV ({len(DV_IDX):,} claims), chosen settings: label accuracy {ACC7:.1f}%, FEVER score {FS7:.1f}%; "
      f"S/R claims with a whole gold set in the 5 sentences {EV7:.1f}% (chance label accuracy 33.3%)")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    row = [int(((GOLD[DV] == g) & (PRED == p)).sum()) for p in range(3)]
    print(f"    {lab:<18}" + "".join(f"{x:>10,}" for x in row))
why = np.where(~HASP[DV], "unknown entity (no page)", np.where(COVER[DV] < CH7[0], "missing fact (cover < t1)",
               "below threshold (t1 <= cover < t2)"))
said_nei = PRED == 2
print("    NOT ENOUGH INFO said, by reason: " + ", ".join(
    f"{r} {int((said_nei & (why == r)).sum()):,}" for r in ("unknown entity (no page)", "missing fact (cover < t1)",
                                                             "below threshold (t1 <= cover < t2)")))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-F2", round(EV7, 2))
LEDGER.result("B5-F3", round(ACC7, 2))
LEDGER.result("B5-F4", round(FS7, 2))
LEDGER.result("B5-F5", round(GRID7[CH7] - GRID7[NONE7], 2))
LEDGER.result("B5-T12" if BUILT7 else "B5-T13", round(time.time() - T7))
if "sync" in globals():
    sync("Cell 7")
print(f"\n  ({time.time() - T7:.0f}s since Cell 7A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 7B")


MC AI BRAIN 5 - CELL 7B: FEVER - EVIDENCE, CONTRADICTION, VERDICT
  READ (13s): 18,999 claims; 193,248 sentences give IDF; contradictions found on the train sample: NUM 5%, NEG 8%, ANT 1%, SIB 35%, ONLY 4%

  TRAIN SAMPLE (9,000 claims, a third of each label), label accuracy %:
    no contradiction (t1 0.1, t2 0.8)                  46.5
    NUM alone (t1 0.5, t2 0.8)                         47.7
    NEG alone (t1 0.1, t2 0.8)                         50.5
    ANT alone (t1 0.1, t2 0.8)                         47.0
    SIB alone (t1 0.6, t2 0.8)                         44.4
    ONLY alone (t1 0.1, t2 0.8)                        49.8
    chosen: t1 0.1, t2 0.8, NUM+NEG+ANT+ONLY           55.2

  PAPER_DEV (9,999 claims), chosen settings: label accuracy 54.2%, FEVER score 50.0%; S/R claims with a whole gold set in the 5 sentences 79.3% (chance label accuracy 33.3%)
    gold \ said          SUPPORTS   REFUTES       NEI
    SUPPORTS               2,267       140       926
    REFUTES        

In [12]:
# =====================================================================
# MC AI BRAIN 5 - CELL 8: FEVER - THE CLAIM ALONE, AND LINKS COUNTED
# FROM TRAIN CLAIMS AND THEIR EVIDENCE
# Needs Cells 7A and 7B. Train claims outside Cell 7A's sample (fit) are
# counted; settings are chosen on the sample; paper_dev (practice) is
# scored once. Test files are not read.
# CLAIM ONLY  multinomial naive Bayes over the claim's words (every word,
#   lower case; add-one smoothing; equal label priors): a reader that
#   never sees evidence
# LINKS  for each counted SUPPORTS or REFUTES claim and its first gold
#   evidence set (sentences read from wiki-pages.zip, saved with
#   Artifacts): C = the claim's base forms, E = the evidence's (page
#   title words too). Each pair (w in C not in E, v in E not in C) is
#   counted under the claim's label. For a pair seen 3 or more times:
#   clash(w, v) = log((R + 1) / (S + 1)) - log(R claims / S claims)
# USE  in Cell 7B's best 2 sentences, for each claim word they miss:
#   LEARNED  a sixth contradiction: some word there gives clash >= c
#   SAME     the word counts as found: some word there gives clash <= -c
#   t1, t2, the 6 contradictions, c (0.5, 1, 1.5, 2) and SAME (off, on)
#   are chosen on the sample: most right (ties: fewer contradictions)
# PREDICTIONS (written before running)
#   B5-G1 claim only, paper_dev label accuracy, %: 45-60
#   B5-G2 links chosen vs Cell 7B chosen, sample label accuracy, points: 1-8
#   B5-G3 paper_dev label accuracy with links, %: 55-62
#   B5-G4 paper_dev REFUTES claims said REFUTES, %: 38-55
#   B5-T14 Cell 8 seconds, building: <= 900; B5-T15 loading: <= 300
# =====================================================================
import json, math, time, zipfile
from array import array
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 8: FEVER - THE CLAIM ALONE, AND COUNTED LINKS"); print("=" * 64)
T8 = time.time()
for need in ("FV", "FV_CLAIMS", "N_TR", "GOLD", "COVER", "CLASH", "HASP", "GRID7", "CH7", "fever_scores", "idf7",
             "S_WORDS", "T_WORDS", "S_NUMS", "SENT_BY_PAGE", "raw_pages", "page_id", "claims_of", "WORD"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A and 7B first ({need} is missing)")
for key, claim, rule in (("B5-G1", "claim only, paper_dev label accuracy, %", between(45, 60)),
                         ("B5-G2", "links chosen vs Cell 7B chosen, sample label accuracy, points", between(1, 8)),
                         ("B5-G3", "paper_dev label accuracy with links, %", between(55, 62)),
                         ("B5-G4", "paper_dev REFUTES claims said REFUTES, %", between(38, 55)),
                         ("B5-T14", "Cell 8 seconds, building", at_most(900)),
                         ("B5-T15", "Cell 8 seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
PICKED = {int(i) for i in FV["train_pick"]}
COUNTED = [c for i, c in enumerate(claims_of("train.jsonl")) if i not in PICKED]

# ---------------- the claim alone ----------------
NBC = [Counter() for _ in LABELS]
for c in COUNTED:
    NBC[LABELS.index(c["label"])].update(WORD.findall(c["claim"].casefold()))
NB_V, NB_T = len(set().union(*NBC)), [sum(k.values()) for k in NBC]


def claim_only(text: str) -> int:
    t = WORD.findall(text.casefold())
    return int(np.argmax([sum(math.log((NBC[l][w] + 1) / (NB_T[l] + NB_V)) for w in t) for l in range(3)]))


NB_DEV = np.array([claim_only(FV_CLAIMS[ci]["claim"]) for ci in DV_IDX])
NB_ACC = 100 * np.mean(NB_DEV == GOLD[DV])
print(f"  CLAIM ONLY ({len(COUNTED):,} train claims counted, {NB_V:,} words): paper_dev label accuracy "
      f"{NB_ACC:.1f}%; said REFUTES for {100 * np.mean(NB_DEV[GOLD[DV] == 1] == 1):.1f}% of REFUTES claims")


# ---------------- gold evidence of the counted claims ----------------
def build8() -> dict:
    t1 = time.time()
    need = {}
    for c in COUNTED:
        sets = gold_sets(c)
        if c["label"] != "NOT ENOUGH INFO" and sets:
            for pg, sid in sets[0]:
                need.setdefault(pg, set()).add(sid)
    pages, sids, texts = [], array("i"), []
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for line in raw_pages(z):
            t = page_id(line)
            if t in need:
                for row in json.loads(line).get("lines", "").split("\n"):
                    parts = row.split("\t")
                    if len(parts) > 1 and parts[0].isdigit() and int(parts[0]) in need[t]:
                        pages.append(t); sids.append(int(parts[0])); texts.append(parts[1])
    blob = "\n".join(texts).encode("utf-8")
    off = np.concatenate([[0], np.cumsum([len(x.encode("utf-8")) + 1 for x in texts])]).astype(np.int64)
    print(f"  GOLD EVIDENCE ({time.time() - t1:.0f}s): {len(texts):,} sentences from {len(need):,} pages")
    return dict(pages=np.array(pages), sids=np.frombuffer(sids, np.int32), blob=np.frombuffer(blob, np.uint8), off=off)


DEPENDS8 = dict(what="gold evidence sentences of the counted FEVER train claims, Brain 5 Cell 8", version=1,
                seed=FV_SEED, per_label=FV_PER, set="first gold evidence set",
                data={n: FEVER_PINS[n][0] for n in ("wiki-pages.zip", "train.jsonl")})
GE, BUILT8 = ARTS.get("fever_train_evidence", DEPENDS8, build8)
gb, go = GE["blob"].tobytes(), GE["off"]
LOOK = {(p, int(s)): gb[go[k]:go[k + 1] - 1].decode("utf-8") for k, (p, s) in
        enumerate(zip(GE["pages"].tolist(), GE["sids"]))}
del gb

# ---------------- links ----------------
t1 = time.time()
PAIR, N_LAB = {}, [0, 0]
for c in COUNTED:
    if c["label"] == "NOT ENOUGH INFO":
        continue
    sets = gold_sets(c)
    if not sets or any((pg, sid) not in LOOK for pg, sid in sets[0]):
        continue
    E = set()
    for pg, sid in sets[0]:
        E |= set(forms(unbracket(LOOK[pg, sid]))) | set(forms(unbracket(pg)))
    C = set(forms(unbracket(c["claim"])))
    k = 0 if c["label"] == "SUPPORTS" else 1
    N_LAB[k] += 1
    for w in C - E:
        for v in E - C:
            PAIR.setdefault((w, v), [0, 0])[k] += 1
PRIOR = math.log(N_LAB[1] / N_LAB[0])
CL = {wv: math.log((r + 1) / (s + 1)) - PRIOR for wv, (s, r) in PAIR.items() if s + r >= 3}
TOPC = sorted(((r, s, wv) for wv, (s, r) in PAIR.items() if s + r >= 3 and CL[wv] >= 1.5), reverse=True)[:12]
TOPS = sorted(((s, r, wv) for wv, (s, r) in PAIR.items() if s + r >= 3 and CL[wv] <= -1.5), reverse=True)[:12]
del PAIR
print(f"  LINKS ({time.time() - t1:.0f}s): {N_LAB[0]:,} SUPPORTS and {N_LAB[1]:,} REFUTES claims lined up with their "
      f"evidence; {len(CL):,} word pairs seen 3+ times")
print("    clash (claim word / evidence word, REFUTES-SUPPORTS): "
      + ", ".join(f"{w}/{v} {r}-{s}" for r, s, (w, v) in TOPC))
print("    same: " + ", ".join(f"{w}/{v} {r}-{s}" for s, r, (w, v) in TOPS))


def best2(ci: int):
    """(claim base forms, the words of Cell 7B's best 2 sentences): the same scoring and order as Cell 7B"""
    text = unbracket(FV_CLAIMS[ci]["claim"])
    cw, cn = set(forms(text)), set(NUMS.findall(text))
    scored = []
    for p in cands(ci):
        for k in range(SENT_BY_PAGE[p], SENT_BY_PAGE[p + 1]):
            ws = S_WORDS[k] | T_WORDS[p]
            scored.append((sum(idf7(w) for w in cw & ws) + NUM_IDF * len(cn & S_NUMS[k]), -len(scored), k))
    scored.sort(reverse=True)
    top = [k for _, _, k in scored[:2]]
    total = sum(idf7(w) for w in cw) + NUM_IDF * len(cn)
    return cw, set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set(), total


CS = (0.5, 1.0, 1.5, 2.0)
LEARN = {c: np.zeros(len(FV_CLAIMS), bool) for c in CS}
COV8 = {(c, s): COVER.copy() for c in CS for s in (False, True)}
for ci in range(len(FV_CLAIMS)):
    cw, ws2, total = best2(ci)
    extra = ws2 - cw
    for w in cw - ws2:
        vals = [CL[w, v] for v in extra if (w, v) in CL]
        if not vals:
            continue
        hi, lo = max(vals), min(vals)
        for c in CS:
            LEARN[c][ci] |= hi >= c
            if lo <= -c and total:
                COV8[c, True][ci] += idf7(w) / total


CLM = {c: np.concatenate([CLASH, LEARN[c][:, None]], 1) for c in CS}


def verdict8(t1_, t2_, use, c, same, idx):
    cov = COV8[c, same][idx]
    cl = CLM[c][idx][:, list(use)].any(1) if use else np.zeros(len(cov), bool)
    out = np.full(len(cov), 2)
    ok = HASP[idx] & (cov >= t1_)
    out[ok & cl] = 1
    out[ok & ~cl & (cov >= t2_)] = 0
    return out


USES8 = [tuple(i for i in range(6) if m >> i & 1) for m in range(64)]
GRID8 = {(a, b, u, c, s): 100 * np.mean(verdict8(a, b, u, c, s, TR) == GOLD[TR])
         for a in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6) for b in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0) if b >= a
         for u in USES8 for c in CS for s in (False, True)}
CH8 = max(GRID8, key=lambda k: (round(GRID8[k], 6), -len(k[2]), -k[4], k[3]))
NAMES8 = KINDS + ("LEARNED",)
print(f"\n  SAMPLE ({N_TR:,} claims), label accuracy %: Cell 7B chosen {GRID7[CH7]:.1f}; with links "
      f"{GRID8[CH8]:.1f} (t1 {CH8[0]}, t2 {CH8[1]}, {'+'.join(NAMES8[i] for i in CH8[2]) or 'none'}, c {CH8[3]}, "
      f"SAME {'on' if CH8[4] else 'off'})")
PRED8 = verdict8(*CH8, DV)
ACC8, FS8, _ = fever_scores(PRED8, DV_IDX)
REF8 = 100 * np.mean(PRED8[GOLD[DV] == 1] == 1)
print(f"\n  PAPER_DEV ({len(DV_IDX):,} claims)          label accuracy  FEVER score  REFUTES said REFUTES")
NB_REF, REF7 = 100 * np.mean(NB_DEV[GOLD[DV] == 1] == 1), 100 * np.mean(PRED[GOLD[DV] == 1] == 1)
print(f"    claim only (no evidence)          {NB_ACC:>10.1f}            -  {NB_REF:>14.1f}")
print(f"    Cell 7B (evidence, cues)          {ACC7:>10.1f}  {FS7:>11.1f}  {REF7:>14.1f}")
print(f"    Cell 8 (evidence, cues, links)    {ACC8:>10.1f}  {FS8:>11.1f}  {REF8:>14.1f}")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    print(f"    {lab:<18}" + "".join(f"{int(((GOLD[DV] == g) & (PRED8 == p)).sum()):>10,}" for p in range(3)))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-G1", round(NB_ACC, 2))
LEDGER.result("B5-G2", round(GRID8[CH8] - GRID7[CH7], 2))
LEDGER.result("B5-G3", round(ACC8, 2))
LEDGER.result("B5-G4", round(REF8, 2))
LEDGER.result("B5-T14" if BUILT8 else "B5-T15", round(time.time() - T8))
if "sync" in globals():
    sync("Cell 8")
print(f"\n  ({time.time() - T8:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 8")


MC AI BRAIN 5 - CELL 8: FEVER - THE CLAIM ALONE, AND COUNTED LINKS
  CLAIM ONLY (136,449 train claims counted, 21,099 words): paper_dev label accuracy 49.3%; said REFUTES for 61.8% of REFUTES claims
  GOLD EVIDENCE (84s): 17,254 sentences from 7,847 pages
  LINKS (10s): 76,371 SUPPORTS and 26,547 REFUTES claims lined up with their evidence; 116,068 word pairs seen 3+ times
    clash (claim word / evidence word, REFUTES-SUPPORTS): exclusively/american 162-4, canadian/american 155-2, refuse/born 140-1, incapable/american 132-2, die/born 130-73, refuse/american 130-3, ever/american 127-2, book/film 121-38, book/american 117-52, ever/born 116-2, exclusively/film 110-2, television/film 109-38
    same: person/born 10-796, person/american 36-674, person/actor 5-376, person/producer 7-301, acts/born 7-291, person/actress 3-252, person/singer 3-250, person/know 5-245, state/american 13-217, unite/american 13-212, acts/american 8-210, person/songwriter 3-172

  SAMPLE (9,000 claims), label accu

In [13]:
# =====================================================================
# MC AI BRAIN 5 - CELL 9A: FEVER - NATURAL-LOGIC ALIGNMENT (NO TRAINING)
# Needs Cells 7A, 7B and 8; Cell 9B joins. Settings are chosen on Cell
# 7A's train sample (fit); paper_dev (practice) is scored once.
# FEVER-SYMMETRIC (Schuster et al. 2019) v0.2, CC BY-SA 3.0: each claim
#   comes with evidence that supports it and with evidence that refutes
#   it, so the claim alone gets 50% by construction. dev (708 pairs) is
#   practice; test (712) is pinned, not read. Only pairs built from
#   paper_dev claims are scored; the rest are counted and held back.
# ALIGN (MacCartney & Manning 2009, natural logic): each claim content
#   word (Cell 6's base forms; operators and single letters left out)
#   gets one relation to the evidence words (Cell 7B's best 2 sentences
#   with page titles; FEVER-Symmetric's sentence with the gold page names,
#   brackets cut, of its source claim):
#   EQ   the same word, or a WordNet synonym (a shared synset)
#   FWD  an evidence word is a kind of it (WordNet hypernym, <= 4 steps)
#   REV  it is a kind of an evidence word: the claim says more
#   INS  none of these: the claim adds what the evidence does not say
#   SAME on: INS or REV counts as found if a link (LINKS) is <= -c
# EXCLUSION (on a word not found, or on the whole claim):
#   ANT  its WordNet antonym is there
#   SIB  an evidence word shares a WordNet hypernym with it (<= 2 steps
#        each, 6+ steps below the top); neither is a kind of the other
#   NUM  the claim's number is not there and another number is
#   LINK a link (LINKS) to an evidence word is >= c
#   TYPE the kind the claim names after is/was (head noun) matches no
#        kind the evidence names after is/was (same, synonym, or a kind
#        of the other within 12 steps); two kinds of person are let pass
# LINKS  v1 (written first): Cell 8's clash(w, v) minus w's own lean,
#   log((R + 1) / (S + 1)) - log(R / S claims) over counted claims whose
#   evidence lacks w. v2 (after lab/look_cell09.py, on fit and practice
#   data, found the lean removes evidence signal): Cell 8's clash as is.
#   Each version gets its own settings chosen on the sample (Cell 9B)
# OPERATORS  not, never, no, none, nor, cannot, n't and the negative
#   implicatives refuse/fail/decline/neglect to, unable to, incapable of
#   ("refused to X" entails "did not X") each flip polarity; only,
#   exclusively, solely make the claim say more
# PREDICTIONS (written before running)
#   B5-N0 Symmetric dev pairs built from paper_dev claims, %: 30-70
#   B5-N1 claim only (Cell 8), Symmetric scored, two-way, %: 49.9-50.1
#   B5-N2 Cell 8's rule, Symmetric scored, two-way, %: 50-62
#   B5-N3 the gate, Symmetric scored, two-way, %: 55-72
#   B5-N4 the gate minus Cell 8's rule, Symmetric, points: 2-15
#   B5-N5 the gate, paper_dev label accuracy, %: 55-61
#   B5-N6 Cells 9A + 9B seconds: <= 600
#   (In the lab N0, N2, N4 and N5 failed and N3 held; N3-N5 score v1.
#   Written after the lab ablation, which showed values near these for
#   v2's form:)
#   B5-N7 v2 gate, Symmetric scored, two-way, %: 66-70
#   B5-N8 v2 gate, paper_dev label accuracy, %: 54-57
# =====================================================================
import json, math, re, time, urllib.request

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 9A: FEVER - NATURAL-LOGIC ALIGNMENT (NO TRAINING)"); print("=" * 64)
T9 = time.time()
for need in ("CL", "COUNTED", "LOOK", "CH8", "GRID8", "NBC", "NB_T", "PRIOR", "ACC8", "FS8", "ups", "NEGW", "EV"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A, 7B and 8 first ({need} is missing)")
for key, claim, rule in (("B5-N0", "Symmetric dev pairs from paper_dev claims, %", between(30, 70)),
                         ("B5-N1", "claim only, Symmetric scored, two-way, %", between(49.9, 50.1)),
                         ("B5-N2", "Cell 8's rule, Symmetric scored, two-way, %", between(50, 62)),
                         ("B5-N3", "the gate, Symmetric scored, two-way, %", between(55, 72)),
                         ("B5-N4", "gate minus Cell 8's rule, Symmetric, points", between(2, 15)),
                         ("B5-N5", "the gate, paper_dev label accuracy, %", between(55, 61)),
                         ("B5-N6", "Cells 9A + 9B seconds", at_most(600)),
                         ("B5-N7", "v2 gate, Symmetric scored, two-way, %", between(66, 70)),
                         ("B5-N8", "v2 gate, paper_dev label accuracy, %", between(54, 57))):
    LEDGER.predict(key, claim, rule)

# ---------------- FEVER-Symmetric, pinned ----------------
SYM_COMMIT = "fa25d17e73ea0694edc78133ec654b9f32f6db73"
SYM_URL = f"https://raw.githubusercontent.com/TalSchuster/FeverSymmetric/{SYM_COMMIT}/symmetric_v0.2/"
SYM_PINS = {"fever_symmetric_dev.jsonl": ("1d43d016da9264de2139cd10edd459ffbeac84f1fd85d60b70ab6ef63d283f81",
                                          466_743, "practice"),
            "fever_symmetric_test.jsonl": ("b2ff560615bd3337c858bc5b09256d04f39aaa034aaae2351bb2d6345c110f83",
                                           485_280, "test")}


def sym_get(name: str):
    want, nbytes, _ = SYM_PINS[name]
    here = DATA / "feversym" / name
    if here.exists() and here.stat().st_size == nbytes and sha256(here) == want:
        return here
    for p in sorted(INPUT.rglob(name)) if INPUT.exists() else []:
        if p.stat().st_size == nbytes and sha256(p) == want:
            return p
    here.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(urllib.request.Request(SYM_URL + name, headers={"User-Agent": "MC_AI"}),
                                timeout=60) as r:
        here.write_bytes(r.read())
    if here.stat().st_size != nbytes or sha256(here) != want:
        here.unlink()
        raise RuntimeError(f"{name}: other bytes than pinned")
    return here


SYM_PATH = sym_get("fever_symmetric_dev.jsonl")
kept = str(SYM_PATH.relative_to(DATA)) if SYM_PATH.is_relative_to(DATA) else str(SYM_PATH)
RECORD[:] = [r for r in RECORD if r["source"] != "feversym"]
RECORD.append(dict(source="feversym", url=SYM_URL + SYM_PATH.name, repo="TalSchuster/FeverSymmetric",
                   commit=SYM_COMMIT, file=kept, sha256=SYM_PINS[SYM_PATH.name][0], bytes=SYM_PATH.stat().st_size))
ROLES[kept], FROM[kept] = "practice", "feversym"
ROLES["feversym/fever_symmetric_test.jsonl (pinned, not fetched)"] = "test"
(DATA / "sources.json").write_text(json.dumps(RECORD, indent=1))
(DATA / "roles.json").write_text(json.dumps(dict(roles=ROLES, sealed=SEALED, TEST4=[k for k, v in SETS.items()
                                                                                    if v == "test"]), indent=1))
SYM_ALL = [json.loads(line) for line in open(SYM_PATH, encoding="utf-8") if line.strip()]
DEV_CI = {int(FV_CLAIMS[ci]["id"]): ci for ci in range(N_TR, len(FV_CLAIMS))}
SEEN_IDS = {int(c["id"]) for c in COUNTED} | {int(FV_CLAIMS[ci]["id"]) for ci in range(N_TR)}


def base_id(r: dict) -> int:
    return int(re.sub(r"000000[234]$", "", str(r["id"])))


SYM = [r for r in SYM_ALL if base_id(r) in DEV_CI]
GROUPS = {base_id(r) for r in SYM_ALL}
n_dev, n_seen = sum(g in DEV_CI for g in GROUPS), sum(g in SEEN_IDS for g in GROUPS)
print(f"  FEVER-SYMMETRIC dev: {len(SYM_ALL)} pairs from {len(GROUPS)} FEVER claims; {n_dev} of them in paper_dev "
      f"({len(SYM)} pairs scored), {n_seen} in train (held back), {len(GROUPS) - n_dev - n_seen} elsewhere (held back)")

# ---------------- the lexicon ----------------
NEG_LEAN = {}
for c in COUNTED:
    sets = gold_sets(c)
    if c["label"] == "NOT ENOUGH INFO" or not sets or any((pg, sid) not in LOOK for pg, sid in sets[0]):
        continue
    E = set()
    for pg, sid in sets[0]:
        E |= set(forms(unbracket(LOOK[pg, sid]))) | set(forms(unbracket(pg)))
    for w in set(forms(unbracket(c["claim"]))) - E:
        NEG_LEAN.setdefault(w, [0, 0])[0 if c["label"] == "SUPPORTS" else 1] += 1
LEAN = {w: math.log((r + 1) / (s + 1)) - PRIOR for w, (s, r) in NEG_LEAN.items()}
CLC = {wv: v - LEAN.get(wv[0], 0.0) for wv, v in CL.items()}
del NEG_LEAN
SYN9, UP9, DEEP9, NOUN9 = {}, {}, {}, {}
POL9 = re.compile(r"\b(?:not|never|no|none|nor|cannot)\b|n't\b|\b(?:refus|fail|declin|neglect)\w* to\b"
                  r"|\bunable to\b|\bincapable of\b", re.I)
ONLY9 = re.compile(r"\b(?:only|exclusively|solely)\b", re.I)
OPS9 = set("never none nor cannot refuse fail decline neglect unable incapable exclusively solely".split())
COP9 = re.compile(r"\b(?:is|was|are|were)\s+(?:(?:not|n't|never)\s+)?(?:(?:a|an|the)\s+)?(.*)", re.I)
TSTOP = set("of that which who whom whose from in on at by for with to as since where when born ( ) [ ] ; : . -- "
            "known based".split())
TSKIP = set("one part member kind type form sort example name term number set piece unit".split())


def syns(w: str) -> set:
    if w not in SYN9:
        SYN9[w] = {s.id for p in "nvar" for s in WN.synsets_of(w, p)[:3]}
    return SYN9[w]


def up_to(w: str, k: int) -> set:
    if (w, k) not in UP9:
        seen, front = set(), {s.id for p in "nv" for s in WN.synsets_of(w, p)[:3]}
        for _ in range(k):
            front = {q.target for s in front for q in WN.synsets[s].pointers if q.symbol in ("@", "@i")} - seen
            seen |= front
        UP9[w, k] = seen
    return UP9[w, k]


def deep(shared: set) -> bool:
    for s in shared:
        if s not in DEEP9:
            DEEP9[s] = len(WN.path_up(s)) >= 7
        if DEEP9[s]:
            return True
    return False


def kin(h: str, g: str) -> bool:
    return h == g or bool(syns(h) & syns(g)) or bool(syns(h) & up_to(g, 12)) or bool(syns(g) & up_to(h, 12))


def person(w: str) -> bool:
    return any(s.lexname == "noun.person" for s in WN.synsets_of(w, "n")[:2])


def heads(text: str) -> frozenset:
    m, out, last = COP9.search(text), set(), ""
    for t in (m.group(1).split()[:16] if m else []):
        t = t.lower()
        if t in TSTOP:
            break
        if t in (",", "and", "or"):
            out.add(last); last = ""
            continue
        if t not in NOUN9:
            NOUN9[t] = WN.base_forms(t, "n") if t.isascii() and t.isalpha() else []
        n = NOUN9[t]
        if n:
            last = n[0]
        elif t.endswith(("ed", "ing")):
            break
    out.add(last)
    return frozenset(out - TSKIP - {""})


def type_clash(H: frozenset, G: frozenset, E: set) -> bool:
    if not H or not G or H & E or any(kin(h, g) for h in H for g in G):
        return False
    return not (any(person(h) for h in H) and any(person(g) for g in G))


def relate(w: str, E: set, others: list):
    sw, rel, part = syns(w), ("EQ" if w in E else ""), w
    for kind in ("EQ", "FWD", "REV"):
        if rel:
            break
        uw = up_to(w, 4) if kind == "REV" else None
        for v in others:
            if (kind == "EQ" and sw & syns(v)) or (kind == "FWD" and sw & up_to(v, 4)) or (uw and syns(v) & uw):
                rel, part = kind, v
                break
    if rel in ("EQ", "FWD"):
        return rel, part, False, False, (0.0, 0.0, 0.0, 0.0)
    u2 = up_to(w, 2)
    sib = not rel and any(deep(u2 & up_to(v, 2)) for v in others)
    vc, vr = [CLC[w, v] for v in others if (w, v) in CLC], [CL[w, v] for v in others if (w, v) in CL]
    return (rel or "INS", part if rel else "", any(a in E for a in ANT.get(w, ())), sib,
            (min(vc, default=0.0), max(vc, default=0.0), min(vr, default=0.0), max(vr, default=0.0)))


def read9(claim: str, E: set, nums: set, ev_pol: int, kinds_e: frozenset):
    """([(word, share, relation, evidence word, ANT, SIB, links)], NUM, TYPE, number share, swap, only)"""
    text = unbracket(claim)
    C = {w for w in forms(text) if len(w) > 1 and w not in OPS9}
    cn = set(NUMS.findall(text))
    total = sum(idf7(w) for w in C) + NUM_IDF * len(cn)
    others = [v for v in E if v not in C]
    words = [(w, idf7(w) / total if total else 0.0, *relate(w, E, others)) for w in sorted(C)]
    return (words, bool(cn) and not cn <= nums and bool(nums - cn), type_clash(heads(text), kinds_e, E),
            NUM_IDF * len(cn & nums) / total if total else 0.0, len(POL9.findall(text)) % 2 != ev_pol,
            bool(ONLY9.search(text)))


def top2(ci: int) -> list:
    text = unbracket(FV_CLAIMS[ci]["claim"])
    cw, cn, scored = set(forms(text)), set(NUMS.findall(text)), []
    for p in cands(ci):
        for k in range(SENT_BY_PAGE[p], SENT_BY_PAGE[p + 1]):
            ws = S_WORDS[k] | T_WORDS[p]
            scored.append((sum(idf7(w) for w in cw & ws) + NUM_IDF * len(cn & S_NUMS[k]), -len(scored), k))
    scored.sort(reverse=True)
    return [k for _, _, k in scored[:2]]


def sym_title(r: dict) -> set:
    c = FV_CLAIMS[DEV_CI[base_id(r)]]
    return set().union(*(set(forms(unbracket(pg).split("(")[0])) for s in gold_sets(c) for pg, _ in s))


t1 = time.time()
R9 = []
for ci in range(len(FV_CLAIMS)):
    top = top2(ci)
    E = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set()
    texts = [unbracket(sentence_text(k)) for k in top]
    R9.append(read9(FV_CLAIMS[ci]["claim"], E, set().union(*(S_NUMS[k] for k in top)) if top else set(),
                    len(POL9.findall(texts[0])) % 2 if top else 0, frozenset().union(*map(heads, texts))))
ESYM = [set(forms(unbracket(r["evidence"]))) | sym_title(r) for r in SYM]
RS = [read9(r["claim"], E, set(NUMS.findall(unbracket(r["evidence"]))), len(POL9.findall(r["evidence"])) % 2,
            heads(unbracket(r["evidence"]))) for r, E in zip(SYM, ESYM)]
print(f"  ALIGNED ({time.time() - t1:.0f}s): {len(R9):,} FEVER claims, {len(RS)} Symmetric pairs, {len(CLC):,} links")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-N0", round(100 * len(SYM) / len(SYM_ALL), 2))
if "sync" in globals():
    sync("Cell 9A")
print("  LAST LINE OF MC AI BRAIN 5 CELL 9A")


MC AI BRAIN 5 - CELL 9A: FEVER - NATURAL-LOGIC ALIGNMENT (NO TRAINING)
  FEVER-SYMMETRIC dev: 708 pairs from 177 FEVER claims; 171 of them in paper_dev (684 pairs scored), 0 in train (held back), 6 elsewhere (held back)
  ALIGNED (11s): 18,999 FEVER claims, 684 Symmetric pairs, 116,068 links

  PREDICTIONS vs RESULTS
    B5-N0 Symmetric dev pairs from paper_dev claims, %: 96.61 (predicted between 30 and 70) -> DID NOT HOLD
  sync: pushed 5 files to kaggle/ on main as commit 0a3bcf6ddd29; GitHub confirms it (468 entries, head bc4992fae92777b7) (2s)
  LAST LINE OF MC AI BRAIN 5 CELL 9A


In [14]:
# =====================================================================
# MC AI BRAIN 5 - CELL 9B: FEVER - THE NATURAL-LOGIC GATE (NO TRAINING)
# Needs Cell 9A (its relations, exclusions and operators). Predictions
# B5-N1 to B5-N8 are written in Cell 9A.
# JOIN  no page: N. Words not found hold <= t_ins of the claim's IDF: S,
#   else N. A chosen exclusion with found words >= t_anc: R. Only: S
#   becomes N. Claim and best sentence of opposite polarity: S and R swap.
#   t_ins, t_anc 0.1-0.6, any set of the 5 exclusions, c (1, 1.5, 2),
#   SAME (off, on): the most right on the sample (ties: fewer exclusions)
# FEVER-Symmetric is two-way: N counts as SUPPORTS (also shown wrong)
# CUES  each exclusion and operator: share of claims it fires on, and of
#   those, the share that are REFUTES, on the sample and on Symmetric
# =====================================================================
import math, time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 9B: FEVER - THE NATURAL-LOGIC GATE (NO TRAINING)"); print("=" * 64)
if "R9" not in globals() or "RS" not in globals():
    raise RuntimeError("run Cell 9A first")
CS9, SRC = (1.0, 1.5, 2.0), ("ANT", "SIB", "NUM", "LINK", "TYPE")


def features(rows: list, raw: bool) -> dict:
    out = {"swap": np.array([r[4] for r in rows], bool), "only": np.array([r[5] for r in rows], bool)}
    for c in CS9:
        for same in (False, True):
            eqf, alt = np.zeros(len(rows)), np.zeros((len(rows), 5), bool)
            for i, (words, nmiss, tcl, nshare, _, _) in enumerate(rows):
                eqf[i] = nshare
                for _, share, rel, _, ant, sib, links in words:
                    lo, hi = links[2:] if raw else links[:2]
                    if rel in ("EQ", "FWD") or (same and lo <= -c):
                        eqf[i] += share
                    else:
                        alt[i, 0] |= ant; alt[i, 1] |= sib; alt[i, 3] |= hi >= c
                alt[i, 2], alt[i, 4] = nmiss, tcl
            out[c, same] = (eqf, alt)
    return out


def gate(F: dict, t_ins: float, t_anc: float, use: tuple, c: float, same: bool, hasp=None) -> np.ndarray:
    eqf, alt = F[c, same]
    st = np.where(1 - eqf <= t_ins + 1e-9, 0, 2)
    if use:
        st = np.where(alt[:, list(use)].any(1) & (eqf >= t_anc - 1e-9), 1, st)
    st = np.where(F["only"] & (st == 0), 2, st)
    st = np.where(F["swap"] & (st < 2), 1 - st, st)
    return np.where(hasp, st, 2) if hasp is not None else st


TT, USE9 = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6), [tuple(i for i in range(5) if m >> i & 1) for m in range(32)]
GOLD_S, V9 = np.array([int(r["label"] == "REFUTES") for r in SYM]), {}
for ver, raw in (("v1", False), ("v2", True)):
    F, FS = features(R9, raw), features(RS, raw)
    FT = {k: ((v[0][:N_TR], v[1][:N_TR]) if isinstance(k, tuple) else v[:N_TR]) for k, v in F.items()}
    grid = {(a, b, u, c, s): 100 * np.mean(gate(FT, a, b, u, c, s, HASP[:N_TR]) == GOLD[:N_TR])
            for a in TT for b in TT for u in USE9 for c in CS9 for s in (False, True)}
    ch = max(grid, key=lambda k: (round(grid[k], 6), -len(k[2]), -k[4]))
    p, ps = gate(F, *ch, HASP), gate(FS, *ch)
    V9[ver] = dict(F=F, FS=FS, ch=ch, sample=grid[ch], P=p, PS=ps, scores=fever_scores(p[DV], DV_IDX),
                   sym=100 * np.mean((ps == 1) == (GOLD_S == 1)) if len(SYM) else float("nan"))
P9, P9S, CH9, ACC9, FSC9 = V9["v2"]["P"], V9["v2"]["PS"], V9["v2"]["ch"], *V9["v2"]["scores"][:2]
F9, FS9 = V9["v2"]["F"], V9["v2"]["FS"]
# ---------------- FEVER-Symmetric: claim only, Cell 8's rule, the gate ----------------
c8, s8 = CH8[3], CH8[4]
NB2, P8S = np.zeros(len(SYM), int), np.zeros(len(SYM), int)
for i, (r, E) in enumerate(zip(SYM, ESYM)):
    cwords = WORD.findall(r["claim"].casefold())
    NB2[i] = int(np.argmax([sum(math.log((NBC[l][w] + 1) / (NB_T[l] + NB_V)) for w in cwords) for l in (0, 1)]))
    text, ev = unbracket(r["claim"]), unbracket(r["evidence"])
    cw, cn, ns = set(forms(text)), set(NUMS.findall(text)), set(NUMS.findall(ev))
    total, extra, miss = sum(idf7(w) for w in cw) + NUM_IDF * len(cn), E - cw, cw - E
    same_w = {w for w in miss if any(CL.get((w, v), 0) <= -c8 for v in extra)} if s8 else set()
    cov = (sum(idf7(w) for w in (cw & E) | same_w) + NUM_IDF * len(cn & ns)) / total if total else 0.0
    cues = [bool(cn) and not cn <= ns and bool(ns - cn), bool(NEGW.search(text)) != bool(NEGW.search(ev)),
            any(a in E for w in miss for a in ANT.get(w, ())),
            any(ups(w) & ups(v) for w in miss for v in extra if v.isalpha()), bool(ONLYW.search(text)),
            any(CL.get((w, v), 0) >= c8 for w in miss for v in extra)]
    clash = any(cues[j] for j in CH8[2])
    P8S[i] = 1 if cov >= CH8[0] and clash else (0 if cov >= CH8[1] else 2)
print(f"\n  FEVER-SYMMETRIC dev, scored pairs ({len(SYM)}, {GOLD_S.sum()} REFUTES)   two-way   NEI wrong   said NEI")
SYMR = {}
for label, p in (("claim only (Cell 8's naive Bayes)", NB2), ("Cell 8's rule, same evidence", P8S),
                 ("gate v1 (corrected links)", V9["v1"]["PS"]), ("gate v2 (Cell 8's links)", P9S)):
    SYMR[label] = 100 * np.mean((p == 1) == (GOLD_S == 1)) if len(SYM) else float("nan")
    wrong = 100 * np.mean(p == GOLD_S) if len(SYM) else float("nan")
    print(f"    {label:<38}{SYMR[label]:>8.1f}{wrong:>12.1f}{int((p == 2).sum()):>11}")
print(f"\n  {'':<28}{'sample':>8}{'paper_dev':>10}{'FEVER score':>12}  chosen (t_ins, t_anc, exclusions, c, SAME)")
print(f"  {'Cell 8':<28}{GRID8[CH8]:>8.1f}{ACC8:>10.1f}{FS8:>12.1f}")
for ver, d in V9.items():
    a, b, u, c, sm = d["ch"]
    print(f"  {'gate ' + ver:<28}{d['sample']:>8.1f}{d['scores'][0]:>10.1f}{d['scores'][1]:>12.1f}  {a}, {b}, "
          f"{'+'.join(SRC[i] for i in u) or 'none'}, {c}, {'on' if sm else 'off'}")
print("  gate v2 on paper_dev:")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    print(f"    {lab:<18}" + "".join(f"{int(((GOLD[DV] == g) & (P9[DV] == p)).sum()):>10,}" for p in range(3)))
print(f"\n  CUES (v2, c {CH9[3]})   fires, sample   then REFUTES   fires, Symm.   then REFUTES")
alt_t, alt_s = F9[CH9[3], CH9[4]][1][:N_TR], FS9[CH9[3], CH9[4]][1]
for name, a, b in zip(SRC + ("POLARITY", "ONLY"), [*alt_t.T, F9["swap"][:N_TR], F9["only"][:N_TR]],
                      [*alt_s.T, FS9["swap"], FS9["only"]]):
    print(f"    {name:<17}{a.mean():>14.1%}{(GOLD[:N_TR][a] == 1).mean() if a.any() else 0:>15.1%}"
          f"{b.mean():>15.1%}{(GOLD_S[b] == 1).mean() if b.any() else 0:>15.1%}")


def proof(row) -> str:
    words, nmiss, tcl, _, swap, only = row
    steps = [f"{w}:{rel}" + (f"({p})" if p and p != w else "") + ("+ANT" if a else "") + ("+SIB" if s else "")
             for w, _, rel, p, a, s, _ in words]
    flags = [n for n, on in (("NUM-clash", nmiss), ("TYPE-clash", tcl), ("polarity-swap", swap), ("only", only)) if on]
    return " ".join(steps + flags)


print("\n  PROOFS (paper_dev: the first claim said SUPPORTS, REFUTES, NOT ENOUGH INFO)")
for want_p in range(3):
    hit = np.flatnonzero(P9[DV] == want_p)
    if len(hit):
        ci = N_TR + int(hit[0])
        print(f"    {FV_CLAIMS[ci]['claim'][:100]}\n      {proof(R9[ci])[:220]} -> {LABELS[P9[ci]]} "
              f"(gold {FV_CLAIMS[ci]['label']})")
if SYM:
    first = base_id(SYM[0])
    print(f"  PROOFS (FEVER-Symmetric, the pairs built from claim {first})")
    for i in (j for j, r in enumerate(SYM) if base_id(r) == first):
        print(f"    {SYM[i]['claim'][:70]} | {SYM[i]['evidence'][:70]}\n      {proof(RS[i])[:220]} -> "
              f"{LABELS[P9S[i]]} (gold {SYM[i]['label']})")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-N1", round(SYMR["claim only (Cell 8's naive Bayes)"], 2))
LEDGER.result("B5-N2", round(SYMR["Cell 8's rule, same evidence"], 2))
LEDGER.result("B5-N3", round(SYMR["gate v1 (corrected links)"], 2))
LEDGER.result("B5-N4", round(SYMR["gate v1 (corrected links)"] - SYMR["Cell 8's rule, same evidence"], 2))
LEDGER.result("B5-N5", round(V9["v1"]["scores"][0], 2))
LEDGER.result("B5-N6", round(time.time() - T9))
LEDGER.result("B5-N7", round(SYMR["gate v2 (Cell 8's links)"], 2))
LEDGER.result("B5-N8", round(ACC9, 2))
if "sync" in globals():
    sync("Cell 9")
print(f"\n  ({time.time() - T9:.0f}s since Cell 9A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 9B")


MC AI BRAIN 5 - CELL 9B: FEVER - THE NATURAL-LOGIC GATE (NO TRAINING)

  FEVER-SYMMETRIC dev, scored pairs (684, 342 REFUTES)   two-way   NEI wrong   said NEI
    claim only (Cell 8's naive Bayes)         50.0        50.0          0
    Cell 8's rule, same evidence              65.4        54.8        206
    gate v1 (corrected links)                 61.3        47.5        278
    gate v2 (Cell 8's links)                  68.1        54.8        233

                                sample paper_dev FEVER score  chosen (t_ins, t_anc, exclusions, c, SAME)
  Cell 8                          59.9      58.4        52.6
  gate v1                         54.3      52.4        48.6  0.2, 0.1, ANT+NUM+TYPE, 2.0, on
  gate v2                         57.4      55.3        50.2  0.2, 0.1, ANT+NUM+LINK, 2.0, on
  gate v2 on paper_dev:
    gold \ said          SUPPORTS   REFUTES       NEI
    SUPPORTS               2,429       275       629
    REFUTES                  693     1,612     1,028
    NO

In [15]:
# =====================================================================
# MC AI BRAIN 5 - CELL 10A: FEVER - WHAT A CLAIM HAS LEFT OVER, AND THE
# DEFINITION NETWORK
# Needs Cells 7A-9B. Cell 10B counts what each reading does and scores.
# The readings below were designed after lab/look_cell10.py and
# lab/look_cell10b.py (fit data; look_cell10 also printed one table of
# FEVER-Symmetric dev, practice). Their weights are counted in Cell 10B
# on fit data only.
# LEFTOVER  a claim word (Cell 6's base forms, or a number) that is not in
#   the evidence, is no WordNet synonym of an evidence word, has no
#   evidence word that is a kind of it, and is not FAMILY
# FAMILY  the word and an evidence word are WordNet "derivationally
#   related forms" or pertainyms (nominate / nomination, Venezuela /
#   Venezuelan): the same word in another part of speech; counts as found
# DEFINITION NETWORK (Mark's; spreading activation, Collins & Loftus 1975)
#   a word's neighbours: the content words of the definitions of its
#   first 2 WordNet senses per part of speech, and its hypernyms' words.
#   A word's weight is its IDF over all WordNet definitions; a path is as
#   strong as its weakest stop (the honeycomb's bottleneck rule)
#   LINK1 one is in the other's definition; MEET1 their definitions share
#   a word; LINK2, MEET2 the same through one middle word of weight >= 4
# EIGHT READINGS of a claim against its evidence
#   KIND     of the strongest leftover: opposite (WordNet antonym there),
#            number (another number there), kind-of (a kind of an evidence
#            word), sibling (Cell 9's SIB), name (not in WordNet), word;
#            or family only; or nothing left
#   SLOT     claim and evidence compared in word order (difflib): the
#            leftover's gap holds 1-3 evidence words (tight swap), more
#            (loose swap), none (added)
#   DEFINED  a leftover has a LINK1 of strength >= 5 to an evidence word
#   GAP      to the evidence words in its own gap: LINK1 (linked), another
#            relation (related), none, at strength >= 5
#   LINKS    Cell 8's counted links, leftover against evidence words:
#            clash >= 2, same <= -2, both, neither. For the train claims
#            the links are counted again on the other half of the train
#            claims (cross-fitting: a count is not applied to the claims
#            it was counted on)
#   SHARE    of the claim's IDF held by the leftovers
#   POLARITY claim and first evidence sentence differ in polarity (Cell 9)
#   TYPE     Cell 9's kind clash after is/was
#   Read for: every 2nd train claim outside the sample that has gold
#   evidence (fit); the sample and paper_dev with Cell 7B's best 2
#   sentences; FEVER-Symmetric dev's scored pairs. Saved with Artifacts.
# PREDICTIONS (written before running)
#   B5-W0 train claims (gold evidence) whose only leftovers are FAMILY, %: 1.5-8
#   B5-W1 counted reader, train (gold evidence), balanced accuracy, %: 66-73
#   B5-W2 counted reader, Symmetric scored, two-way, %: 67-74
#   B5-W3 counted reader minus Cell 9's gate v2, Symmetric, points: 0-6
#   B5-W4 with minus without the definition network, Symmetric, points: -1.5-2
#   B5-W5 counted reader, three labels, paper_dev label accuracy, %: 55-61
#   B5-W6 its paper_dev FEVER score, %: 49-55
#   B5-T16 Cells 10A + 10B seconds, building: <= 900; B5-T17 loading: <= 300
#   (The first lab run applied Cell 8's links to the claims they were
#   counted on: train 83.9, Symmetric 75.3; without the links 79.1 and
#   76.5. W1-W3 did not hold there. Written after it, for this version:)
#   B5-W7 counted reader, train, balanced accuracy, %: 78-82
#   B5-W8 counted reader, Symmetric scored, two-way, %: 74-78
# =====================================================================
import difflib, math, re, time
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 10A: FEVER - LEFTOVERS AND THE DEFINITION NETWORK"); print("=" * 64)
T10 = time.time()
for need in ("V9", "ESYM", "SYM", "syns", "up_to", "deep", "heads", "type_clash", "POL9", "OPS9", "CL", "LOOK",
             "COUNTED", "top2", "base", "ANT"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A to 9B first ({need} is missing)")
for key, claim, rule in (("B5-W0", "train claims (gold evidence) whose only leftovers are FAMILY, %", between(1.5, 8)),
                         ("B5-W1", "counted reader, train (gold evidence), balanced accuracy, %", between(66, 73)),
                         ("B5-W2", "counted reader, Symmetric scored, two-way, %", between(67, 74)),
                         ("B5-W3", "counted reader minus gate v2, Symmetric, points", between(0, 6)),
                         ("B5-W4", "with minus without the definition network, Symmetric, points", between(-1.5, 2)),
                         ("B5-W5", "counted reader, three labels, paper_dev label accuracy, %", between(55, 61)),
                         ("B5-W6", "counted reader, three labels, paper_dev FEVER score, %", between(49, 55)),
                         ("B5-W7", "counted reader, cross-fitted links, train, balanced accuracy, %", between(78, 82)),
                         ("B5-W8", "counted reader, cross-fitted links, Symmetric, two-way, %", between(74, 78)),
                         ("B5-T16", "Cells 10A + 10B seconds, building", at_most(900)),
                         ("B5-T17", "Cells 10A + 10B seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
F10 = (("KIND", ("opposite", "number", "kind-of", "sibling", "name", "word", "family only", "nothing left")),
       ("SLOT", ("tight swap", "loose swap", "added")),
       ("DEFINED", ("linked", "not linked", "not in WordNet")),
       ("GAP", ("linked", "related", "unrelated", "not in WordNet")),
       ("LINKS", ("clash", "same", "both", "neither")),
       ("SHARE", ("to 0.2", "to 0.4", "to 0.6", "over 0.6")),
       ("POLARITY", ("same", "swapped")),
       ("TYPE", ("no clash", "clash")))
COND10, T_DEF, HUB10 = (1, 2, 3, 4, 5), 5.0, 4.0
TOK10 = re.compile(r"\d+(?:[.,]\d+)*|[^\W\d_]+")

# ---------------- word families and the definition network ----------------
FAM, GDF, NEAR, TWO, NOFAM = {}, Counter(), {}, {}, frozenset()
for s in WN.synsets.values():
    GDF.update(set(forms(s.definition)))
    for q in s.pointers:
        if q.symbol in ("+", "\\") and q.source_word > 0 and q.target_word > 0:
            a = s.words[q.source_word - 1].split("(")[0].lower()
            b = WN.synsets[q.target].words[q.target_word - 1].split("(")[0].lower()
            if "_" not in a + b and a != b:
                FAM.setdefault(a, set()).add(b)
                FAM.setdefault(b, set()).add(a)
N_G = len(WN.synsets)


def gidf(x: str) -> float:
    d = GDF.get(x, 0)
    return math.log(1 + (N_G - d + 0.5) / (d + 0.5))


def near(w: str) -> frozenset:
    if w not in NEAR:
        out = set()
        for p in "nvar":
            for s in WN.synsets_of(w, p)[:2]:
                out |= set(forms(s.definition))
                for q in s.pointers:
                    if q.symbol in ("@", "@i"):
                        out |= set(forms(" ".join(WN.synsets[q.target].words).replace("_", " ")))
        NEAR[w] = frozenset(x for x in out if len(x) > 1 and x != w)
    return NEAR[w]


def two(w: str) -> dict:
    """word reached in 2 steps -> the weight of the strongest middle word (middle words of weight >= HUB10)"""
    if w not in TWO:
        out = {}
        for a in near(w):
            ga = gidf(a)
            if ga >= HUB10:
                for x in near(a):
                    if x != w and out.get(x, 0) < ga:
                        out[x] = ga
        TWO[w] = out
    return TWO[w]


def link1(w: str, v: str) -> float:
    return max(gidf(v) if v in near(w) else 0.0, gidf(w) if w in near(v) else 0.0)


def relate10(w: str, v: str) -> tuple:
    """strength of LINK1, and of the strongest of MEET1, LINK2, MEET2 with the stop it goes through"""
    a, b, a2, b2 = near(w), near(v), two(w), two(v)
    rest = [(gidf(m), m) for m in a & b]
    rest += [(min(a2[v], gidf(v)), v)] if v in a2 else []
    rest += [(min(b2[w], gidf(w)), w)] if w in b2 else []
    rest += [(min(gidf(m), b2[m]), m) for m in a if m in b2] + [(min(gidf(m), a2[m]), m) for m in b if m in a2]
    return link1(w, v), max(rest, default=(0.0, ""))


def seq10(text: str) -> list:
    out = []
    for t in TOK10.findall(text.lower()):
        if t[0].isdigit():
            out.append(t)
        elif t.isascii() and len(t) <= MAX_LEN and t not in STOPW:
            b = base(t)
            if len(b) > 1 and b not in STOPW and b not in OPS9:
                out.append(b)
    return out


def kind10(w: str, E: set, V: list, other_number: bool) -> int:
    if w[0].isdigit():
        return 1 if other_number else 5
    if any(a in E for a in ANT.get(w, ())):
        return 0
    if any(syns(v) & up_to(w, 4) for v in V):
        return 2
    u2 = up_to(w, 2)
    if any(deep(u2 & up_to(v, 2)) for v in V):
        return 3
    return 5 if syns(w) else 4


def read10(claim: str, E: set, sents: list, tell: bool = False, links: dict = None):
    """The 8 readings as numbers; with tell, also (leftovers, family words, the strongest network path)"""
    links = CL if links is None else links
    text, ev = unbracket(claim), [unbracket(s) for s in sents]
    c_seq, e_seq = seq10(text), seq10(" ".join(ev))
    C = set(c_seq)
    V = [v for v in E if v not in C and v.isalpha()]
    e_nums = {t for t in e_seq if t[0].isdigit()}
    left, fam = [], []
    for w in dict.fromkeys(c_seq):
        if w[0].isdigit():
            if w not in e_nums:
                left.append(w)
        elif w in E or any(syns(w) & syns(v) or syns(w) & up_to(v, 4) for v in V):
            continue
        elif FAM.get(w, NOFAM) & E:
            fam.append(w)
        else:
            left.append(w)
    pol = int(len(POL9.findall(text)) % 2 != (len(POL9.findall(ev[0])) % 2 if ev else 0))
    typ = int(type_clash(heads(text), frozenset().union(*map(heads, ev)), E))
    if not left:
        out = (6 if fam else 7, 0, 0, 0, 0, 0, pol, typ)
        return (out, [], fam, None) if tell else out
    total = sum(idf7(w) if w.isalpha() else NUM_IDF for w in C)
    share = sum(idf7(w) if w.isalpha() else NUM_IDF for w in left) / total
    kind = min(kind10(w, E, V, bool(e_nums - C)) for w in left)
    L, Vs, slot, gap = set(left), set(V), 2, {w: set() for w in left}
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, e_seq, c_seq, autojunk=False).get_opcodes():
        if tag == "replace":
            hit = L.intersection(c_seq[j1:j2])
            if hit:
                slot = min(slot, 0 if i2 - i1 <= 3 else 1)
                for w in hit:
                    gap[w].update(v for v in e_seq[i1:i2] if v in Vs)
    lw = [w for w in left if w.isalpha() and syns(w)]
    d1, d2, path = 2, 3, None
    if lw:
        d1 = int(not any(link1(w, v) >= T_DEF for w in lw for v in V if syns(v)))
        best_link, best_rest = 0.0, (0.0, "", "", "")
        for w in lw:
            for v in gap[w]:
                if syns(v):
                    lk, (st, stop) = relate10(w, v)
                    best_link = max(best_link, lk)
                    best_rest = max(best_rest, (st, w, stop, v))
        d2 = 0 if best_link >= T_DEF else 1 if best_rest[0] >= T_DEF else 2
        path = best_rest if best_rest[0] >= T_DEF else None
    vals = [links[w, v] for w in left if w.isalpha() for v in V if (w, v) in links]
    hi, lo = max(vals, default=0.0) >= 2, min(vals, default=0.0) <= -2
    out = (kind, slot, d1, d2, 2 if hi and lo else 0 if hi else 1 if lo else 3,
           max(0, min(3, math.ceil(share * 5 - 1e-9) - 1)), pol, typ)
    return (out, left, fam, path) if tell else out


def gold_evidence(c: dict):
    """(evidence words, evidence sentences) of a counted train claim's first gold set; None without one"""
    sets = gold_sets(c)
    if c["label"] == "NOT ENOUGH INFO" or not sets or any((pg, sid) not in LOOK for pg, sid in sets[0]):
        return None
    E = set()
    for pg, sid in sets[0]:
        E |= set(forms(unbracket(LOOK[pg, sid]))) | set(forms(unbracket(pg)))
    return E, [LOOK[pg, sid] for pg, sid in sets[0]]


def found_evidence(ci: int):
    top = top2(ci)
    E = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set()
    return E, [sentence_text(k) for k in top]


def build10() -> dict:
    t1 = time.time()
    rows, labs, half, n_lab, n = [], [], {}, [0, 0], 0
    for c in COUNTED:
        got = gold_evidence(c)
        if got is None:
            continue
        n += 1
        if n % 2:
            C, k = set(forms(unbracket(c["claim"]))), int(c["label"] == "REFUTES")
            n_lab[k] += 1
            for w in C - got[0]:
                for v in got[0] - C:
                    half.setdefault((w, v), [0, 0])[k] += 1
    prior = math.log(n_lab[1] / n_lab[0])
    half = {wv: math.log((r + 1) / (s + 1)) - prior for wv, (s, r) in half.items() if s + r >= 3}
    n = 0
    for c in COUNTED:
        got = gold_evidence(c)
        if got is None:
            continue
        n += 1
        if n % 2 == 0:
            rows.append(read10(c["claim"], *got, links=half))
            labs.append(int(c["label"] == "REFUTES"))
    fv = [read10(FV_CLAIMS[ci]["claim"], *found_evidence(ci)) for ci in range(len(FV_CLAIMS))]
    sym = [read10(r["claim"], E, [r["evidence"]]) for r, E in zip(SYM, ESYM)]
    print(f"  READ ({time.time() - t1:.0f}s): {len(rows):,} train claims with gold evidence ({len(half):,} links "
          f"counted on the other half), {len(fv):,} sample and paper_dev claims, {len(sym)} Symmetric pairs")
    return dict(train=np.array(rows, np.int8), train_y=np.array(labs, np.int8), fv=np.array(fv, np.int8),
                sym=np.array(sym, np.int8).reshape(-1, 8))


DEPENDS10 = dict(what="8 readings of FEVER claims against evidence, Brain 5 Cell 10A", version=2, every=2,
                 strength=T_DEF, hub=HUB10, senses=2, seed=FV_SEED, per_label=FV_PER, synsets=N_G, pairs=len(SYM),
                 data={n: FEVER_PINS[n][0] for n in ("wiki-pages.zip", "train.jsonl", "paper_dev.jsonl")},
                 symmetric=SYM_PINS["fever_symmetric_dev.jsonl"][0])
FEAT10, BUILT10 = ARTS.get("fever_readings", DEPENDS10, build10)
X_TR, Y_TR = FEAT10["train"].astype(int), FEAT10["train_y"].astype(int)
X_FV, X_SY = FEAT10["fv"].astype(int), FEAT10["sym"].astype(int)
FAM_ONLY = 100 * np.mean(X_TR[:, 0] == 6)
print(f"  READINGS {'built and saved' if BUILT10 else 'loaded'}: {len(X_TR):,} train claims; word families of "
      f"{len(FAM):,} words; {len(GDF):,} words weighed over {N_G:,} definitions")
print(f"  KIND of leftover, train claims with gold evidence ({100 * Y_TR.mean():.1f}% REFUTES):")
for k, name in enumerate(F10[0][1]):
    rows = X_TR[:, 0] == k
    print(f"    {name:<14}{rows.mean():>7.1%} of claims, {Y_TR[rows].mean() if rows.any() else 0:>6.1%} REFUTES")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-W0", round(FAM_ONLY, 2))
if "sync" in globals():
    sync("Cell 10A")
print("  LAST LINE OF MC AI BRAIN 5 CELL 10A")


MC AI BRAIN 5 - CELL 10A: FEVER - LEFTOVERS AND THE DEFINITION NETWORK
  READ (42s): 51,459 train claims with gold evidence (55,241 links counted on the other half), 18,999 sample and paper_dev claims, 684 Symmetric pairs
  READINGS built and saved: 51,459 train claims; word families of 30,670 words; 32,777 words weighed over 117,659 definitions
  KIND of leftover, train claims with gold evidence (25.8% REFUTES):
    opposite         0.7% of claims,  77.9% REFUTES
    number           5.0% of claims,  62.0% REFUTES
    kind-of          3.6% of claims,  38.4% REFUTES
    sibling          7.5% of claims,  52.3% REFUTES
    name             3.1% of claims,  59.4% REFUTES
    word            33.4% of claims,  31.5% REFUTES
    family only      3.3% of claims,   8.4% REFUTES
    nothing left    43.4% of claims,   9.7% REFUTES

  PREDICTIONS vs RESULTS
    B5-W0 train claims (gold evidence) whose only leftovers are FAMILY, %: 3.34 (predicted between 1.5 and 8) -> HELD
  sync: pushed 5 files 

In [16]:
# =====================================================================
# MC AI BRAIN 5 - CELL 10B: FEVER - A READER THAT COUNTS WHAT EACH
# READING DOES
# Needs Cell 10A (its 8 readings). Predictions B5-W1 to B5-W8 are written
# in Cell 10A.
# COUNT  naive Bayes over the readings (counting; add-one smoothing; equal
#   label priors): P(reading's value | label) is counted, the label with
#   the largest product wins. SLOT, DEFINED, GAP, LINKS and SHARE are
#   counted and used only for claims with something left over
#   two labels   counted on train claims outside the sample with their
#                gold evidence (fit): SUPPORTS or REFUTES, the reading
#                task FEVER-Symmetric scores
#   three labels counted on Cell 7A's sample with the evidence Cell 7B
#                found (fit); a claim with no page is NOT ENOUGH INFO
# WEIGHT of a value = log P(value | REFUTES) - log P(value | SUPPORTS)
#   from the two-label count: above 0 speaks for REFUTES. A verdict's
#   weights add up, so each verdict prints with its reasons
# HUD CHECK  next to each weight, the share of REFUTES among FEVER-
#   Symmetric pairs with that value (practice; shown, not used)
# WITHOUT  the count repeated without the definition network (DEFINED and
#   GAP), without Cell 8's links, and with KIND alone
# =====================================================================
import time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 10B: FEVER - A READER THAT COUNTS"); print("=" * 64)
if "X_TR" not in globals() or "read10" not in globals():
    raise RuntimeError("run Cell 10A first")
NV10, ALL10 = [len(v) for _, v in F10], tuple(range(len(F10)))


def count10(X: np.ndarray, y: np.ndarray, n_labels: int, use: tuple) -> dict:
    """reading -> log P(value | label), add-one smoothing"""
    has, model = X[:, 0] < 6, {}
    for f in use:
        rows = has if f in COND10 else np.ones(len(X), bool)
        c = np.zeros((n_labels, NV10[f]))
        np.add.at(c, (y[rows], X[rows, f]), 1)
        model[f] = np.log((c + 1) / (c.sum(1, keepdims=True) + NV10[f]))
    return model


def score10(model: dict, X: np.ndarray) -> np.ndarray:
    """(labels, claims) summed log P, equal priors"""
    has, s = X[:, 0] < 6, 0
    for f, m in model.items():
        s = s + (m[:, X[:, f]] * has if f in COND10 else m[:, X[:, f]])
    return s


def balanced(pred: np.ndarray, y: np.ndarray) -> float:
    return 100 * np.mean([np.mean(pred[y == g] == g) for g in sorted(set(y.tolist()))])


# ---------------- two labels: reading with the evidence given ----------------
GOLD_Y = np.array([int(r["label"] == "REFUTES") for r in SYM])
SETS10 = (("all 8 readings", ALL10), ("without the definition network", (0, 1, 4, 5, 6, 7)),
          ("without Cell 8's links", (0, 1, 2, 3, 5, 6, 7)), ("KIND alone", (0,)))
READ10 = {}
print(f"  TWO LABELS, counted on {len(X_TR):,} train claims with gold evidence ({100 * Y_TR.mean():.1f}% REFUTES)")
print(f"    {'':<34}{'train, balanced':>16}{'Symmetric, two-way':>20}")
for label, use in SETS10:
    m = count10(X_TR, Y_TR, 2, use)
    p_tr, p_sy = score10(m, X_TR).argmax(0), score10(m, X_SY).argmax(0)
    READ10[label] = (m, balanced(p_tr, Y_TR), 100 * np.mean(p_sy == GOLD_Y) if len(SYM) else float("nan"), p_sy)
    print(f"    {label:<34}{READ10[label][1]:>16.1f}{READ10[label][2]:>20.1f}")
M2, BAL10, SYM10, P_SY = READ10["all 8 readings"]
R8S = SYMR["Cell 8's rule, same evidence"]
print(f"    Cell 9's gate v2 {V9['v2']['sym']:.1f}; Cell 8's rule {R8S:.1f}; the claim alone 50.0")

print("\n  WHAT EACH READING DOES (weight above 0 speaks for REFUTES)")
print(f"    {'':<26}{'of train':>9}{'REFUTES':>9}{'weight':>8}   {'Symmetric: pairs':>17}{'REFUTES':>9}")
for f, (name, values) in enumerate(F10):
    tr = X_TR[:, 0] < 6 if f in COND10 else np.ones(len(X_TR), bool)
    sy = X_SY[:, 0] < 6 if f in COND10 else np.ones(len(X_SY), bool)
    for k, value in enumerate(values):
        a, b = tr & (X_TR[:, f] == k), sy & (X_SY[:, f] == k)
        if a.any():
            print(f"    {name + ' ' + value:<26}{a.sum() / tr.sum():>9.1%}{Y_TR[a].mean():>9.1%}"
                  f"{M2[f][1, k] - M2[f][0, k]:>+8.2f}   {int(b.sum()):>17}"
                  f"{GOLD_Y[b].mean() if b.any() else 0:>9.1%}")

# ---------------- three labels: the whole pipeline ----------------
HAS_TR = HASP[:N_TR]
THREE10 = {}
for label, use in SETS10[:2]:
    m3 = count10(X_FV[:N_TR][HAS_TR], GOLD[:N_TR][HAS_TR], 3, use)
    p3 = np.where(HASP, score10(m3, X_FV).argmax(0), 2)
    THREE10[label] = (p3, 100 * np.mean(p3[:N_TR] == GOLD[:N_TR]), *fever_scores(p3[DV], DV_IDX)[:2])
P10, SAMPLE10, ACC10, FS10 = THREE10["all 8 readings"]
print(f"\n  THREE LABELS, counted on the sample ({int(HAS_TR.sum()):,} claims with a page)")
print(f"    {'':<40}{'sample':>6}{'paper_dev':>11}{'FEVER score':>13}")
print(f"    {'Cell 8 (cues and links)':<40}{GRID8[CH8]:>6.1f}{ACC8:>11.1f}{FS8:>13.1f}")
G2 = V9["v2"]
print(f"    {'Cell 9 gate v2':<40}{G2['sample']:>6.1f}{G2['scores'][0]:>11.1f}{G2['scores'][1]:>13.1f}")
for label, (_, a, b, c) in THREE10.items():
    print(f"    {'counted, ' + label:<40}{a:>6.1f}{b:>11.1f}{c:>13.1f}")
print(f"    REFUTES said REFUTES {100 * np.mean(P10[DV][GOLD[DV] == 1] == 1):.1f}%; counted reader on paper_dev:")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    print(f"    {lab:<18}" + "".join(f"{int(((GOLD[DV] == g) & (P10[DV] == p)).sum()):>10,}" for p in range(3)))


def reasons(x: np.ndarray) -> str:
    parts = [(M2[f][1, x[f]] - M2[f][0, x[f]], f"{F10[f][0]} {F10[f][1][x[f]]}") for f in ALL10
             if f not in COND10 or x[0] < 6]
    return ", ".join(f"{name} {w:+.2f}" for w, name in sorted(parts, key=lambda t: -abs(t[0]))[:5])


if SYM:
    first = base_id(SYM[0])
    print(f"\n  VERDICTS WITH REASONS (FEVER-Symmetric, the pairs built from claim {first}; two labels)")
    for i in (j for j, r in enumerate(SYM) if base_id(r) == first):
        _, left, fam, path = read10(SYM[i]["claim"], ESYM[i], [SYM[i]["evidence"]], tell=True)
        net = f"; network: {path[1]} -> {path[2]} <- {path[3]} ({path[0]:.1f})" if path else ""
        print(f"    {SYM[i]['claim'][:66]} | {SYM[i]['evidence'][:66]}\n      left over: {' '.join(left) or '-'}"
              f"{'; family: ' + ' '.join(fam) if fam else ''}{net}\n      {reasons(X_SY[i])} -> "
              f"{LABELS[P_SY[i]]} (gold {SYM[i]['label']})")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-W1", round(BAL10, 2))
LEDGER.result("B5-W2", round(SYM10, 2))
LEDGER.result("B5-W3", round(SYM10 - V9["v2"]["sym"], 2))
LEDGER.result("B5-W4", round(SYM10 - READ10["without the definition network"][2], 2))
LEDGER.result("B5-W5", round(ACC10, 2))
LEDGER.result("B5-W6", round(FS10, 2))
LEDGER.result("B5-W7", round(BAL10, 2))
LEDGER.result("B5-W8", round(SYM10, 2))
LEDGER.result("B5-T16" if BUILT10 else "B5-T17", round(time.time() - T10))
if "sync" in globals():
    sync("Cell 10")
print(f"\n  ({time.time() - T10:.0f}s since Cell 10A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 10B")


MC AI BRAIN 5 - CELL 10B: FEVER - A READER THAT COUNTS
  TWO LABELS, counted on 51,459 train claims with gold evidence (25.8% REFUTES)
                                       train, balanced  Symmetric, two-way
    all 8 readings                                81.5                77.5
    without the definition network                80.9                76.0
    without Cell 8's links                        79.1                76.5
    KIND alone                                    69.8                70.0
    Cell 9's gate v2 68.1; Cell 8's rule 65.4; the claim alone 50.0

  WHAT EACH READING DOES (weight above 0 speaks for REFUTES)
                               of train  REFUTES  weight    Symmetric: pairs  REFUTES
    KIND opposite                  0.7%    77.9%   +2.31                  10    90.0%
    KIND number                    5.0%    62.0%   +1.54                  36    80.6%
    KIND kind-of                   3.6%    38.4%   +0.58                  21    61.9%
    KIND sibling